In [1]:
# ==============================================================================
# TECHMIND v1.1.0
# Notebook 05 — Reentrenamiento y mejora del modelo
# ==============================================================================

from pathlib import Path
from copy import deepcopy
import json
import unicodedata
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")


# ==============================================================================
# CONFIGURACIÓN
# ==============================================================================

VERSION_MODELO = "1.1.0"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS_CV = 5
N_JOBS = 1

COLUMNA_OBJETIVO = "categoria"
COLUMNA_TEXTO = "texto_combinado_ponderado"


# ==============================================================================
# LOCALIZAR PROYECTO
# ==============================================================================

BASES_BUSQUEDA = [
    Path.cwd(),
    Path.cwd().parent,
]


def localizar_archivo(*rutas_relativas):
    for base in BASES_BUSQUEDA:
        for ruta_relativa in rutas_relativas:
            ruta = (base / ruta_relativa).resolve()

            if ruta.exists():
                return ruta

    return None


PATH_DATASET = localizar_archivo(
    "data/processed/techmind_modelado.csv",
    "../data/processed/techmind_modelado.csv",
)

PATH_MODELO_V1 = localizar_archivo(
    "artifacts/techmind_modelo_final.joblib",
    "models/techmind_modelo_final.joblib",
    "../artifacts/techmind_modelo_final.joblib",
    "../models/techmind_modelo_final.joblib",
)


if PATH_DATASET is None:
    raise FileNotFoundError(
        "No se encontró techmind_modelado.csv."
    )

if PATH_MODELO_V1 is None:
    raise FileNotFoundError(
        "No se encontró techmind_modelo_final.joblib."
    )


print("Dataset:")
print(PATH_DATASET)

print("\nModelo v1.0.0:")
print(PATH_MODELO_V1)


# ==============================================================================
# DIRECTORIOS V1.1
# ==============================================================================

PROJECT_ROOT = PATH_DATASET.parents[2]

REPORTS_V11 = PROJECT_ROOT / "reports" / "v1.1.0"
MODELS_V11 = PROJECT_ROOT / "models" / "v1.1.0"

REPORTS_V11.mkdir(parents=True, exist_ok=True)
MODELS_V11.mkdir(parents=True, exist_ok=True)


# ==============================================================================
# CARGA
# ==============================================================================

df_modelos = pd.read_csv(PATH_DATASET)

columnas_requeridas = {
    COLUMNA_OBJETIVO,
    COLUMNA_TEXTO
}

faltantes = columnas_requeridas - set(df_modelos.columns)

if faltantes:
    raise ValueError(
        f"Faltan columnas obligatorias: {faltantes}"
    )


df_modelos[COLUMNA_TEXTO] = (
    df_modelos[COLUMNA_TEXTO]
    .fillna("")
    .astype(str)
)

df_modelos[COLUMNA_OBJETIVO] = (
    df_modelos[COLUMNA_OBJETIVO]
    .astype(str)
)


print("\nRegistros:", len(df_modelos))
print("Clases:", sorted(df_modelos[COLUMNA_OBJETIVO].unique()))
print("\nDistribución:")
print(df_modelos[COLUMNA_OBJETIVO].value_counts())

Dataset:
C:\Users\MAMÁ\Downloads\data\processed\techmind_modelado.csv

Modelo v1.0.0:
C:\Users\MAMÁ\Downloads\models\techmind_modelo_final.joblib

Registros: 4583
Clases: ['backend', 'cloud', 'datascience', 'frontend']

Distribución:
categoria
frontend       1165
backend        1159
cloud          1157
datascience    1102
Name: count, dtype: int64


In [2]:
# ==============================================================================
# SPLIT REPRODUCIBLE
# ==============================================================================

indices = df_modelos.index.to_numpy()

indices_train, indices_test = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_modelos[COLUMNA_OBJETIVO],
)


df_train = df_modelos.loc[indices_train].copy()
df_test = df_modelos.loc[indices_test].copy()


X_train = (
    df_train[COLUMNA_TEXTO]
    .fillna("")
    .astype(str)
)

y_train = df_train[COLUMNA_OBJETIVO]


X_test = (
    df_test[COLUMNA_TEXTO]
    .fillna("")
    .astype(str)
)

y_test = df_test[COLUMNA_OBJETIVO]


print("Train:", len(df_train))
print("Test reservado:", len(df_test))


if len(df_modelos) == 4583:
    assert len(df_train) == 3666
    assert len(df_test) == 917


print("\nDistribución train:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribución test:")
print(y_test.value_counts(normalize=True).round(4))


cv_estratificada = StratifiedKFold(
    n_splits=N_SPLITS_CV,
    shuffle=True,
    random_state=RANDOM_STATE,
)


print("\nSplit preparado correctamente.")
print("IMPORTANTE: el test permanecerá sin utilizar hasta la evaluación final.")

Train: 3666
Test reservado: 917

Distribución train:
categoria
frontend       0.2542
backend        0.2529
cloud          0.2523
datascience    0.2406
Name: proportion, dtype: float64

Distribución test:
categoria
frontend       0.2541
backend        0.2530
cloud          0.2530
datascience    0.2399
Name: proportion, dtype: float64

Split preparado correctamente.
IMPORTANTE: el test permanecerá sin utilizar hasta la evaluación final.


In [3]:
# ==============================================================================
# STOPWORDS CONTROLADAS — TECHMIND v1.1
# ==============================================================================

STOPWORDS_ES_BASE = {
    "a",
    "al",
    "algo",
    "alguna",
    "algunas",
    "alguno",
    "algunos",
    "ante",
    "antes",
    "cada",
    "como",
    "con",
    "cual",
    "cuando",
    "de",
    "del",
    "desde",
    "donde",
    "durante",
    "e",
    "el",
    "ella",
    "ellas",
    "ellos",
    "en",
    "entre",
    "era",
    "es",
    "esa",
    "esas",
    "ese",
    "esos",
    "esta",
    "estas",
    "este",
    "estos",
    "fue",
    "ha",
    "hay",
    "la",
    "las",
    "le",
    "les",
    "lo",
    "los",
    "más",
    "me",
    "mi",
    "mis",
    "mucho",
    "muy",
    "nos",
    "o",
    "otra",
    "otras",
    "otro",
    "otros",
    "para",
    "pero",
    "por",
    "porque",
    "que",
    "se",
    "sin",
    "sobre",
    "su",
    "sus",
    "también",
    "te",
    "todo",
    "todos",
    "tu",
    "tus",
    "un",
    "una",
    "unas",
    "uno",
    "unos",
    "y",
    "ya",
}


# ==============================================================================
# NORMALIZACIÓN DE STOPWORDS
# ==============================================================================

def normalizar_unicode(texto):
    texto = str(texto).lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    return texto


STOPWORDS_ES = sorted({
    normalizar_unicode(palabra)
    for palabra in STOPWORDS_ES_BASE
})


# ==============================================================================
# TERMINOLOGÍA QUE NUNCA DEBE SER STOPWORD
# ==============================================================================

TERMINOS_TECNICOS_PROTEGIDOS = {
    "api",
    "rest",
    "backend",
    "frontend",
    "cloud",
    "datascience",
    "data",
    "datos",
    "java",
    "spring",
    "boot",
    "python",
    "pandas",
    "docker",
    "kubernetes",
    "aws",
    "azure",
    "react",
    "javascript",
    "node",
    "sql",
    "tensorflow",
    "pytorch",
    "hibernate",
    "jpa",
    "maven",
}


interseccion = (
    set(STOPWORDS_ES)
    & TERMINOS_TECNICOS_PROTEGIDOS
)


if interseccion:
    raise ValueError(
        f"Términos técnicos detectados como stopwords: {interseccion}"
    )


print("Stopwords configuradas:", len(STOPWORDS_ES))

print("\nEjemplo:")
print(STOPWORDS_ES[:30])

print("\nTérminos técnicos protegidos:")
print(sorted(TERMINOS_TECNICOS_PROTEGIDOS))

print("\nConfiguración de stopwords aprobada.")

Stopwords configuradas: 80

Ejemplo:
['a', 'al', 'algo', 'alguna', 'algunas', 'alguno', 'algunos', 'ante', 'antes', 'cada', 'como', 'con', 'cual', 'cuando', 'de', 'del', 'desde', 'donde', 'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre', 'era', 'es', 'esa', 'esas']

Términos técnicos protegidos:
['api', 'aws', 'azure', 'backend', 'boot', 'cloud', 'data', 'datascience', 'datos', 'docker', 'frontend', 'hibernate', 'java', 'javascript', 'jpa', 'kubernetes', 'maven', 'node', 'pandas', 'python', 'pytorch', 'react', 'rest', 'spring', 'sql', 'tensorflow']

Configuración de stopwords aprobada.


In [4]:
# ==============================================================================
# AUDITORÍA DEL MODELO v1.0.0
# ==============================================================================

modelo_v1 = joblib.load(PATH_MODELO_V1)


print("Pipeline v1.0.0:")
print(modelo_v1)


if "tfidf" not in modelo_v1.named_steps:
    raise ValueError(
        "El pipeline no contiene el paso 'tfidf'."
    )

if "clasificador" not in modelo_v1.named_steps:
    raise ValueError(
        "El pipeline no contiene el paso 'clasificador'."
    )


tfidf_v1 = modelo_v1.named_steps["tfidf"]
clasificador_v1 = modelo_v1.named_steps["clasificador"]


features_v1 = np.asarray(
    tfidf_v1.get_feature_names_out()
)

clases_v1 = np.asarray(
    clasificador_v1.classes_
)

coeficientes_v1 = np.asarray(
    clasificador_v1.coef_
)


print("\nClases:")
print(clases_v1)

print("\nCaracterísticas TF-IDF:")
print(len(features_v1))


# ==============================================================================
# TOP TÉRMINOS POSITIVOS POR CLASE
# ==============================================================================

registros_top = []

TOP_N = 100


for indice_clase, categoria in enumerate(clases_v1):

    orden = np.argsort(
        coeficientes_v1[indice_clase]
    )[::-1][:TOP_N]

    for ranking, feature_idx in enumerate(orden, 1):

        termino = features_v1[feature_idx]

        registros_top.append({
            "categoria": categoria,
            "ranking": ranking,
            "termino": termino,
            "coeficiente": float(
                coeficientes_v1[indice_clase, feature_idx]
            ),
            "es_stopword": termino in STOPWORDS_ES,
        })


df_top_v1 = pd.DataFrame(registros_top)


# ==============================================================================
# CONTAMINACIÓN POR STOPWORDS
# ==============================================================================

df_stopwords_top_v1 = (
    df_top_v1[
        df_top_v1["es_stopword"]
    ]
    .copy()
)


resumen_stopwords_v1 = (
    df_stopwords_top_v1
    .groupby("categoria")
    .size()
    .rename("stopwords_top_100")
    .reset_index()
)


print("\nStopwords entre los 100 términos más positivos:")
print(resumen_stopwords_v1)


print("\nStopwords aprendidas como señales:")
print(
    df_stopwords_top_v1[
        [
            "categoria",
            "ranking",
            "termino",
            "coeficiente"
        ]
    ]
    .sort_values(
        ["categoria", "ranking"]
    )
    .head(50)
)


# ==============================================================================
# INSPECCIÓN DE LOS TÉRMINOS ENCONTRADOS POR BACKEND
# ==============================================================================

terminos_problematicos = [
    "de",
    "para",
    "con",
    "un",
    "como",
    "a",
    "y",
    "el",
    "una",
]


registros_problematicos = []


for termino in terminos_problematicos:

    posiciones = np.where(
        features_v1 == termino
    )[0]

    if len(posiciones) == 0:
        continue

    idx = posiciones[0]

    for indice_clase, categoria in enumerate(clases_v1):

        registros_problematicos.append({
            "termino": termino,
            "categoria": categoria,
            "coeficiente": float(
                coeficientes_v1[indice_clase, idx]
            ),
        })


df_terminos_problematicos = pd.DataFrame(
    registros_problematicos
)


print("\nCoeficientes de términos problemáticos:")
print(
    df_terminos_problematicos
    .sort_values(
        ["termino", "coeficiente"],
        ascending=[True, False]
    )
)


# ==============================================================================
# EXPORTAR AUDITORÍA
# ==============================================================================

df_top_v1.to_csv(
    REPORTS_V11 / "auditoria_features_v1_0_0.csv",
    index=False
)

df_terminos_problematicos.to_csv(
    REPORTS_V11 / "terminos_problematicos_v1_0_0.csv",
    index=False
)


print("\nAuditoría v1.0.0 completada.")

Pipeline v1.0.0:
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(lowercase=False, max_df=0.95,
                                 max_features=30000, ngram_range=(1, 2),
                                 sublinear_tf=True,
                                 token_pattern='(?u)(?:\\b\\w[\\w.-]*\\+\\+|\\b\\w[\\w.-]*#|\\b\\w+(?:/\\w+)+\\b|\\.net\\b|\\b\\w[\\w.-]*\\b)')),
                ('clasificador',
                 SGDClassifier(alpha=3e-05, max_iter=3000, random_state=42))])

Clases:
['backend' 'cloud' 'datascience' 'frontend']

Características TF-IDF:
30000

Stopwords entre los 100 términos más positivos:
Empty DataFrame
Columns: [categoria, stopwords_top_100]
Index: []

Stopwords aprendidas como señales:
Empty DataFrame
Columns: [categoria, ranking, termino, coeficiente]
Index: []

Coeficientes de términos problemáticos:
   termino    categoria  coeficiente
20       a      backend     0.741006
23       a     frontend     0.114184
21       a        cloud    -0.353787
22      

In [5]:
# ==============================================================================
# CONSTRUCCIÓN DE CANDIDATOS v1.1
# ==============================================================================

parametros_tfidf_v1 = (
    tfidf_v1
    .get_params(deep=False)
    .copy()
)


def crear_vectorizador(**cambios):

    parametros = parametros_tfidf_v1.copy()

    parametros.update(cambios)

    return TfidfVectorizer(
        **parametros
    )


def crear_pipeline(vectorizador):

    return Pipeline([
        (
            "tfidf",
            vectorizador
        ),
        (
            "clasificador",
            clone(clasificador_v1)
        ),
    ])


# ------------------------------------------------------------------------------
# CONTROL
# ------------------------------------------------------------------------------

pipeline_control = clone(modelo_v1)


# ------------------------------------------------------------------------------
# CANDIDATO A — SOLO NORMALIZACIÓN DE ACENTOS
# ------------------------------------------------------------------------------

pipeline_acentos = crear_pipeline(
    crear_vectorizador(
        strip_accents="unicode",
    )
)


# ------------------------------------------------------------------------------
# CANDIDATO B — ACENTOS + STOPWORDS
# ------------------------------------------------------------------------------

pipeline_stopwords = crear_pipeline(
    crear_vectorizador(
        strip_accents="unicode",
        stop_words=STOPWORDS_ES,
    )
)


# ------------------------------------------------------------------------------
# CANDIDATO C — STOPWORDS + min_df=2
# ------------------------------------------------------------------------------

pipeline_stopwords_df2 = crear_pipeline(
    crear_vectorizador(
        strip_accents="unicode",
        stop_words=STOPWORDS_ES,
        min_df=2,
    )
)


# ------------------------------------------------------------------------------
# CANDIDATO D — STOPWORDS + min_df=3
# ------------------------------------------------------------------------------

pipeline_stopwords_df3 = crear_pipeline(
    crear_vectorizador(
        strip_accents="unicode",
        stop_words=STOPWORDS_ES,
        min_df=3,
    )
)


CANDIDATOS_V11 = {
    "v1_control": pipeline_control,
    "acentos": pipeline_acentos,
    "stopwords_acentos": pipeline_stopwords,
    "stopwords_acentos_min_df2": pipeline_stopwords_df2,
    "stopwords_acentos_min_df3": pipeline_stopwords_df3,
}


print("Candidatos:")
for nombre in CANDIDATOS_V11:
    print(" -", nombre)

Candidatos:
 - v1_control
 - acentos
 - stopwords_acentos
 - stopwords_acentos_min_df2
 - stopwords_acentos_min_df3


In [6]:
# ==============================================================================
# VALIDACIÓN CRUZADA DE CANDIDATOS — TECHMIND v1.1.0
# ==============================================================================

# Windows + ruta con caracteres Unicode (ej. "MAMÁ"):
# evitamos multiprocessing de joblib.
N_JOBS = 1


SCORING = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}


resultados_cv_v11 = []


for nombre, pipeline in CANDIDATOS_V11.items():

    print("\n" + "=" * 80)
    print("Evaluando:", nombre)
    print("=" * 80)

    resultado = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_estratificada,
        scoring=SCORING,
        return_train_score=True,
        n_jobs=N_JOBS,
        error_score="raise",
    )

    registro = {
        "modelo": nombre,

        "accuracy_cv": np.mean(
            resultado["test_accuracy"]
        ),

        "precision_macro_cv": np.mean(
            resultado["test_precision_macro"]
        ),

        "recall_macro_cv": np.mean(
            resultado["test_recall_macro"]
        ),

        "f1_macro_cv": np.mean(
            resultado["test_f1_macro"]
        ),

        "f1_macro_std": np.std(
            resultado["test_f1_macro"]
        ),

        "f1_weighted_cv": np.mean(
            resultado["test_f1_weighted"]
        ),

        "f1_train": np.mean(
            resultado["train_f1_macro"]
        ),
    }

    registro["brecha_train_cv"] = (
        registro["f1_train"]
        - registro["f1_macro_cv"]
    )

    resultados_cv_v11.append(
        registro
    )

    print(
        f"F1 Macro CV: "
        f"{registro['f1_macro_cv']:.4f}"
    )

    print(
        f"Std: "
        f"{registro['f1_macro_std']:.4f}"
    )

    print(
        f"F1 Train: "
        f"{registro['f1_train']:.4f}"
    )

    print(
        f"Brecha train-CV: "
        f"{registro['brecha_train_cv']:.4f}"
    )


# ==============================================================================
# TABLA COMPARATIVA
# ==============================================================================

df_resultados_cv_v11 = pd.DataFrame(
    resultados_cv_v11
)

df_resultados_cv_v11 = (
    df_resultados_cv_v11
    .sort_values(
        "f1_macro_cv",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("COMPARACIÓN v1.1")
print("=" * 80)

print(
    df_resultados_cv_v11[
        [
            "modelo",
            "accuracy_cv",
            "f1_macro_cv",
            "f1_macro_std",
            "f1_train",
            "brecha_train_cv",
        ]
    ]
    .round(4)
)


# ==============================================================================
# COMPARACIÓN CONTRA CONTROL v1.0
# ==============================================================================

F1_CONTROL = float(
    df_resultados_cv_v11.loc[
        df_resultados_cv_v11["modelo"] == "v1_control",
        "f1_macro_cv"
    ].iloc[0]
)


df_resultados_cv_v11["diferencia_vs_v1"] = (
    df_resultados_cv_v11["f1_macro_cv"]
    - F1_CONTROL
)


print("\nComparación respecto al control v1.0:")
print(
    df_resultados_cv_v11[
        [
            "modelo",
            "f1_macro_cv",
            "f1_macro_std",
            "diferencia_vs_v1",
        ]
    ]
    .round(4)
)


# ==============================================================================
# MEJOR CANDIDATO
# ==============================================================================

mejor_fila_v11 = df_resultados_cv_v11.iloc[0]

MEJOR_CANDIDATO_V11_NOMBRE = (
    mejor_fila_v11["modelo"]
)

MEJOR_F1_V11 = float(
    mejor_fila_v11["f1_macro_cv"]
)

MEJOR_STD_V11 = float(
    mejor_fila_v11["f1_macro_std"]
)

MEJOR_PIPELINE_V11 = CANDIDATOS_V11[
    MEJOR_CANDIDATO_V11_NOMBRE
]


print("\n" + "=" * 80)
print("MEJOR CANDIDATO v1.1")
print("=" * 80)

print(
    "Modelo:",
    MEJOR_CANDIDATO_V11_NOMBRE
)

print(
    f"F1 Macro CV: {MEJOR_F1_V11:.4f}"
)

print(
    f"Std: {MEJOR_STD_V11:.4f}"
)

print(
    f"Diferencia vs control: "
    f"{MEJOR_F1_V11 - F1_CONTROL:+.4f}"
)


# ==============================================================================
# EXPORTACIÓN
# ==============================================================================

PATH_COMPARACION_V11 = (
    REPORTS_V11
    / "comparacion_candidatos_cv.csv"
)

df_resultados_cv_v11.to_csv(
    PATH_COMPARACION_V11,
    index=False,
    encoding="utf-8-sig",
)


print("\nResultados exportados:")
print(PATH_COMPARACION_V11)

print("\nValidación cruzada v1.1 completada.")


Evaluando: v1_control
F1 Macro CV: 0.8432
Std: 0.0116
F1 Train: 1.0000
Brecha train-CV: 0.1568

Evaluando: acentos
F1 Macro CV: 0.8416
Std: 0.0097
F1 Train: 1.0000
Brecha train-CV: 0.1584

Evaluando: stopwords_acentos
F1 Macro CV: 0.8449
Std: 0.0113
F1 Train: 1.0000
Brecha train-CV: 0.1551

Evaluando: stopwords_acentos_min_df2
F1 Macro CV: 0.8279
Std: 0.0131
F1 Train: 1.0000
Brecha train-CV: 0.1721

Evaluando: stopwords_acentos_min_df3
F1 Macro CV: 0.8118
Std: 0.0128
F1 Train: 0.9996
Brecha train-CV: 0.1878

COMPARACIÓN v1.1
                      modelo  accuracy_cv  f1_macro_cv  f1_macro_std  \
0          stopwords_acentos       0.8437       0.8449        0.0113   
1                 v1_control       0.8418       0.8432        0.0116   
2                    acentos       0.8402       0.8416        0.0097   
3  stopwords_acentos_min_df2       0.8271       0.8279        0.0131   
4  stopwords_acentos_min_df3       0.8110       0.8118        0.0128   

   f1_train  brecha_train_cv  
0   

In [7]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 7 — AUDITORÍA DE VOCABULARIO Y PRUEBAS REALES
# ==============================================================================

from sklearn.base import clone

import json
import re

import numpy as np
import pandas as pd


# ==============================================================================
# 7.1 CONFIGURACIÓN
# ==============================================================================

NOMBRE_MODELO_CONTROL = "v1_control"
NOMBRE_MODELO_V11 = "stopwords_acentos"

TOP_N_AUDITORIA = 100
TOP_N_EXPLICACION = 12


if NOMBRE_MODELO_CONTROL not in CANDIDATOS_V11:
    raise KeyError(
        f"No existe el candidato {NOMBRE_MODELO_CONTROL}."
    )

if NOMBRE_MODELO_V11 not in CANDIDATOS_V11:
    raise KeyError(
        f"No existe el candidato {NOMBRE_MODELO_V11}."
    )


# ==============================================================================
# 7.2 ENTRENAR SOLO CON TRAIN
# ==============================================================================

modelo_control_auditado = clone(
    CANDIDATOS_V11[NOMBRE_MODELO_CONTROL]
)

modelo_v11_auditado = clone(
    CANDIDATOS_V11[NOMBRE_MODELO_V11]
)


print("Entrenando control v1.0.0...")

modelo_control_auditado.fit(
    X_train,
    y_train
)


print("Entrenando candidato v1.1.0...")

modelo_v11_auditado.fit(
    X_train,
    y_train
)


print("\nModelos entrenados únicamente con train.")
print("El test reservado continúa sin utilizarse.")

Entrenando control v1.0.0...
Entrenando candidato v1.1.0...

Modelos entrenados únicamente con train.
El test reservado continúa sin utilizarse.


In [8]:
# ==============================================================================
# 7.3 FUNCIONES DE AUDITORÍA
# ==============================================================================

def obtener_componentes_pipeline(modelo):

    if "tfidf" not in modelo.named_steps:
        raise KeyError(
            "El pipeline no contiene el paso 'tfidf'."
        )

    if "clasificador" not in modelo.named_steps:
        raise KeyError(
            "El pipeline no contiene el paso 'clasificador'."
        )

    vectorizador = modelo.named_steps["tfidf"]
    clasificador = modelo.named_steps["clasificador"]

    features = np.asarray(
        vectorizador.get_feature_names_out()
    )

    clases = np.asarray(
        clasificador.classes_
    )

    coeficientes = np.asarray(
        clasificador.coef_
    )

    return (
        vectorizador,
        clasificador,
        features,
        clases,
        coeficientes,
    )


def auditar_modelo_vocabulario(
    nombre_modelo,
    modelo,
    top_n=100,
):

    (
        vectorizador,
        clasificador,
        features,
        clases,
        coeficientes,
    ) = obtener_componentes_pipeline(modelo)

    vocabulario = set(features)

    stopwords_en_vocabulario = sorted(
        vocabulario.intersection(
            set(STOPWORDS_ES)
        )
    )

    terminos_tecnicos_presentes = sorted(
        vocabulario.intersection(
            TERMINOS_TECNICOS_PROTEGIDOS
        )
    )

    terminos_tecnicos_ausentes = sorted(
        TERMINOS_TECNICOS_PROTEGIDOS
        - vocabulario
    )

    registros_top = []

    for indice_clase, categoria in enumerate(clases):

        orden = np.argsort(
            coeficientes[indice_clase]
        )[::-1][:top_n]

        for ranking, feature_idx in enumerate(
            orden,
            start=1,
        ):

            termino = str(
                features[feature_idx]
            )

            registros_top.append({
                "modelo": nombre_modelo,
                "categoria": str(categoria),
                "ranking": ranking,
                "termino": termino,
                "coeficiente": float(
                    coeficientes[
                        indice_clase,
                        feature_idx
                    ]
                ),
                "es_stopword": (
                    termino in STOPWORDS_ES
                ),
                "es_tecnico_protegido": (
                    termino
                    in TERMINOS_TECNICOS_PROTEGIDOS
                ),
            })

    df_top = pd.DataFrame(
        registros_top
    )

    stopwords_top = int(
        df_top["es_stopword"].sum()
    )

    resumen = {
        "modelo": nombre_modelo,
        "cantidad_features": int(
            len(features)
        ),
        "cantidad_clases": int(
            len(clases)
        ),
        "stopwords_en_vocabulario": int(
            len(stopwords_en_vocabulario)
        ),
        "stopwords_top_features": stopwords_top,
        "terminos_tecnicos_presentes": int(
            len(terminos_tecnicos_presentes)
        ),
        "terminos_tecnicos_ausentes": int(
            len(terminos_tecnicos_ausentes)
        ),
        "lista_stopwords_vocabulario": (
            stopwords_en_vocabulario
        ),
        "lista_tecnicos_presentes": (
            terminos_tecnicos_presentes
        ),
        "lista_tecnicos_ausentes": (
            terminos_tecnicos_ausentes
        ),
    }

    return resumen, df_top

In [9]:
# ==============================================================================
# 7.4 AUDITORÍA COMPARATIVA
# ==============================================================================

resumen_vocabulario_control, df_top_control = (
    auditar_modelo_vocabulario(
        nombre_modelo="v1.0.0_control",
        modelo=modelo_control_auditado,
        top_n=TOP_N_AUDITORIA,
    )
)


resumen_vocabulario_v11, df_top_v11 = (
    auditar_modelo_vocabulario(
        nombre_modelo="v1.1.0_stopwords_acentos",
        modelo=modelo_v11_auditado,
        top_n=TOP_N_AUDITORIA,
    )
)


df_resumen_vocabulario = pd.DataFrame([
    {
        "modelo": resumen_vocabulario_control["modelo"],
        "cantidad_features": (
            resumen_vocabulario_control[
                "cantidad_features"
            ]
        ),
        "stopwords_en_vocabulario": (
            resumen_vocabulario_control[
                "stopwords_en_vocabulario"
            ]
        ),
        "stopwords_top_100_por_clase": (
            resumen_vocabulario_control[
                "stopwords_top_features"
            ]
        ),
        "terminos_tecnicos_presentes": (
            resumen_vocabulario_control[
                "terminos_tecnicos_presentes"
            ]
        ),
        "terminos_tecnicos_ausentes": (
            resumen_vocabulario_control[
                "terminos_tecnicos_ausentes"
            ]
        ),
    },
    {
        "modelo": resumen_vocabulario_v11["modelo"],
        "cantidad_features": (
            resumen_vocabulario_v11[
                "cantidad_features"
            ]
        ),
        "stopwords_en_vocabulario": (
            resumen_vocabulario_v11[
                "stopwords_en_vocabulario"
            ]
        ),
        "stopwords_top_100_por_clase": (
            resumen_vocabulario_v11[
                "stopwords_top_features"
            ]
        ),
        "terminos_tecnicos_presentes": (
            resumen_vocabulario_v11[
                "terminos_tecnicos_presentes"
            ]
        ),
        "terminos_tecnicos_ausentes": (
            resumen_vocabulario_v11[
                "terminos_tecnicos_ausentes"
            ]
        ),
    },
])


print("=" * 80)
print("AUDITORÍA DE VOCABULARIO")
print("=" * 80)

print(
    df_resumen_vocabulario
)


print("\nStopwords presentes en v1.0.0:")

print(
    resumen_vocabulario_control[
        "lista_stopwords_vocabulario"
    ][:50]
)


print("\nStopwords presentes en v1.1.0:")

print(
    resumen_vocabulario_v11[
        "lista_stopwords_vocabulario"
    ][:50]
)


print("\nTérminos técnicos ausentes en v1.1.0:")

print(
    resumen_vocabulario_v11[
        "lista_tecnicos_ausentes"
    ]
)

AUDITORÍA DE VOCABULARIO
                     modelo  cantidad_features  stopwords_en_vocabulario  \
0            v1.0.0_control              30000                        42   
1  v1.1.0_stopwords_acentos              30000                         0   

   stopwords_top_100_por_clase  terminos_tecnicos_presentes  \
0                            0                           22   
1                            0                           22   

   terminos_tecnicos_ausentes  
0                           4  
1                           4  

Stopwords presentes en v1.0.0:
['a', 'alguna', 'antes', 'cada', 'como', 'con', 'cuando', 'de', 'del', 'desde', 'durante', 'e', 'el', 'en', 'entre', 'era', 'es', 'hay', 'la', 'las', 'le', 'les', 'los', 'me', 'mi', 'o', 'para', 'pero', 'por', 'que', 'se', 'sin', 'sobre', 'te', 'todo', 'todos', 'tu', 'tus', 'un', 'una', 'y', 'ya']

Stopwords presentes en v1.1.0:
[]

Términos técnicos ausentes en v1.1.0:
['datascience', 'hibernate', 'jpa', 'tensorflow']


In [10]:
# ==============================================================================
# 7.5 CASOS EXTERNOS DE DIAGNÓSTICO
# ==============================================================================

CASOS_DIAGNOSTICO = [
    {
        "caso_id": "backend_real_01",
        "categoria_esperada": "backend",
        "texto": (
            "Spring Boot es un framework de Java para desarrollar "
            "aplicaciones backend. Permite crear servicios REST, "
            "controladores, inyección de dependencias, seguridad con "
            "Spring Security, acceso a bases de datos mediante JPA y "
            "Hibernate, utilizando Maven como gestor de dependencias."
        ),
    },
    {
        "caso_id": "backend_real_02",
        "categoria_esperada": "backend",
        "texto": (
            "Este contenido explica cómo crear una API REST con "
            "Spring Boot y Java, incluyendo el uso de controladores, "
            "servicios y repositorios."
        ),
    },
    {
        "caso_id": "backend_adicional",
        "categoria_esperada": "backend",
        "texto": (
            "Desarrollo de microservicios con Java, Spring Boot, "
            "Hibernate, JPA, Maven y endpoints REST."
        ),
    },
    {
        "caso_id": "cloud_01",
        "categoria_esperada": "cloud",
        "texto": (
            "Despliegue de infraestructura en AWS utilizando "
            "Terraform, Kubernetes, Docker, balanceadores y "
            "servicios administrados en la nube."
        ),
    },
    {
        "caso_id": "cloud_02",
        "categoria_esperada": "cloud",
        "texto": (
            "Configuración de máquinas virtuales, redes privadas, "
            "contenedores y almacenamiento en Oracle Cloud."
        ),
    },
    {
        "caso_id": "datascience_01",
        "categoria_esperada": "datascience",
        "texto": (
            "Análisis de datos con Python, pandas, NumPy y "
            "scikit-learn para entrenar un modelo de clasificación."
        ),
    },
    {
        "caso_id": "datascience_02",
        "categoria_esperada": "datascience",
        "texto": (
            "Entrenamiento de modelos de machine learning, "
            "validación cruzada y evaluación mediante F1 score."
        ),
    },
    {
        "caso_id": "frontend_01",
        "categoria_esperada": "frontend",
        "texto": (
            "Creación de una interfaz responsive con React, "
            "JavaScript, HTML y CSS."
        ),
    },
    {
        "caso_id": "frontend_02",
        "categoria_esperada": "frontend",
        "texto": (
            "Desarrollo de componentes visuales, formularios y "
            "navegación del lado del cliente."
        ),
    },
]


df_casos_diagnostico = pd.DataFrame(
    CASOS_DIAGNOSTICO
)


print("Casos externos:", len(df_casos_diagnostico))

print(
    df_casos_diagnostico[
        [
            "caso_id",
            "categoria_esperada",
        ]
    ]
)

Casos externos: 9
             caso_id categoria_esperada
0    backend_real_01            backend
1    backend_real_02            backend
2  backend_adicional            backend
3           cloud_01              cloud
4           cloud_02              cloud
5     datascience_01        datascience
6     datascience_02        datascience
7        frontend_01           frontend
8        frontend_02           frontend


In [11]:
# ==============================================================================
# 7.6 EVALUACIÓN DE CASOS EXTERNOS
# ==============================================================================

def evaluar_casos_externos(
    nombre_modelo,
    modelo,
    df_casos,
):

    (
        vectorizador,
        clasificador,
        features,
        clases,
        coeficientes,
    ) = obtener_componentes_pipeline(modelo)

    textos = (
        df_casos["texto"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    matriz_tfidf = vectorizador.transform(
        textos
    )

    predicciones = modelo.predict(
        textos
    )

    puntuaciones = modelo.decision_function(
        textos
    )

    puntuaciones = np.asarray(
        puntuaciones
    )

    if puntuaciones.ndim == 1:
        puntuaciones = np.column_stack([
            -puntuaciones,
            puntuaciones,
        ])

    registros = []

    for posicion in range(len(textos)):

        scores = puntuaciones[posicion]

        orden = np.argsort(
            scores
        )[::-1]

        indice_ganador = int(
            orden[0]
        )

        indice_segundo = int(
            orden[1]
        )

        categoria_predicha = str(
            clases[indice_ganador]
        )

        segunda_categoria = str(
            clases[indice_segundo]
        )

        margen = float(
            scores[indice_ganador]
            - scores[indice_segundo]
        )

        terminos_activos = int(
            matriz_tfidf[posicion].getnnz()
        )

        categoria_esperada = str(
            df_casos.iloc[posicion][
                "categoria_esperada"
            ]
        )

        registros.append({
            "modelo": nombre_modelo,
            "caso_id": (
                df_casos.iloc[posicion][
                    "caso_id"
                ]
            ),
            "categoria_esperada": (
                categoria_esperada
            ),
            "categoria_predicha": (
                categoria_predicha
            ),
            "segunda_categoria": (
                segunda_categoria
            ),
            "prediccion_correcta": (
                categoria_predicha
                == categoria_esperada
            ),
            "puntuacion_ganadora": float(
                scores[indice_ganador]
            ),
            "puntuacion_segunda": float(
                scores[indice_segundo]
            ),
            "margen_decision": margen,
            "terminos_activos": (
                terminos_activos
            ),
            "texto": textos[posicion],
        })

    return pd.DataFrame(
        registros
    )

In [12]:
df_diagnostico_control = evaluar_casos_externos(
    nombre_modelo="v1.0.0_control",
    modelo=modelo_control_auditado,
    df_casos=df_casos_diagnostico,
)


df_diagnostico_v11 = evaluar_casos_externos(
    nombre_modelo="v1.1.0_stopwords_acentos",
    modelo=modelo_v11_auditado,
    df_casos=df_casos_diagnostico,
)


df_diagnostico_completo = pd.concat(
    [
        df_diagnostico_control,
        df_diagnostico_v11,
    ],
    ignore_index=True,
)


print("=" * 80)
print("RESULTADOS DE CASOS EXTERNOS")
print("=" * 80)

print(
    df_diagnostico_completo[
        [
            "modelo",
            "caso_id",
            "categoria_esperada",
            "categoria_predicha",
            "segunda_categoria",
            "prediccion_correcta",
            "margen_decision",
            "terminos_activos",
        ]
    ]
    .round(4)
)

RESULTADOS DE CASOS EXTERNOS
                      modelo            caso_id categoria_esperada  \
0             v1.0.0_control    backend_real_01            backend   
1             v1.0.0_control    backend_real_02            backend   
2             v1.0.0_control  backend_adicional            backend   
3             v1.0.0_control           cloud_01              cloud   
4             v1.0.0_control           cloud_02              cloud   
5             v1.0.0_control     datascience_01        datascience   
6             v1.0.0_control     datascience_02        datascience   
7             v1.0.0_control        frontend_01           frontend   
8             v1.0.0_control        frontend_02           frontend   
9   v1.1.0_stopwords_acentos    backend_real_01            backend   
10  v1.1.0_stopwords_acentos    backend_real_02            backend   
11  v1.1.0_stopwords_acentos  backend_adicional            backend   
12  v1.1.0_stopwords_acentos           cloud_01              

In [13]:
# ==============================================================================
# 7.7 COMPARACIÓN DIRECTA
# ==============================================================================

columnas_comparacion = [
    "caso_id",
    "categoria_esperada",
    "categoria_predicha",
    "segunda_categoria",
    "prediccion_correcta",
    "margen_decision",
    "terminos_activos",
]


df_control_comparacion = (
    df_diagnostico_control[
        columnas_comparacion
    ]
    .rename(columns={
        "categoria_predicha": (
            "prediccion_v1_0"
        ),
        "segunda_categoria": (
            "segunda_v1_0"
        ),
        "prediccion_correcta": (
            "correcta_v1_0"
        ),
        "margen_decision": (
            "margen_v1_0"
        ),
        "terminos_activos": (
            "terminos_activos_v1_0"
        ),
    })
)


df_v11_comparacion = (
    df_diagnostico_v11[
        [
            "caso_id",
            "categoria_predicha",
            "segunda_categoria",
            "prediccion_correcta",
            "margen_decision",
            "terminos_activos",
        ]
    ]
    .rename(columns={
        "categoria_predicha": (
            "prediccion_v1_1"
        ),
        "segunda_categoria": (
            "segunda_v1_1"
        ),
        "prediccion_correcta": (
            "correcta_v1_1"
        ),
        "margen_decision": (
            "margen_v1_1"
        ),
        "terminos_activos": (
            "terminos_activos_v1_1"
        ),
    })
)


df_comparacion_diagnostico = (
    df_control_comparacion
    .merge(
        df_v11_comparacion,
        on="caso_id",
        how="inner",
    )
)


df_comparacion_diagnostico[
    "cambio_margen"
] = (
    df_comparacion_diagnostico[
        "margen_v1_1"
    ]
    - df_comparacion_diagnostico[
        "margen_v1_0"
    ]
)


df_comparacion_diagnostico[
    "mejora_clasificacion"
] = (
    ~df_comparacion_diagnostico[
        "correcta_v1_0"
    ]
    & df_comparacion_diagnostico[
        "correcta_v1_1"
    ]
)


df_comparacion_diagnostico[
    "empeora_clasificacion"
] = (
    df_comparacion_diagnostico[
        "correcta_v1_0"
    ]
    & ~df_comparacion_diagnostico[
        "correcta_v1_1"
    ]
)


print("=" * 80)
print("COMPARACIÓN DIRECTA")
print("=" * 80)

print(
    df_comparacion_diagnostico[
        [
            "caso_id",
            "categoria_esperada",
            "prediccion_v1_0",
            "prediccion_v1_1",
            "correcta_v1_0",
            "correcta_v1_1",
            "margen_v1_0",
            "margen_v1_1",
            "cambio_margen",
        ]
    ]
    .round(4)
)

COMPARACIÓN DIRECTA
             caso_id categoria_esperada prediccion_v1_0 prediccion_v1_1  \
0    backend_real_01            backend         backend         backend   
1    backend_real_02            backend           cloud        frontend   
2  backend_adicional            backend         backend         backend   
3           cloud_01              cloud           cloud           cloud   
4           cloud_02              cloud           cloud        frontend   
5     datascience_01        datascience     datascience     datascience   
6     datascience_02        datascience     datascience     datascience   
7        frontend_01           frontend           cloud        frontend   
8        frontend_02           frontend     datascience        frontend   

   correcta_v1_0  correcta_v1_1  margen_v1_0  margen_v1_1  cambio_margen  
0           True           True       1.9676       2.1515         0.1839  
1          False          False       0.0660       0.3240         0.2579  
2   

In [14]:
# ==============================================================================
# 7.8 EXPLICACIÓN MATEMÁTICA DE UNA PREDICCIÓN
# ==============================================================================

def explicar_prediccion_auditada(
    modelo,
    texto,
    top_n=12,
):

    (
        vectorizador,
        clasificador,
        features,
        clases,
        coeficientes,
    ) = obtener_componentes_pipeline(
        modelo
    )

    # --------------------------------------------------------------------------
    # TRANSFORMAR TEXTO
    # --------------------------------------------------------------------------

    matriz = vectorizador.transform([
        texto
    ])

    scores = modelo.decision_function([
        texto
    ])

    scores = np.asarray(
        scores
    ).reshape(-1)

    # --------------------------------------------------------------------------
    # IDENTIFICAR LAS DOS CATEGORÍAS PRINCIPALES
    # --------------------------------------------------------------------------

    orden_clases = np.argsort(
        scores
    )[::-1]

    indice_ganador = int(
        orden_clases[0]
    )

    indice_segundo = int(
        orden_clases[1]
    )

    categoria_ganadora = str(
        clases[indice_ganador]
    )

    segunda_categoria = str(
        clases[indice_segundo]
    )

    margen = float(
        scores[indice_ganador]
        - scores[indice_segundo]
    )

    # --------------------------------------------------------------------------
    # CARACTERÍSTICAS ACTIVAS
    # --------------------------------------------------------------------------

    fila = matriz.getrow(0)

    indices_activos = fila.indices
    valores_tfidf = fila.data

    # --------------------------------------------------------------------------
    # CASO SIN COBERTURA DE VOCABULARIO
    # --------------------------------------------------------------------------

    columnas_explicacion = [
        "termino",
        "tfidf",
        "contribucion_ganadora",
        "contribucion_diferencial",
    ]

    if fila.getnnz() == 0:

        df_vacio = pd.DataFrame(
            columns=columnas_explicacion
        )

        return {
            "categoria_ganadora": categoria_ganadora,
            "segunda_categoria": segunda_categoria,
            "margen": margen,
            "terminos_activos": 0,
            "terminos_positivos": df_vacio.copy(),
            "terminos_diferenciales": df_vacio.copy(),
            "sin_cobertura_vocabulario": True,
            "prediccion_utilizable": False,
        }

    # --------------------------------------------------------------------------
    # CONTRIBUCIÓN PARA LA CATEGORÍA GANADORA
    # --------------------------------------------------------------------------

    contribuciones_ganador = (
        valores_tfidf
        * coeficientes[
            indice_ganador,
            indices_activos
        ]
    )

    # --------------------------------------------------------------------------
    # CONTRIBUCIÓN DIFERENCIAL GANADOR VS SEGUNDO
    # --------------------------------------------------------------------------

    coeficiente_diferencial = (
        coeficientes[indice_ganador]
        - coeficientes[indice_segundo]
    )

    contribuciones_diferenciales = (
        valores_tfidf
        * coeficiente_diferencial[
            indices_activos
        ]
    )

    # --------------------------------------------------------------------------
    # TABLA DE EXPLICACIÓN
    # --------------------------------------------------------------------------

    registros = []

    for posicion_local, feature_idx in enumerate(
        indices_activos
    ):

        registros.append({
            "termino": str(
                features[feature_idx]
            ),
            "tfidf": float(
                valores_tfidf[
                    posicion_local
                ]
            ),
            "contribucion_ganadora": float(
                contribuciones_ganador[
                    posicion_local
                ]
            ),
            "contribucion_diferencial": float(
                contribuciones_diferenciales[
                    posicion_local
                ]
            ),
        })

    df_explicacion = pd.DataFrame(
        registros,
        columns=columnas_explicacion,
    )

    # --------------------------------------------------------------------------
    # TÉRMINOS POSITIVOS
    # --------------------------------------------------------------------------

    terminos_positivos = (
        df_explicacion[
            df_explicacion[
                "contribucion_ganadora"
            ] > 0
        ]
        .sort_values(
            "contribucion_ganadora",
            ascending=False,
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------------------------
    # TÉRMINOS DIFERENCIALES
    # --------------------------------------------------------------------------

    terminos_diferenciales = (
        df_explicacion
        .sort_values(
            "contribucion_diferencial",
            ascending=False,
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------------------------
    # RESULTADO
    # --------------------------------------------------------------------------

    return {
        "categoria_ganadora": categoria_ganadora,
        "segunda_categoria": segunda_categoria,
        "margen": margen,
        "terminos_activos": int(
            fila.getnnz()
        ),
        "terminos_positivos": terminos_positivos,
        "terminos_diferenciales": terminos_diferenciales,
        "sin_cobertura_vocabulario": False,
        "prediccion_utilizable": True,
    }

In [15]:
# ==============================================================================
# EXPLICAR LOS DOS CASOS REALES DE BACKEND
# ==============================================================================

registros_explicaciones_v11 = []


for caso_id in [
    "backend_real_01",
    "backend_real_02",
]:

    fila_caso = (
        df_casos_diagnostico[
            df_casos_diagnostico[
                "caso_id"
            ] == caso_id
        ]
        .iloc[0]
    )

    explicacion = explicar_prediccion_auditada(
        modelo=modelo_v11_auditado,
        texto=fila_caso["texto"],
        top_n=TOP_N_EXPLICACION,
    )

    print("\n" + "=" * 80)
    print("CASO:", caso_id)
    print("=" * 80)

    print(
        "Categoría ganadora:",
        explicacion[
            "categoria_ganadora"
        ]
    )

    print(
        "Segunda categoría:",
        explicacion[
            "segunda_categoria"
        ]
    )

    print(
        "Margen:",
        round(
            explicacion["margen"],
            4
        )
    )

    print(
        "Términos activos:",
        explicacion[
            "terminos_activos"
        ]
    )

    print(
        "Sin cobertura de vocabulario:",
        explicacion[
            "sin_cobertura_vocabulario"
        ]
    )

    print(
        "Predicción utilizable:",
        explicacion[
            "prediccion_utilizable"
        ]
    )

    # --------------------------------------------------------------------------
    # TÉRMINOS POSITIVOS
    # --------------------------------------------------------------------------

    print("\nTérminos positivos:")

    df_positivos = explicacion[
        "terminos_positivos"
    ]

    if df_positivos.empty:

        print(
            "No existen términos positivos: "
            "el texto no activó características conocidas."
        )

    else:

        print(
            df_positivos[
                [
                    "termino",
                    "tfidf",
                    "contribucion_ganadora",
                ]
            ]
            .round(6)
        )

    # --------------------------------------------------------------------------
    # TÉRMINOS DIFERENCIALES
    # --------------------------------------------------------------------------

    print("\nTérminos diferenciales:")

    df_diferenciales = explicacion[
        "terminos_diferenciales"
    ]

    if df_diferenciales.empty:

        print(
            "No existen términos diferenciales: "
            "el texto no activó características conocidas."
        )

    else:

        print(
            df_diferenciales[
                [
                    "termino",
                    "contribucion_diferencial",
                ]
            ]
            .round(6)
        )

    # --------------------------------------------------------------------------
    # EXPORTAR REGISTROS SOLO CUANDO EXISTEN
    # --------------------------------------------------------------------------

    for _, termino_fila in df_positivos.iterrows():

        registros_explicaciones_v11.append({
            "caso_id": caso_id,
            "categoria_ganadora": explicacion[
                "categoria_ganadora"
            ],
            "segunda_categoria": explicacion[
                "segunda_categoria"
            ],
            "margen": explicacion[
                "margen"
            ],
            "terminos_activos": explicacion[
                "terminos_activos"
            ],
            "tipo": "positivo",
            "termino": termino_fila[
                "termino"
            ],
            "contribucion": termino_fila[
                "contribucion_ganadora"
            ],
        })

    for _, termino_fila in df_diferenciales.iterrows():

        registros_explicaciones_v11.append({
            "caso_id": caso_id,
            "categoria_ganadora": explicacion[
                "categoria_ganadora"
            ],
            "segunda_categoria": explicacion[
                "segunda_categoria"
            ],
            "margen": explicacion[
                "margen"
            ],
            "terminos_activos": explicacion[
                "terminos_activos"
            ],
            "tipo": "diferencial",
            "termino": termino_fila[
                "termino"
            ],
            "contribucion": termino_fila[
                "contribucion_diferencial"
            ],
        })


df_explicaciones_backend_v11 = pd.DataFrame(
    registros_explicaciones_v11
)


print("\n" + "=" * 80)
print("EXPLICACIONES COMPLETADAS")
print("=" * 80)

print(
    "Registros generados:",
    len(df_explicaciones_backend_v11)
)


CASO: backend_real_01
Categoría ganadora: backend
Segunda categoría: frontend
Margen: 2.1515
Términos activos: 5
Sin cobertura de vocabulario: False
Predicción utilizable: True

Términos positivos:
     termino     tfidf  contribucion_ganadora
0    backend  0.260995               1.711258
1  framework  0.330784               0.304689

Términos diferenciales:
        termino  contribucion_diferencial
0       backend                  2.428846
1     framework                  0.315249
2         datos                  0.072307
3     seguridad                 -0.027299
4  aplicaciones                 -0.313618

CASO: backend_real_02
Categoría ganadora: frontend
Segunda categoría: backend
Margen: 0.324
Términos activos: 0
Sin cobertura de vocabulario: True
Predicción utilizable: False

Términos positivos:
No existen términos positivos: el texto no activó características conocidas.

Términos diferenciales:
No existen términos diferenciales: el texto no activó características conocidas.

EXPL

In [16]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 8.1 — CANDIDATOS WORD + CHAR TF-IDF
# ==============================================================================

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion


# ==============================================================================
# VECTOR DE PALABRAS YA APROBADO PROVISIONALMENTE
# ==============================================================================

vectorizador_word_v11 = clone(
    modelo_v11_auditado.named_steps["tfidf"]
)


# ==============================================================================
# FUNCIÓN PARA CREAR MODELO HÍBRIDO
# ==============================================================================

def crear_pipeline_word_char(
    char_ngram_range=(3, 5),
    char_min_df=2,
    char_max_features=30000,
):

    word_vectorizer = clone(
        vectorizador_word_v11
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=char_ngram_range,
        min_df=char_min_df,
        max_features=char_max_features,
        lowercase=True,
        strip_accents="unicode",
        sublinear_tf=True,
        norm="l2",
    )

    features = FeatureUnion([
        (
            "word",
            word_vectorizer
        ),
        (
            "char",
            char_vectorizer
        ),
    ])

    pipeline = Pipeline([
        (
            "features",
            features
        ),
        (
            "clasificador",
            clone(clasificador_v1)
        ),
    ])

    return pipeline


# ==============================================================================
# CANDIDATOS
# ==============================================================================

pipeline_word_only = clone(
    modelo_v11_auditado
)


pipeline_word_char_35 = crear_pipeline_word_char(
    char_ngram_range=(3, 5),
    char_min_df=2,
    char_max_features=30000,
)


pipeline_word_char_36 = crear_pipeline_word_char(
    char_ngram_range=(3, 6),
    char_min_df=2,
    char_max_features=30000,
)


CANDIDATOS_CHAR_V11 = {
    "word_stopwords_acentos": (
        pipeline_word_only
    ),
    "word_char_3_5": (
        pipeline_word_char_35
    ),
    "word_char_3_6": (
        pipeline_word_char_36
    ),
}


print("Candidatos preparados:")

for nombre in CANDIDATOS_CHAR_V11:
    print(" -", nombre)

Candidatos preparados:
 - word_stopwords_acentos
 - word_char_3_5
 - word_char_3_6


In [17]:
# ==============================================================================
# CELDA 8.2 — VALIDACIÓN CRUZADA WORD + CHAR
# ==============================================================================

resultados_char_v11 = []


for nombre, modelo in CANDIDATOS_CHAR_V11.items():

    print("\n" + "=" * 80)
    print("Evaluando:", nombre)
    print("=" * 80)

    resultado = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=cv_estratificada,
        scoring=SCORING,
        return_train_score=True,

        # Windows + MAMÁ:
        n_jobs=1,

        error_score="raise",
    )


    registro = {
        "modelo": nombre,

        "accuracy_cv": float(
            np.mean(
                resultado[
                    "test_accuracy"
                ]
            )
        ),

        "precision_macro_cv": float(
            np.mean(
                resultado[
                    "test_precision_macro"
                ]
            )
        ),

        "recall_macro_cv": float(
            np.mean(
                resultado[
                    "test_recall_macro"
                ]
            )
        ),

        "f1_macro_cv": float(
            np.mean(
                resultado[
                    "test_f1_macro"
                ]
            )
        ),

        "f1_macro_std": float(
            np.std(
                resultado[
                    "test_f1_macro"
                ]
            )
        ),

        "f1_train": float(
            np.mean(
                resultado[
                    "train_f1_macro"
                ]
            )
        ),
    }


    registro[
        "brecha_train_cv"
    ] = (
        registro["f1_train"]
        - registro["f1_macro_cv"]
    )


    resultados_char_v11.append(
        registro
    )


    print(
        f"F1 Macro CV: "
        f"{registro['f1_macro_cv']:.4f}"
    )

    print(
        f"Std: "
        f"{registro['f1_macro_std']:.4f}"
    )

    print(
        f"F1 Train: "
        f"{registro['f1_train']:.4f}"
    )

    print(
        f"Brecha train-CV: "
        f"{registro['brecha_train_cv']:.4f}"
    )


Evaluando: word_stopwords_acentos
F1 Macro CV: 0.8449
Std: 0.0113
F1 Train: 1.0000
Brecha train-CV: 0.1551

Evaluando: word_char_3_5
F1 Macro CV: 0.8409
Std: 0.0137
F1 Train: 1.0000
Brecha train-CV: 0.1591

Evaluando: word_char_3_6
F1 Macro CV: 0.8463
Std: 0.0138
F1 Train: 1.0000
Brecha train-CV: 0.1537


In [18]:
# ==============================================================================
# CELDA 8.3 — TABLA COMPARATIVA
# ==============================================================================

df_resultados_char_v11 = pd.DataFrame(
    resultados_char_v11
)


df_resultados_char_v11 = (
    df_resultados_char_v11
    .sort_values(
        "f1_macro_cv",
        ascending=False
    )
    .reset_index(drop=True)
)


F1_WORD_ONLY = float(
    df_resultados_char_v11.loc[
        df_resultados_char_v11[
            "modelo"
        ] == "word_stopwords_acentos",
        "f1_macro_cv",
    ].iloc[0]
)


df_resultados_char_v11[
    "diferencia_vs_word"
] = (
    df_resultados_char_v11[
        "f1_macro_cv"
    ]
    - F1_WORD_ONLY
)


print("=" * 80)
print("COMPARACIÓN WORD vs WORD + CHAR")
print("=" * 80)


print(
    df_resultados_char_v11[
        [
            "modelo",
            "accuracy_cv",
            "f1_macro_cv",
            "f1_macro_std",
            "f1_train",
            "brecha_train_cv",
            "diferencia_vs_word",
        ]
    ]
    .round(4)
)


PATH_COMPARACION_CHAR = (
    REPORTS_V11
    / "comparacion_word_char_v1_1.csv"
)


df_resultados_char_v11.to_csv(
    PATH_COMPARACION_CHAR,
    index=False,
    encoding="utf-8-sig",
)


print("\nExportado:")
print(PATH_COMPARACION_CHAR)

COMPARACIÓN WORD vs WORD + CHAR
                   modelo  accuracy_cv  f1_macro_cv  f1_macro_std  f1_train  \
0           word_char_3_6       0.8456       0.8463        0.0138       1.0   
1  word_stopwords_acentos       0.8437       0.8449        0.0113       1.0   
2           word_char_3_5       0.8399       0.8409        0.0137       1.0   

   brecha_train_cv  diferencia_vs_word  
0           0.1537              0.0015  
1           0.1551              0.0000  
2           0.1591             -0.0040  

Exportado:
C:\Users\MAMÁ\Downloads\reports\v1.1.0\comparacion_word_char_v1_1.csv


In [19]:
# ==============================================================================
# CELDA 8.4 — ENTRENAR SOBRE TRAIN PARA PRUEBAS EXTERNAS
# ==============================================================================

modelos_char_entrenados = {}


for nombre, modelo in CANDIDATOS_CHAR_V11.items():

    print("Entrenando:", nombre)

    modelo_entrenado = clone(
        modelo
    )

    modelo_entrenado.fit(
        X_train,
        y_train
    )

    modelos_char_entrenados[
        nombre
    ] = modelo_entrenado


print(
    "\nModelos entrenados:",
    len(modelos_char_entrenados)
)

Entrenando: word_stopwords_acentos
Entrenando: word_char_3_5
Entrenando: word_char_3_6

Modelos entrenados: 3


In [20]:
# ==============================================================================
# CELDA 8.5 — MATRIZ DE CARACTERÍSTICAS GENÉRICA
# ==============================================================================

def transformar_textos_modelo(
    modelo,
    textos,
):

    if "tfidf" in modelo.named_steps:

        return (
            modelo.named_steps[
                "tfidf"
            ]
            .transform(textos)
        )

    if "features" in modelo.named_steps:

        return (
            modelo.named_steps[
                "features"
            ]
            .transform(textos)
        )

    raise KeyError(
        "No se encontró transformador "
        "'tfidf' ni 'features'."
    )

In [21]:
# ==============================================================================
# CELDA 8.6 — DIAGNÓSTICO EXTERNO WORD + CHAR
# ==============================================================================

def evaluar_modelo_externo_v11(
    nombre_modelo,
    modelo,
    df_casos,
):

    textos = (
        df_casos["texto"]
        .fillna("")
        .astype(str)
        .tolist()
    )


    matriz = transformar_textos_modelo(
        modelo,
        textos
    )


    predicciones = modelo.predict(
        textos
    )


    scores = np.asarray(
        modelo.decision_function(
            textos
        )
    )


    clases = np.asarray(
        modelo.named_steps[
            "clasificador"
        ].classes_
    )


    registros = []


    for i in range(len(textos)):

        fila_scores = scores[i]

        orden = np.argsort(
            fila_scores
        )[::-1]

        ganador = int(
            orden[0]
        )

        segundo = int(
            orden[1]
        )

        categoria_predicha = str(
            clases[ganador]
        )

        segunda_categoria = str(
            clases[segundo]
        )


        terminos_activos = int(
            matriz[i].getnnz()
        )


        margen = float(
            fila_scores[ganador]
            - fila_scores[segundo]
        )


        sin_cobertura = (
            terminos_activos == 0
        )


        categoria_esperada = str(
            df_casos.iloc[i][
                "categoria_esperada"
            ]
        )


        prediccion_correcta = (
            categoria_predicha
            == categoria_esperada
            and not sin_cobertura
        )


        registros.append({
            "modelo": nombre_modelo,

            "caso_id": (
                df_casos.iloc[i][
                    "caso_id"
                ]
            ),

            "categoria_esperada": (
                categoria_esperada
            ),

            "categoria_predicha": (
                categoria_predicha
            ),

            "segunda_categoria": (
                segunda_categoria
            ),

            "prediccion_correcta": (
                prediccion_correcta
            ),

            "margen_decision": margen,

            "features_activas": (
                terminos_activos
            ),

            "sin_cobertura": (
                sin_cobertura
            ),

            "prediccion_utilizable": (
                not sin_cobertura
            ),
        })


    return pd.DataFrame(
        registros
    )

In [22]:
# ==============================================================================
# EVALUAR TODOS LOS CANDIDATOS
# ==============================================================================

resultados_diagnostico_char = []


for nombre, modelo in (
    modelos_char_entrenados.items()
):

    df_resultado = (
        evaluar_modelo_externo_v11(
            nombre_modelo=nombre,
            modelo=modelo,
            df_casos=df_casos_diagnostico,
        )
    )

    resultados_diagnostico_char.append(
        df_resultado
    )


df_diagnostico_char = pd.concat(
    resultados_diagnostico_char,
    ignore_index=True,
)


print("=" * 80)
print("PRUEBAS EXTERNAS")
print("=" * 80)


print(
    df_diagnostico_char[
        [
            "modelo",
            "caso_id",
            "categoria_esperada",
            "categoria_predicha",
            "segunda_categoria",
            "prediccion_correcta",
            "margen_decision",
            "features_activas",
            "sin_cobertura",
        ]
    ]
    .round(4)
)

PRUEBAS EXTERNAS
                    modelo            caso_id categoria_esperada  \
0   word_stopwords_acentos    backend_real_01            backend   
1   word_stopwords_acentos    backend_real_02            backend   
2   word_stopwords_acentos  backend_adicional            backend   
3   word_stopwords_acentos           cloud_01              cloud   
4   word_stopwords_acentos           cloud_02              cloud   
5   word_stopwords_acentos     datascience_01        datascience   
6   word_stopwords_acentos     datascience_02        datascience   
7   word_stopwords_acentos        frontend_01           frontend   
8   word_stopwords_acentos        frontend_02           frontend   
9            word_char_3_5    backend_real_01            backend   
10           word_char_3_5    backend_real_02            backend   
11           word_char_3_5  backend_adicional            backend   
12           word_char_3_5           cloud_01              cloud   
13           word_char_3_5     

In [23]:
# ==============================================================================
# CELDA 8.7 — RESUMEN DIAGNÓSTICO
# ==============================================================================

df_resumen_diagnostico_char = (
    df_diagnostico_char
    .groupby("modelo")
    .agg(
        casos=(
            "caso_id",
            "count"
        ),

        casos_correctos=(
            "prediccion_correcta",
            "sum"
        ),

        casos_sin_cobertura=(
            "sin_cobertura",
            "sum"
        ),

        features_activas_promedio=(
            "features_activas",
            "mean"
        ),

        margen_promedio=(
            "margen_decision",
            "mean"
        ),
    )
    .reset_index()
)


df_resumen_diagnostico_char[
    "accuracy_diagnostico"
] = (
    df_resumen_diagnostico_char[
        "casos_correctos"
    ]
    / df_resumen_diagnostico_char[
        "casos"
    ]
)


print("=" * 80)
print("RESUMEN DE PRUEBAS EXTERNAS")
print("=" * 80)


print(
    df_resumen_diagnostico_char
    .round(4)
)

RESUMEN DE PRUEBAS EXTERNAS
                   modelo  casos  casos_correctos  casos_sin_cobertura  \
0           word_char_3_5      9                9                    0   
1           word_char_3_6      9                9                    0   
2  word_stopwords_acentos      9                6                    3   

   features_activas_promedio  margen_promedio  accuracy_diagnostico  
0                   200.5556           2.3753                1.0000  
1                   206.2222           2.6572                1.0000  
2                     1.6667           1.0509                0.6667  


In [24]:
# ==============================================================================
# CELDA 8.8 — CASOS CRÍTICOS BACKEND
# ==============================================================================

df_backend_criticos_char = (
    df_diagnostico_char[
        df_diagnostico_char[
            "caso_id"
        ].isin([
            "backend_real_01",
            "backend_real_02",
        ])
    ]
    .copy()
)


print("=" * 80)
print("CASOS CRÍTICOS BACKEND")
print("=" * 80)


print(
    df_backend_criticos_char[
        [
            "modelo",
            "caso_id",
            "categoria_predicha",
            "segunda_categoria",
            "prediccion_correcta",
            "margen_decision",
            "features_activas",
            "sin_cobertura",
        ]
    ]
    .round(4)
)

CASOS CRÍTICOS BACKEND
                    modelo          caso_id categoria_predicha  \
0   word_stopwords_acentos  backend_real_01            backend   
1   word_stopwords_acentos  backend_real_02           frontend   
9            word_char_3_5  backend_real_01            backend   
10           word_char_3_5  backend_real_02            backend   
18           word_char_3_6  backend_real_01            backend   
19           word_char_3_6  backend_real_02            backend   

   segunda_categoria  prediccion_correcta  margen_decision  features_activas  \
0           frontend                 True           2.1515                 5   
1            backend                False           0.3240                 0   
9              cloud                 True           2.4111               429   
10             cloud                 True           1.1894               207   
18             cloud                 True           2.4991               442   
19             cloud              

In [25]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 8.9 — SELECCIÓN INTEGRAL DEL CANDIDATO WORD + CHAR
# ==============================================================================

# ==============================================================================
# RESUMEN EXTERNO
# ==============================================================================

df_metricas_externas = (
    df_resumen_diagnostico_char[
        [
            "modelo",
            "casos",
            "casos_correctos",
            "casos_sin_cobertura",
            "features_activas_promedio",
            "margen_promedio",
            "accuracy_diagnostico",
        ]
    ]
    .copy()
)


# ==============================================================================
# UNIR CV + DIAGNÓSTICO
# ==============================================================================

df_seleccion_integral_v11 = (
    df_resultados_char_v11
    .merge(
        df_metricas_externas,
        on="modelo",
        how="left",
    )
)


# ==============================================================================
# CRITERIOS
# ==============================================================================

F1_REFERENCIA_WORD = float(
    df_seleccion_integral_v11.loc[
        df_seleccion_integral_v11[
            "modelo"
        ] == "word_stopwords_acentos",
        "f1_macro_cv",
    ].iloc[0]
)


# Permitimos como máximo una pérdida de 0.005
# frente al mejor word-only.
UMBRAL_PERDIDA_F1 = 0.005


df_seleccion_integral_v11[
    "criterio_f1"
] = (
    df_seleccion_integral_v11[
        "f1_macro_cv"
    ]
    >= (
        F1_REFERENCIA_WORD
        - UMBRAL_PERDIDA_F1
    )
)


df_seleccion_integral_v11[
    "criterio_cobertura"
] = (
    df_seleccion_integral_v11[
        "casos_sin_cobertura"
    ] == 0
)


df_seleccion_integral_v11[
    "criterio_diagnostico"
] = (
    df_seleccion_integral_v11[
        "accuracy_diagnostico"
    ] == 1.0
)


df_seleccion_integral_v11[
    "criterio_estabilidad"
] = (
    df_seleccion_integral_v11[
        "f1_macro_std"
    ] <= 0.02
)


df_seleccion_integral_v11[
    "criterios_aprobados"
] = (
    df_seleccion_integral_v11[
        [
            "criterio_f1",
            "criterio_cobertura",
            "criterio_diagnostico",
            "criterio_estabilidad",
        ]
    ]
    .all(axis=1)
)


# ==============================================================================
# PRIORIZACIÓN
#
# 1. Debe aprobar criterios.
# 2. Mayor F1 Macro CV.
# 3. Menor desviación.
# 4. Menor brecha train-CV.
# 5. Mayor margen externo.
# ==============================================================================

df_seleccion_integral_v11 = (
    df_seleccion_integral_v11
    .sort_values(
        by=[
            "criterios_aprobados",
            "f1_macro_cv",
            "f1_macro_std",
            "brecha_train_cv",
            "margen_promedio",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print("=" * 100)
print("SELECCIÓN INTEGRAL — TECHMIND v1.1.0")
print("=" * 100)


print(
    df_seleccion_integral_v11[
        [
            "modelo",
            "f1_macro_cv",
            "f1_macro_std",
            "brecha_train_cv",
            "casos_correctos",
            "casos_sin_cobertura",
            "accuracy_diagnostico",
            "margen_promedio",
            "criterio_f1",
            "criterio_cobertura",
            "criterio_diagnostico",
            "criterio_estabilidad",
            "criterios_aprobados",
        ]
    ]
    .round(4)
)


# ==============================================================================
# SELECCIONAR GANADOR
# ==============================================================================

df_aprobados_v11 = (
    df_seleccion_integral_v11[
        df_seleccion_integral_v11[
            "criterios_aprobados"
        ]
    ]
    .copy()
)


if df_aprobados_v11.empty:

    CANDIDATO_WORD_CHAR_APROBADO = False
    NOMBRE_CANDIDATO_WORD_CHAR = None
    MODELO_CANDIDATO_WORD_CHAR = None

    print("\nNo existe todavía un candidato aprobado.")

else:

    mejor_fila = (
        df_aprobados_v11
        .iloc[0]
    )

    NOMBRE_CANDIDATO_WORD_CHAR = (
        mejor_fila["modelo"]
    )

    MODELO_CANDIDATO_WORD_CHAR = (
        modelos_char_entrenados[
            NOMBRE_CANDIDATO_WORD_CHAR
        ]
    )

    CANDIDATO_WORD_CHAR_APROBADO = True

    print("\n" + "=" * 100)
    print("CANDIDATO SELECCIONADO")
    print("=" * 100)

    print(
        "Modelo:",
        NOMBRE_CANDIDATO_WORD_CHAR
    )

    print(
        f"F1 Macro CV: "
        f"{mejor_fila['f1_macro_cv']:.4f}"
    )

    print(
        f"Std: "
        f"{mejor_fila['f1_macro_std']:.4f}"
    )

    print(
        f"Brecha train-CV: "
        f"{mejor_fila['brecha_train_cv']:.4f}"
    )

    print(
        "Casos externos:",
        int(mejor_fila["casos_correctos"]),
        "/",
        int(mejor_fila["casos"]),
    )

    print(
        "Casos sin cobertura:",
        int(
            mejor_fila[
                "casos_sin_cobertura"
            ]
        )
    )

    print(
        f"Margen externo promedio: "
        f"{mejor_fila['margen_promedio']:.4f}"
    )

    print(
        "\nCandidato word+char aprobado:",
        CANDIDATO_WORD_CHAR_APROBADO
    )


# ==============================================================================
# EXPORTACIÓN
# ==============================================================================

PATH_SELECCION_INTEGRAL = (
    REPORTS_V11
    / "seleccion_integral_word_char.csv"
)


df_seleccion_integral_v11.to_csv(
    PATH_SELECCION_INTEGRAL,
    index=False,
    encoding="utf-8-sig",
)


print("\nExportado:")
print(PATH_SELECCION_INTEGRAL)

SELECCIÓN INTEGRAL — TECHMIND v1.1.0
                   modelo  f1_macro_cv  f1_macro_std  brecha_train_cv  \
0           word_char_3_6       0.8463        0.0138           0.1537   
1           word_char_3_5       0.8409        0.0137           0.1591   
2  word_stopwords_acentos       0.8449        0.0113           0.1551   

   casos_correctos  casos_sin_cobertura  accuracy_diagnostico  \
0                9                    0                1.0000   
1                9                    0                1.0000   
2                6                    3                0.6667   

   margen_promedio  criterio_f1  criterio_cobertura  criterio_diagnostico  \
0           2.6572         True                True                  True   
1           2.3753         True                True                  True   
2           1.0509         True               False                 False   

   criterio_estabilidad  criterios_aprobados  
0                  True                 True  
1     

In [26]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 9.1 — CONFIGURACIÓN ACTUAL DEL CLASIFICADOR
# ==============================================================================

from sklearn.base import clone
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

import numpy as np
import pandas as pd


# ==============================================================================
# CANDIDATO PRINCIPAL
# ==============================================================================

NOMBRE_BASE_V11 = "word_char_3_6"

pipeline_base_v11 = clone(
    CANDIDATOS_CHAR_V11[
        NOMBRE_BASE_V11
    ]
)


clasificador_base_v11 = (
    pipeline_base_v11
    .named_steps[
        "clasificador"
    ]
)


parametros_clasificador_v11 = (
    clasificador_base_v11
    .get_params()
)


ALPHA_BASE_V11 = float(
    parametros_clasificador_v11[
        "alpha"
    ]
)

LOSS_BASE_V11 = (
    parametros_clasificador_v11[
        "loss"
    ]
)

PENALTY_BASE_V11 = (
    parametros_clasificador_v11[
        "penalty"
    ]
)


print("=" * 80)
print("CLASIFICADOR ACTUAL")
print("=" * 80)

print(
    "Clase:",
    type(
        clasificador_base_v11
    ).__name__
)

print(
    "Loss:",
    LOSS_BASE_V11
)

print(
    "Alpha:",
    ALPHA_BASE_V11
)

print(
    "Penalty:",
    PENALTY_BASE_V11
)

print(
    "Random state:",
    parametros_clasificador_v11.get(
        "random_state"
    )
)

CLASIFICADOR ACTUAL
Clase: SGDClassifier
Loss: hinge
Alpha: 3e-05
Penalty: l2
Random state: 42


In [27]:
# ==============================================================================
# CELDA 9.2 — ESPACIO DE BÚSQUEDA
# ==============================================================================

factores_alpha = [
    0.25,
    0.50,
    1.00,
    2.00,
    4.00,
    8.00,
]


ALPHAS_V11 = sorted({
    float(
        ALPHA_BASE_V11
        * factor
    )
    for factor in factores_alpha
})


print("Alpha actual:")
print(ALPHA_BASE_V11)

print("\nValores a evaluar:")

for alpha in ALPHAS_V11:
    print(
        f" - {alpha:.8g}"
    )

Alpha actual:
3e-05

Valores a evaluar:
 - 7.5e-06
 - 1.5e-05
 - 3e-05
 - 6e-05
 - 0.00012
 - 0.00024


In [28]:
# ==============================================================================
# CELDA 9.3 — OPTIMIZACIÓN DE ALPHA
# ==============================================================================

PARAM_GRID_V11 = {
    "clasificador__alpha": (
        ALPHAS_V11
    )
}


busqueda_alpha_v11 = GridSearchCV(
    estimator=pipeline_base_v11,
    param_grid=PARAM_GRID_V11,

    scoring="f1_macro",

    cv=cv_estratificada,

    # Windows + ruta MAMÁ
    n_jobs=1,

    refit=True,

    return_train_score=True,

    error_score="raise",

    verbose=1,
)


print("=" * 80)
print("OPTIMIZANDO TECHMIND v1.1.0")
print("=" * 80)

print(
    "Arquitectura:",
    NOMBRE_BASE_V11
)

print(
    "Combinaciones:",
    len(ALPHAS_V11)
)

print(
    "Folds:",
    N_SPLITS_CV
)


busqueda_alpha_v11.fit(
    X_train,
    y_train
)


print("\nOptimización completada.")

OPTIMIZANDO TECHMIND v1.1.0
Arquitectura: word_char_3_6
Combinaciones: 6
Folds: 5
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Optimización completada.


In [29]:
# ==============================================================================
# CELDA 9.4 — RESULTADOS DE REGULARIZACIÓN
# ==============================================================================

df_grid_v11 = pd.DataFrame(
    busqueda_alpha_v11.cv_results_
)


df_regularizacion_v11 = pd.DataFrame({
    "alpha": (
        df_grid_v11[
            "param_clasificador__alpha"
        ]
        .astype(float)
    ),

    "f1_macro_cv": (
        df_grid_v11[
            "mean_test_score"
        ]
    ),

    "f1_macro_std": (
        df_grid_v11[
            "std_test_score"
        ]
    ),

    "f1_train": (
        df_grid_v11[
            "mean_train_score"
        ]
    ),
})


df_regularizacion_v11[
    "brecha_train_cv"
] = (
    df_regularizacion_v11[
        "f1_train"
    ]
    - df_regularizacion_v11[
        "f1_macro_cv"
    ]
)


df_regularizacion_v11[
    "es_alpha_original"
] = np.isclose(
    df_regularizacion_v11[
        "alpha"
    ],
    ALPHA_BASE_V11,
)


df_regularizacion_v11 = (
    df_regularizacion_v11
    .sort_values(
        [
            "f1_macro_cv",
            "brecha_train_cv",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print("=" * 80)
print("RESULTADOS DE REGULARIZACIÓN")
print("=" * 80)


print(
    df_regularizacion_v11
    .round({
        "alpha": 8,
        "f1_macro_cv": 4,
        "f1_macro_std": 4,
        "f1_train": 4,
        "brecha_train_cv": 4,
    })
)

RESULTADOS DE REGULARIZACIÓN
      alpha  f1_macro_cv  f1_macro_std  f1_train  brecha_train_cv  \
0  0.000240       0.8493        0.0133    0.9997           0.1504   
1  0.000120       0.8487        0.0104    1.0000           0.1513   
2  0.000060       0.8473        0.0113    1.0000           0.1527   
3  0.000030       0.8463        0.0138    1.0000           0.1537   
4  0.000015       0.8392        0.0123    1.0000           0.1608   
5  0.000008       0.8347        0.0147    1.0000           0.1653   

   es_alpha_original  
0              False  
1              False  
2              False  
3               True  
4              False  
5              False  


In [30]:
# ==============================================================================
# CELDA 9.5 — MEJOR CONFIGURACIÓN
# ==============================================================================

MEJOR_ALPHA_V11 = float(
    busqueda_alpha_v11.best_params_[
        "clasificador__alpha"
    ]
)


MEJOR_F1_OPT_V11 = float(
    busqueda_alpha_v11.best_score_
)


MODELO_OPTIMIZADO_V11 = (
    busqueda_alpha_v11
    .best_estimator_
)


fila_mejor_v11 = (
    df_regularizacion_v11[
        np.isclose(
            df_regularizacion_v11[
                "alpha"
            ],
            MEJOR_ALPHA_V11,
        )
    ]
    .iloc[0]
)


MEJOR_STD_OPT_V11 = float(
    fila_mejor_v11[
        "f1_macro_std"
    ]
)


MEJOR_F1_TRAIN_OPT_V11 = float(
    fila_mejor_v11[
        "f1_train"
    ]
)


MEJOR_BRECHA_OPT_V11 = float(
    fila_mejor_v11[
        "brecha_train_cv"
    ]
)


print("=" * 80)
print("MEJOR CONFIGURACIÓN v1.1.0")
print("=" * 80)

print(
    "Alpha anterior:",
    ALPHA_BASE_V11
)

print(
    "Alpha seleccionado:",
    MEJOR_ALPHA_V11
)

print(
    f"F1 Macro CV: "
    f"{MEJOR_F1_OPT_V11:.4f}"
)

print(
    f"Std: "
    f"{MEJOR_STD_OPT_V11:.4f}"
)

print(
    f"F1 Train: "
    f"{MEJOR_F1_TRAIN_OPT_V11:.4f}"
)

print(
    f"Brecha train-CV: "
    f"{MEJOR_BRECHA_OPT_V11:.4f}"
)

MEJOR CONFIGURACIÓN v1.1.0
Alpha anterior: 3e-05
Alpha seleccionado: 0.00024
F1 Macro CV: 0.8493
Std: 0.0133
F1 Train: 0.9997
Brecha train-CV: 0.1504


In [31]:
# ==============================================================================
# CELDA 9.6 — COMPARACIÓN PRE / POST OPTIMIZACIÓN
# ==============================================================================

fila_base_char36 = (
    df_resultados_char_v11[
        df_resultados_char_v11[
            "modelo"
        ] == "word_char_3_6"
    ]
    .iloc[0]
)


F1_BASE_CHAR36 = float(
    fila_base_char36[
        "f1_macro_cv"
    ]
)

STD_BASE_CHAR36 = float(
    fila_base_char36[
        "f1_macro_std"
    ]
)

BRECHA_BASE_CHAR36 = float(
    fila_base_char36[
        "brecha_train_cv"
    ]
)


df_comparacion_opt_v11 = pd.DataFrame([
    {
        "modelo": (
            "word_char_3_6_base"
        ),

        "alpha": (
            ALPHA_BASE_V11
        ),

        "f1_macro_cv": (
            F1_BASE_CHAR36
        ),

        "f1_macro_std": (
            STD_BASE_CHAR36
        ),

        "brecha_train_cv": (
            BRECHA_BASE_CHAR36
        ),
    },

    {
        "modelo": (
            "word_char_3_6_optimizado"
        ),

        "alpha": (
            MEJOR_ALPHA_V11
        ),

        "f1_macro_cv": (
            MEJOR_F1_OPT_V11
        ),

        "f1_macro_std": (
            MEJOR_STD_OPT_V11
        ),

        "brecha_train_cv": (
            MEJOR_BRECHA_OPT_V11
        ),
    },
])


print("=" * 80)
print("COMPARACIÓN PRE / POST OPTIMIZACIÓN")
print("=" * 80)


print(
    df_comparacion_opt_v11
    .round(4)
)


print(
    "\nCambio F1:",
    f"{MEJOR_F1_OPT_V11 - F1_BASE_CHAR36:+.4f}"
)

print(
    "Cambio brecha:",
    f"{MEJOR_BRECHA_OPT_V11 - BRECHA_BASE_CHAR36:+.4f}"
)

COMPARACIÓN PRE / POST OPTIMIZACIÓN
                     modelo   alpha  f1_macro_cv  f1_macro_std  \
0        word_char_3_6_base  0.0000       0.8463        0.0138   
1  word_char_3_6_optimizado  0.0002       0.8493        0.0133   

   brecha_train_cv  
0           0.1537  
1           0.1504  

Cambio F1: +0.0029
Cambio brecha: -0.0032


In [32]:
# ==============================================================================
# CELDA 9.7 — PRUEBA DE REGRESIÓN EXTERNA
# ==============================================================================

df_diagnostico_optimizado_v11 = (
    evaluar_modelo_externo_v11(
        nombre_modelo=(
            "word_char_3_6_optimizado"
        ),

        modelo=MODELO_OPTIMIZADO_V11,

        df_casos=(
            df_casos_diagnostico
        ),
    )
)


print("=" * 80)
print("PRUEBAS EXTERNAS — MODELO OPTIMIZADO")
print("=" * 80)


print(
    df_diagnostico_optimizado_v11[
        [
            "caso_id",
            "categoria_esperada",
            "categoria_predicha",
            "segunda_categoria",
            "prediccion_correcta",
            "margen_decision",
            "features_activas",
            "sin_cobertura",
        ]
    ]
    .round(4)
)

PRUEBAS EXTERNAS — MODELO OPTIMIZADO
             caso_id categoria_esperada categoria_predicha segunda_categoria  \
0    backend_real_01            backend            backend             cloud   
1    backend_real_02            backend            backend             cloud   
2  backend_adicional            backend            backend             cloud   
3           cloud_01              cloud              cloud          frontend   
4           cloud_02              cloud              cloud          frontend   
5     datascience_01        datascience        datascience           backend   
6     datascience_02        datascience        datascience             cloud   
7        frontend_01           frontend           frontend           backend   
8        frontend_02           frontend           frontend           backend   

   prediccion_correcta  margen_decision  features_activas  sin_cobertura  
0                 True           1.8771               442          False  
1           

In [33]:
# ==============================================================================
# CELDA 9.8 — RESUMEN DE REGRESIÓN
# ==============================================================================

CASOS_CORRECTOS_OPT_V11 = int(
    df_diagnostico_optimizado_v11[
        "prediccion_correcta"
    ].sum()
)


CASOS_SIN_COBERTURA_OPT_V11 = int(
    df_diagnostico_optimizado_v11[
        "sin_cobertura"
    ].sum()
)


MARGEN_PROMEDIO_OPT_V11 = float(
    df_diagnostico_optimizado_v11[
        "margen_decision"
    ].mean()
)


print("=" * 80)
print("RESUMEN DE REGRESIÓN")
print("=" * 80)

print(
    "Casos correctos:",
    CASOS_CORRECTOS_OPT_V11,
    "/",
    len(
        df_diagnostico_optimizado_v11
    )
)

print(
    "Casos sin cobertura:",
    CASOS_SIN_COBERTURA_OPT_V11
)

print(
    f"Margen promedio: "
    f"{MARGEN_PROMEDIO_OPT_V11:.4f}"
)

RESUMEN DE REGRESIÓN
Casos correctos: 9 / 9
Casos sin cobertura: 0
Margen promedio: 1.7085


In [34]:
# ==============================================================================
# CELDA 9.9 — SELECCIÓN DEFINITIVA PRE-TEST
# ==============================================================================

CRITERIO_CV_OPT = (
    MEJOR_F1_OPT_V11
    >= F1_BASE_CHAR36 - 0.001
)


CRITERIO_COBERTURA_OPT = (
    CASOS_SIN_COBERTURA_OPT_V11
    == 0
)


CRITERIO_REGRESION_OPT = (
    CASOS_CORRECTOS_OPT_V11
    == len(
        df_diagnostico_optimizado_v11
    )
)


CRITERIO_ESTABILIDAD_OPT = (
    MEJOR_STD_OPT_V11
    <= 0.02
)


CRITERIO_BRECHA_OPT = (
    MEJOR_BRECHA_OPT_V11
    <= BRECHA_BASE_CHAR36 + 0.005
)


MODELO_OPTIMIZADO_APROBADO_V11 = all([
    CRITERIO_CV_OPT,
    CRITERIO_COBERTURA_OPT,
    CRITERIO_REGRESION_OPT,
    CRITERIO_ESTABILIDAD_OPT,
    CRITERIO_BRECHA_OPT,
])


print("=" * 80)
print("APROBACIÓN PRE-TEST — TECHMIND v1.1.0")
print("=" * 80)

print(
    "Rendimiento CV:",
    CRITERIO_CV_OPT
)

print(
    "Cobertura externa:",
    CRITERIO_COBERTURA_OPT
)

print(
    "Regresión externa:",
    CRITERIO_REGRESION_OPT
)

print(
    "Estabilidad:",
    CRITERIO_ESTABILIDAD_OPT
)

print(
    "Brecha train-CV:",
    CRITERIO_BRECHA_OPT
)


print(
    "\nModelo optimizado aprobado:",
    MODELO_OPTIMIZADO_APROBADO_V11
)


if MODELO_OPTIMIZADO_APROBADO_V11:

    MODELO_CANDIDATO_FINAL_V11 = (
        MODELO_OPTIMIZADO_V11
    )

    NOMBRE_MODELO_FINAL_V11 = (
        "TF-IDF Word + Char 3-6 "
        "+ SGDClassifier optimizado"
    )

else:

    MODELO_CANDIDATO_FINAL_V11 = (
        modelos_char_entrenados[
            "word_char_3_6"
        ]
    )

    NOMBRE_MODELO_FINAL_V11 = (
        "TF-IDF Word + Char 3-6 "
        "+ SGDClassifier"
    )


print(
    "\nCandidato que pasa a la siguiente fase:"
)

print(
    NOMBRE_MODELO_FINAL_V11
)

APROBACIÓN PRE-TEST — TECHMIND v1.1.0
Rendimiento CV: True
Cobertura externa: True
Regresión externa: True
Estabilidad: True
Brecha train-CV: True

Modelo optimizado aprobado: True

Candidato que pasa a la siguiente fase:
TF-IDF Word + Char 3-6 + SGDClassifier optimizado


In [35]:
# ==============================================================================
# CELDA 9.10 — EXPORTACIÓN
# ==============================================================================

PATH_REGULARIZACION_V11 = (
    REPORTS_V11
    / "regularizacion_sgd_v1_1.csv"
)


PATH_COMPARACION_OPT_V11 = (
    REPORTS_V11
    / "comparacion_pre_post_optimizacion.csv"
)


PATH_DIAGNOSTICO_OPT_V11 = (
    REPORTS_V11
    / "diagnostico_externo_modelo_optimizado.csv"
)


df_regularizacion_v11.to_csv(
    PATH_REGULARIZACION_V11,
    index=False,
    encoding="utf-8-sig",
)


df_comparacion_opt_v11.to_csv(
    PATH_COMPARACION_OPT_V11,
    index=False,
    encoding="utf-8-sig",
)


df_diagnostico_optimizado_v11.to_csv(
    PATH_DIAGNOSTICO_OPT_V11,
    index=False,
    encoding="utf-8-sig",
)


print("Exportados:")

print(
    " -",
    PATH_REGULARIZACION_V11
)

print(
    " -",
    PATH_COMPARACION_OPT_V11
)

print(
    " -",
    PATH_DIAGNOSTICO_OPT_V11
)

Exportados:
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\regularizacion_sgd_v1_1.csv
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\comparacion_pre_post_optimizacion.csv
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\diagnostico_externo_modelo_optimizado.csv


In [36]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 10.1 — PREDICCIONES OUT-OF-FOLD
# ==============================================================================

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

import numpy as np
import pandas as pd


# ==============================================================================
# CONTENEDOR
# ==============================================================================

registros_oof_v11 = []


print("=" * 80)
print("GENERANDO PREDICCIONES OUT-OF-FOLD")
print("=" * 80)

print(
    "Modelo:",
    NOMBRE_MODELO_FINAL_V11
)

print(
    "Documentos train:",
    len(X_train)
)

print(
    "Folds:",
    N_SPLITS_CV
)


# ==============================================================================
# CV MANUAL
# ==============================================================================

for fold, (
    idx_train_fold,
    idx_valid_fold,
) in enumerate(
    cv_estratificada.split(
        X_train,
        y_train,
    ),
    start=1,
):

    print(
        f"\nProcesando fold {fold}/{N_SPLITS_CV}..."
    )

    X_fold_train = X_train.iloc[
        idx_train_fold
    ]

    y_fold_train = y_train.iloc[
        idx_train_fold
    ]

    X_fold_valid = X_train.iloc[
        idx_valid_fold
    ]

    y_fold_valid = y_train.iloc[
        idx_valid_fold
    ]


    # --------------------------------------------------------------------------
    # ENTRENAR MODELO INDEPENDIENTE
    # --------------------------------------------------------------------------

    modelo_fold = clone(
        MODELO_CANDIDATO_FINAL_V11
    )

    modelo_fold.fit(
        X_fold_train,
        y_fold_train
    )


    # --------------------------------------------------------------------------
    # PREDICCIONES
    # --------------------------------------------------------------------------

    predicciones_fold = (
        modelo_fold.predict(
            X_fold_valid
        )
    )


    scores_fold = np.asarray(
        modelo_fold.decision_function(
            X_fold_valid
        )
    )


    clases_fold = np.asarray(
        modelo_fold.named_steps[
            "clasificador"
        ].classes_
    )


    # --------------------------------------------------------------------------
    # TRANSFORMADORES WORD + CHAR
    # --------------------------------------------------------------------------

    feature_union = (
        modelo_fold.named_steps[
            "features"
        ]
    )

    transformadores = dict(
        feature_union.transformer_list
    )

    vectorizador_word = (
        transformadores["word"]
    )

    vectorizador_char = (
        transformadores["char"]
    )


    matriz_word = (
        vectorizador_word.transform(
            X_fold_valid
        )
    )

    matriz_char = (
        vectorizador_char.transform(
            X_fold_valid
        )
    )


    # --------------------------------------------------------------------------
    # REGISTROS
    # --------------------------------------------------------------------------

    for posicion_local in range(
        len(X_fold_valid)
    ):

        fila_scores = (
            scores_fold[posicion_local]
        )

        orden = np.argsort(
            fila_scores
        )[::-1]

        ganador = int(
            orden[0]
        )

        segundo = int(
            orden[1]
        )

        categoria_predicha = str(
            clases_fold[ganador]
        )

        segunda_categoria = str(
            clases_fold[segundo]
        )

        categoria_real = str(
            y_fold_valid.iloc[
                posicion_local
            ]
        )


        margen = float(
            fila_scores[ganador]
            - fila_scores[segundo]
        )


        word_activas = int(
            matriz_word[
                posicion_local
            ].getnnz()
        )

        char_activas = int(
            matriz_char[
                posicion_local
            ].getnnz()
        )

        total_activas = (
            word_activas
            + char_activas
        )


        registros_oof_v11.append({
            "fold": fold,

            "indice_original": int(
                X_fold_valid.index[
                    posicion_local
                ]
            ),

            "categoria_real": (
                categoria_real
            ),

            "categoria_predicha": (
                categoria_predicha
            ),

            "segunda_categoria": (
                segunda_categoria
            ),

            "prediccion_correcta": (
                categoria_predicha
                == categoria_real
            ),

            "puntuacion_ganadora": float(
                fila_scores[ganador]
            ),

            "puntuacion_segunda": float(
                fila_scores[segundo]
            ),

            "margen_decision": margen,

            "word_features_activas": (
                word_activas
            ),

            "char_features_activas": (
                char_activas
            ),

            "features_activas_total": (
                total_activas
            ),
        })


df_oof_v11 = pd.DataFrame(
    registros_oof_v11
)


print("\nPredicciones OOF generadas:")
print(len(df_oof_v11))


assert len(df_oof_v11) == len(
    X_train
)

assert (
    df_oof_v11[
        "indice_original"
    ].nunique()
    == len(X_train)
)


print(
    "Todos los documentos tienen "
    "exactamente una predicción OOF."
)

GENERANDO PREDICCIONES OUT-OF-FOLD
Modelo: TF-IDF Word + Char 3-6 + SGDClassifier optimizado
Documentos train: 3666
Folds: 5

Procesando fold 1/5...

Procesando fold 2/5...

Procesando fold 3/5...

Procesando fold 4/5...

Procesando fold 5/5...

Predicciones OOF generadas:
3666
Todos los documentos tienen exactamente una predicción OOF.


In [37]:
# ==============================================================================
# CELDA 10.2 — MÉTRICAS OOF
# ==============================================================================

ACCURACY_OOF_V11 = accuracy_score(
    df_oof_v11["categoria_real"],
    df_oof_v11["categoria_predicha"],
)


F1_MACRO_OOF_V11 = f1_score(
    df_oof_v11["categoria_real"],
    df_oof_v11["categoria_predicha"],
    average="macro",
)


PRECISION_MACRO_OOF_V11 = (
    precision_score(
        df_oof_v11[
            "categoria_real"
        ],
        df_oof_v11[
            "categoria_predicha"
        ],
        average="macro",
    )
)


RECALL_MACRO_OOF_V11 = (
    recall_score(
        df_oof_v11[
            "categoria_real"
        ],
        df_oof_v11[
            "categoria_predicha"
        ],
        average="macro",
    )
)


print("=" * 80)
print("RENDIMIENTO OUT-OF-FOLD")
print("=" * 80)

print(
    f"Accuracy:        "
    f"{ACCURACY_OOF_V11:.4f}"
)

print(
    f"Precision Macro: "
    f"{PRECISION_MACRO_OOF_V11:.4f}"
)

print(
    f"Recall Macro:    "
    f"{RECALL_MACRO_OOF_V11:.4f}"
)

print(
    f"F1 Macro:        "
    f"{F1_MACRO_OOF_V11:.4f}"
)

print(
    f"F1 CV previo:    "
    f"{MEJOR_F1_OPT_V11:.4f}"
)

print(
    f"Diferencia:      "
    f"{F1_MACRO_OOF_V11 - MEJOR_F1_OPT_V11:+.4f}"
)

RENDIMIENTO OUT-OF-FOLD
Accuracy:        0.8483
Precision Macro: 0.8501
Recall Macro:    0.8486
F1 Macro:        0.8492
F1 CV previo:    0.8493
Diferencia:      -0.0001


In [38]:
# ==============================================================================
# CELDA 10.3 — DISTRIBUCIÓN DE MÁRGENES
# ==============================================================================

CUANTILES = [
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
]


cuantiles_margen = (
    df_oof_v11[
        "margen_decision"
    ]
    .quantile(
        CUANTILES
    )
)


print("=" * 80)
print("CUANTILES — MARGEN DE DECISIÓN")
print("=" * 80)

print(
    cuantiles_margen.round(4)
)


print("\nMargen promedio — correctas:")

print(
    round(
        df_oof_v11.loc[
            df_oof_v11[
                "prediccion_correcta"
            ],
            "margen_decision",
        ].mean(),
        4,
    )
)


print(
    "\nMargen promedio — incorrectas:"
)

print(
    round(
        df_oof_v11.loc[
            ~df_oof_v11[
                "prediccion_correcta"
            ],
            "margen_decision",
        ].mean(),
        4,
    )
)

CUANTILES — MARGEN DE DECISIÓN
0.05    0.1224
0.10    0.2507
0.25    0.6753
0.50    1.4897
0.75    2.3492
0.90    2.9189
0.95    3.3049
Name: margen_decision, dtype: float64

Margen promedio — correctas:
1.7527

Margen promedio — incorrectas:
0.5165


In [39]:
# ==============================================================================
# CELDA 10.4 — CALIBRAR UMBRAL DE REVISIÓN
# ==============================================================================

MAX_REVIEW_RATE = 0.25


error_real = (
    ~df_oof_v11[
        "prediccion_correcta"
    ]
)


registros_umbral = []


for quantile in np.arange(
    0.05,
    0.51,
    0.01,
):

    umbral = float(
        df_oof_v11[
            "margen_decision"
        ].quantile(
            quantile
        )
    )


    revision = (
        df_oof_v11[
            "margen_decision"
        ] < umbral
    )


    cantidad_revision = int(
        revision.sum()
    )


    errores_totales = int(
        error_real.sum()
    )


    errores_detectados = int(
        (
            revision
            & error_real
        ).sum()
    )


    review_rate = float(
        revision.mean()
    )


    if cantidad_revision > 0:

        precision_revision = (
            errores_detectados
            / cantidad_revision
        )

    else:

        precision_revision = 0.0


    if errores_totales > 0:

        recall_errores = (
            errores_detectados
            / errores_totales
        )

    else:

        recall_errores = 0.0


    aceptadas = ~revision


    if aceptadas.sum() > 0:

        accuracy_aceptadas = float(
            df_oof_v11.loc[
                aceptadas,
                "prediccion_correcta",
            ].mean()
        )

    else:

        accuracy_aceptadas = np.nan


    registros_umbral.append({
        "quantile": float(
            quantile
        ),

        "umbral_margen": (
            umbral
        ),

        "review_rate": (
            review_rate
        ),

        "errores_detectados": (
            errores_detectados
        ),

        "errores_totales": (
            errores_totales
        ),

        "recall_errores": (
            recall_errores
        ),

        "precision_revision": (
            precision_revision
        ),

        "accuracy_aceptadas": (
            accuracy_aceptadas
        ),
    })


df_calibracion_margen = pd.DataFrame(
    registros_umbral
)

In [40]:
# ==============================================================================
# SELECCIÓN DEL UMBRAL
# ==============================================================================

df_umbral_validos = (
    df_calibracion_margen[
        df_calibracion_margen[
            "review_rate"
        ] <= MAX_REVIEW_RATE
    ]
    .copy()
)


if df_umbral_validos.empty:

    raise RuntimeError(
        "No fue posible encontrar "
        "un umbral operacional."
    )


df_umbral_validos = (
    df_umbral_validos
    .sort_values(
        [
            "recall_errores",
            "accuracy_aceptadas",
            "review_rate",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


mejor_umbral = (
    df_umbral_validos.iloc[0]
)


UMBRAL_MARGEN_REVISION_V11 = float(
    mejor_umbral[
        "umbral_margen"
    ]
)


print("=" * 80)
print("UMBRAL DE REVISIÓN SELECCIONADO")
print("=" * 80)

print(
    f"Margen mínimo: "
    f"{UMBRAL_MARGEN_REVISION_V11:.4f}"
)

print(
    f"Tasa de revisión: "
    f"{mejor_umbral['review_rate']:.2%}"
)

print(
    f"Errores capturados: "
    f"{mejor_umbral['recall_errores']:.2%}"
)

print(
    f"Precisión dentro de revisión: "
    f"{mejor_umbral['precision_revision']:.2%}"
)

print(
    f"Accuracy de aceptadas: "
    f"{mejor_umbral['accuracy_aceptadas']:.2%}"
)

UMBRAL DE REVISIÓN SELECCIONADO
Margen mínimo: 0.6435
Tasa de revisión: 24.00%
Errores capturados: 71.58%
Precisión dentro de revisión: 45.23%
Accuracy de aceptadas: 94.33%


In [41]:
# ==============================================================================
# CELDA 10.5 — CALIBRACIÓN DE COBERTURA
# ==============================================================================

df_oof_correctas = (
    df_oof_v11[
        df_oof_v11[
            "prediccion_correcta"
        ]
    ]
)


UMBRAL_FEATURES_POCAS_V11 = int(
    max(
        1,
        np.floor(
            df_oof_correctas[
                "features_activas_total"
            ].quantile(
                0.05
            )
        ),
    )
)


print("=" * 80)
print("COBERTURA DEL MODELO")
print("=" * 80)

print(
    "Features activas mínimas:",
    int(
        df_oof_v11[
            "features_activas_total"
        ].min()
    )
)

print(
    "Mediana:",
    round(
        df_oof_v11[
            "features_activas_total"
        ].median(),
        2,
    )
)

print(
    "P05 correctas:",
    UMBRAL_FEATURES_POCAS_V11
)


print("\nDocumentos sin cobertura total:")

print(
    int(
        (
            df_oof_v11[
                "features_activas_total"
            ] == 0
        ).sum()
    )
)

COBERTURA DEL MODELO
Features activas mínimas: 5
Mediana: 307.0
P05 correctas: 178

Documentos sin cobertura total:
0


In [42]:
# ==============================================================================
# CELDA 10.6 — POLÍTICA OPERACIONAL v1.1
# ==============================================================================

def asignar_estado_operacional_v11(
    margen,
    features_activas,
):

    if features_activas == 0:

        return (
            "rechazada",
            "Sin cobertura del vocabulario.",
        )


    if (
        margen
        < UMBRAL_MARGEN_REVISION_V11
    ):

        return (
            "revision",
            "Margen de decisión reducido.",
        )


    if (
        features_activas
        <= UMBRAL_FEATURES_POCAS_V11
    ):

        return (
            "revision",
            "Cobertura reducida de características.",
        )


    return (
        "aceptada",
        "Predicción con evidencia suficiente.",
    )

In [43]:
# ==============================================================================
# APLICAR POLÍTICA
# ==============================================================================

estados = []
motivos = []


for _, fila in df_oof_v11.iterrows():

    estado, motivo = (
        asignar_estado_operacional_v11(
            margen=float(
                fila[
                    "margen_decision"
                ]
            ),

            features_activas=int(
                fila[
                    "features_activas_total"
                ]
            ),
        )
    )

    estados.append(
        estado
    )

    motivos.append(
        motivo
    )


df_oof_v11[
    "estado_operacional"
] = estados


df_oof_v11[
    "motivo_operacional"
] = motivos

In [44]:
# ==============================================================================
# CELDA 10.7 — EVALUACIÓN OPERACIONAL
# ==============================================================================

df_estado_operacional = (
    df_oof_v11[
        "estado_operacional"
    ]
    .value_counts()
    .rename_axis(
        "estado"
    )
    .reset_index(
        name="documentos"
    )
)


df_estado_operacional[
    "porcentaje"
] = (
    df_estado_operacional[
        "documentos"
    ]
    / len(df_oof_v11)
)


print("=" * 80)
print("DISTRIBUCIÓN OPERACIONAL")
print("=" * 80)

print(
    df_estado_operacional
    .round(4)
)

DISTRIBUCIÓN OPERACIONAL
     estado  documentos  porcentaje
0  aceptada        2662      0.7261
1  revision        1004      0.2739


In [45]:
# ==============================================================================
# CALIDAD POR ESTADO
# ==============================================================================

df_calidad_estado = (
    df_oof_v11
    .groupby(
        "estado_operacional"
    )
    .agg(
        documentos=(
            "prediccion_correcta",
            "size"
        ),

        correctas=(
            "prediccion_correcta",
            "sum"
        ),

        accuracy=(
            "prediccion_correcta",
            "mean"
        ),

        margen_promedio=(
            "margen_decision",
            "mean"
        ),

        features_promedio=(
            "features_activas_total",
            "mean"
        ),
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("CALIDAD POR ESTADO")
print("=" * 80)

print(
    df_calidad_estado
    .round(4)
)


CALIDAD POR ESTADO
  estado_operacional  documentos  correctas  accuracy  margen_promedio  \
0           aceptada        2662       2507    0.9418           1.9622   
1           revision        1004        603    0.6006           0.5126   

   features_promedio  
0           316.0383  
1           289.7161  


In [46]:
# ==============================================================================
# CELDA 10.8 — CAPTURA DE ERRORES
# ==============================================================================

errores_oof = (
    ~df_oof_v11[
        "prediccion_correcta"
    ]
)


en_revision_o_rechazo = (
    df_oof_v11[
        "estado_operacional"
    ].isin([
        "revision",
        "rechazada",
    ])
)


TOTAL_ERRORES_OOF = int(
    errores_oof.sum()
)


ERRORES_DETECTADOS_OOF = int(
    (
        errores_oof
        & en_revision_o_rechazo
    ).sum()
)


if TOTAL_ERRORES_OOF > 0:

    TASA_CAPTURA_ERRORES_OOF = (
        ERRORES_DETECTADOS_OOF
        / TOTAL_ERRORES_OOF
    )

else:

    TASA_CAPTURA_ERRORES_OOF = 0.0


ACEPTADAS_MASK = (
    df_oof_v11[
        "estado_operacional"
    ] == "aceptada"
)


ACCURACY_ACEPTADAS_OOF = float(
    df_oof_v11.loc[
        ACEPTADAS_MASK,
        "prediccion_correcta",
    ].mean()
)


print("=" * 80)
print("EFECTIVIDAD OPERACIONAL")
print("=" * 80)

print(
    "Errores totales:",
    TOTAL_ERRORES_OOF
)

print(
    "Errores enviados a revisión/rechazo:",
    ERRORES_DETECTADOS_OOF
)

print(
    f"Tasa de captura de errores: "
    f"{TASA_CAPTURA_ERRORES_OOF:.2%}"
)

print(
    f"Accuracy de predicciones aceptadas: "
    f"{ACCURACY_ACEPTADAS_OOF:.2%}"
)

print(
    f"Accuracy global OOF: "
    f"{ACCURACY_OOF_V11:.2%}"
)

EFECTIVIDAD OPERACIONAL
Errores totales: 556
Errores enviados a revisión/rechazo: 401
Tasa de captura de errores: 72.12%
Accuracy de predicciones aceptadas: 94.18%
Accuracy global OOF: 84.83%


In [47]:
# ==============================================================================
# CELDA 10.9 — NIVELES DE MARGEN
# ==============================================================================

UMBRAL_MARGEN_P10_V11 = float(
    df_oof_v11[
        "margen_decision"
    ].quantile(
        0.10
    )
)

UMBRAL_MARGEN_P25_V11 = float(
    df_oof_v11[
        "margen_decision"
    ].quantile(
        0.25
    )
)

UMBRAL_MARGEN_P50_V11 = float(
    df_oof_v11[
        "margen_decision"
    ].quantile(
        0.50
    )
)

UMBRAL_MARGEN_P90_V11 = float(
    df_oof_v11[
        "margen_decision"
    ].quantile(
        0.90
    )
)


print("=" * 80)
print("UMBRALES DE MARGEN v1.1.0")
print("=" * 80)

print(
    f"P10: {UMBRAL_MARGEN_P10_V11:.4f}"
)

print(
    f"P25: {UMBRAL_MARGEN_P25_V11:.4f}"
)

print(
    f"P50: {UMBRAL_MARGEN_P50_V11:.4f}"
)

print(
    f"P90: {UMBRAL_MARGEN_P90_V11:.4f}"
)

print(
    f"\nUmbral operacional revisión: "
    f"{UMBRAL_MARGEN_REVISION_V11:.4f}"
)

print(
    "Umbral cobertura reducida:",
    UMBRAL_FEATURES_POCAS_V11
)

UMBRALES DE MARGEN v1.1.0
P10: 0.2507
P25: 0.6753
P50: 1.4897
P90: 2.9189

Umbral operacional revisión: 0.6435
Umbral cobertura reducida: 178


In [48]:
# ==============================================================================
# CELDA 10.10 — CONFIGURACIÓN OPERACIONAL
# ==============================================================================

CONFIGURACION_INFERENCIA_V11 = {
    "model_version": "1.1.0",

    "architecture": (
        "TF-IDF Word + Char 3-6 "
        "+ SGDClassifier"
    ),

    "margin_is_probability": False,

    "margin_thresholds": {
        "review": float(
            UMBRAL_MARGEN_REVISION_V11
        ),

        "p10": float(
            UMBRAL_MARGEN_P10_V11
        ),

        "p25": float(
            UMBRAL_MARGEN_P25_V11
        ),

        "p50": float(
            UMBRAL_MARGEN_P50_V11
        ),

        "p90": float(
            UMBRAL_MARGEN_P90_V11
        ),
    },

    "coverage": {
        "reject_if_total_features": 0,

        "few_features_threshold": int(
            UMBRAL_FEATURES_POCAS_V11
        ),

        "uses_word_features": True,
        "uses_char_features": True,
    },

    "operational_states": [
        "aceptada",
        "revision",
        "rechazada",
    ],

    "calibration": {
        "method": (
            "5-fold out-of-fold "
            "predictions"
        ),

        "documents": int(
            len(df_oof_v11)
        ),

        "accuracy_oof": float(
            ACCURACY_OOF_V11
        ),

        "f1_macro_oof": float(
            F1_MACRO_OOF_V11
        ),

        "accepted_accuracy": float(
            ACCURACY_ACEPTADAS_OOF
        ),

        "error_capture_rate": float(
            TASA_CAPTURA_ERRORES_OOF
        ),
    },
}

In [49]:
# ==============================================================================
# CELDA 10.11 — EXPORTACIÓN
# ==============================================================================

PATH_OOF_V11 = (
    REPORTS_V11
    / "predicciones_oof_v1_1.csv"
)


PATH_CALIBRACION_MARGEN_V11 = (
    REPORTS_V11
    / "calibracion_margen_v1_1.csv"
)


PATH_CALIDAD_ESTADO_V11 = (
    REPORTS_V11
    / "calidad_operacional_v1_1.csv"
)


PATH_CONFIG_INFERENCIA_V11 = (
    REPORTS_V11
    / "configuracion_inferencia_v1_1.json"
)


df_oof_v11.to_csv(
    PATH_OOF_V11,
    index=False,
    encoding="utf-8-sig",
)


df_calibracion_margen.to_csv(
    PATH_CALIBRACION_MARGEN_V11,
    index=False,
    encoding="utf-8-sig",
)


df_calidad_estado.to_csv(
    PATH_CALIDAD_ESTADO_V11,
    index=False,
    encoding="utf-8-sig",
)


with open(
    PATH_CONFIG_INFERENCIA_V11,
    "w",
    encoding="utf-8",
) as archivo:

    json.dump(
        CONFIGURACION_INFERENCIA_V11,
        archivo,
        ensure_ascii=False,
        indent=2,
    )


print("Archivos exportados:")

for ruta in [
    PATH_OOF_V11,
    PATH_CALIBRACION_MARGEN_V11,
    PATH_CALIDAD_ESTADO_V11,
    PATH_CONFIG_INFERENCIA_V11,
]:

    print(" -", ruta)

Archivos exportados:
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\predicciones_oof_v1_1.csv
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\calibracion_margen_v1_1.csv
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\calidad_operacional_v1_1.csv
 - C:\Users\MAMÁ\Downloads\reports\v1.1.0\configuracion_inferencia_v1_1.json


In [50]:
# ==============================================================================
# CELDA 10.12 — VALIDACIÓN FINAL DE CALIBRACIÓN
# ==============================================================================

DIFERENCIA_OOF_CV = abs(
    F1_MACRO_OOF_V11
    - MEJOR_F1_OPT_V11
)


CRITERIO_OOF_CONSISTENTE = (
    DIFERENCIA_OOF_CV
    <= 0.01
)


CRITERIO_ACEPTADAS_MEJORAN = (
    ACCURACY_ACEPTADAS_OOF
    >= ACCURACY_OOF_V11
)


TASA_REVISION_TOTAL = float(
    (
        df_oof_v11[
            "estado_operacional"
        ] != "aceptada"
    ).mean()
)


CRITERIO_REVISION_RAZONABLE = (
    TASA_REVISION_TOTAL
    <= 0.35
)


CRITERIO_COBERTURA_CONFIGURADA = (
    UMBRAL_FEATURES_POCAS_V11
    >= 1
)


CALIBRACION_OPERATIVA_APROBADA_V11 = all([
    CRITERIO_OOF_CONSISTENTE,
    CRITERIO_ACEPTADAS_MEJORAN,
    CRITERIO_REVISION_RAZONABLE,
    CRITERIO_COBERTURA_CONFIGURADA,
])


print("=" * 80)
print("APROBACIÓN — CALIBRACIÓN OPERATIVA v1.1.0")
print("=" * 80)

print(
    "OOF consistente con CV:",
    CRITERIO_OOF_CONSISTENTE
)

print(
    "Accuracy aceptadas mejora:",
    CRITERIO_ACEPTADAS_MEJORAN
)

print(
    "Tasa de revisión razonable:",
    CRITERIO_REVISION_RAZONABLE
)

print(
    "Cobertura configurada:",
    CRITERIO_COBERTURA_CONFIGURADA
)


print(
    f"\nTasa revisión/rechazo: "
    f"{TASA_REVISION_TOTAL:.2%}"
)


print(
    "\nCalibración operacional aprobada:",
    CALIBRACION_OPERATIVA_APROBADA_V11
)

APROBACIÓN — CALIBRACIÓN OPERATIVA v1.1.0
OOF consistente con CV: True
Accuracy aceptadas mejora: True
Tasa de revisión razonable: True
Cobertura configurada: True

Tasa revisión/rechazo: 27.39%

Calibración operacional aprobada: True


In [51]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 11.1 — EVALUACIÓN FINAL SOBRE TEST RESERVADO
# ==============================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from pathlib import Path

import hashlib
import json
import joblib
import numpy as np
import pandas as pd


# ==============================================================================
# SEGURIDAD METODOLÓGICA
# ==============================================================================

print("=" * 80)
print("EVALUACIÓN FINAL — TECHMIND v1.1.0")
print("=" * 80)

print("Modelo:")
print(NOMBRE_MODELO_FINAL_V11)

print("\nTrain utilizado:")
print(len(X_train))

print("\nTest reservado:")
print(len(X_test))

print(
    "\nIMPORTANTE: estos resultados no se utilizarán "
    "para volver a optimizar el modelo."
)


# ==============================================================================
# PREDICCIÓN
# ==============================================================================

modelo_final_v11 = (
    MODELO_CANDIDATO_FINAL_V11
)


y_pred_v11 = modelo_final_v11.predict(
    X_test
)


scores_test_v11 = np.asarray(
    modelo_final_v11.decision_function(
        X_test
    )
)


clases_v11 = np.asarray(
    modelo_final_v11.named_steps[
        "clasificador"
    ].classes_
)


# ==============================================================================
# MÉTRICAS
# ==============================================================================

ACCURACY_TEST_V11 = float(
    accuracy_score(
        y_test,
        y_pred_v11
    )
)


PRECISION_MACRO_TEST_V11 = float(
    precision_score(
        y_test,
        y_pred_v11,
        average="macro",
    )
)


RECALL_MACRO_TEST_V11 = float(
    recall_score(
        y_test,
        y_pred_v11,
        average="macro",
    )
)


F1_MACRO_TEST_V11 = float(
    f1_score(
        y_test,
        y_pred_v11,
        average="macro",
    )
)


F1_WEIGHTED_TEST_V11 = float(
    f1_score(
        y_test,
        y_pred_v11,
        average="weighted",
    )
)


DIFERENCIA_TEST_CV_V11 = (
    F1_MACRO_TEST_V11
    - MEJOR_F1_OPT_V11
)


print("\n" + "=" * 80)
print("MÉTRICAS DEFINITIVAS v1.1.0")
print("=" * 80)

print(
    f"Accuracy:        "
    f"{ACCURACY_TEST_V11:.4f}"
)

print(
    f"Precision Macro: "
    f"{PRECISION_MACRO_TEST_V11:.4f}"
)

print(
    f"Recall Macro:    "
    f"{RECALL_MACRO_TEST_V11:.4f}"
)

print(
    f"F1 Macro:        "
    f"{F1_MACRO_TEST_V11:.4f}"
)

print(
    f"F1 Weighted:     "
    f"{F1_WEIGHTED_TEST_V11:.4f}"
)

print(
    f"\nF1 Macro CV:     "
    f"{MEJOR_F1_OPT_V11:.4f}"
)

print(
    f"Diferencia test-CV: "
    f"{DIFERENCIA_TEST_CV_V11:+.4f}"
)

EVALUACIÓN FINAL — TECHMIND v1.1.0
Modelo:
TF-IDF Word + Char 3-6 + SGDClassifier optimizado

Train utilizado:
3666

Test reservado:
917

IMPORTANTE: estos resultados no se utilizarán para volver a optimizar el modelo.

MÉTRICAS DEFINITIVAS v1.1.0
Accuracy:        0.8430
Precision Macro: 0.8455
Recall Macro:    0.8434
F1 Macro:        0.8441
F1 Weighted:     0.8435

F1 Macro CV:     0.8493
Diferencia test-CV: -0.0052


In [52]:
# ==============================================================================
# CELDA 11.2 — REPORTE POR CATEGORÍA
# ==============================================================================

reporte_v11 = classification_report(
    y_test,
    y_pred_v11,
    labels=clases_v11,
    output_dict=True,
    zero_division=0,
)


df_reporte_v11 = (
    pd.DataFrame(
        reporte_v11
    )
    .T
)


df_metricas_categoria_v11 = (
    df_reporte_v11
    .loc[
        clases_v11,
        [
            "precision",
            "recall",
            "f1-score",
            "support",
        ]
    ]
    .reset_index()
    .rename(
        columns={
            "index": "categoria",
            "f1-score": "f1",
        }
    )
)


print("=" * 80)
print("MÉTRICAS POR CATEGORÍA")
print("=" * 80)

print(
    df_metricas_categoria_v11
    .round(4)
)


MEJOR_CATEGORIA_V11 = (
    df_metricas_categoria_v11
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)


CATEGORIA_DIFICIL_V11 = (
    df_metricas_categoria_v11
    .sort_values(
        "f1",
        ascending=True
    )
    .iloc[0]
)


print(
    "\nMejor categoría:",
    MEJOR_CATEGORIA_V11[
        "categoria"
    ],
    f"(F1={MEJOR_CATEGORIA_V11['f1']:.4f})"
)


print(
    "Categoría más difícil:",
    CATEGORIA_DIFICIL_V11[
        "categoria"
    ],
    f"(F1={CATEGORIA_DIFICIL_V11['f1']:.4f})"
)

MÉTRICAS POR CATEGORÍA
     categoria  precision  recall      f1  support
0      backend     0.7805  0.8276  0.8033    232.0
1        cloud     0.8383  0.8491  0.8437    232.0
2  datascience     0.9104  0.8773  0.8935    220.0
3     frontend     0.8527  0.8197  0.8359    233.0

Mejor categoría: datascience (F1=0.8935)
Categoría más difícil: backend (F1=0.8033)


In [53]:
# ==============================================================================
# CELDA 11.3 — MATRIZ DE CONFUSIÓN
# ==============================================================================

matriz_confusion_v11 = confusion_matrix(
    y_test,
    y_pred_v11,
    labels=clases_v11,
)


df_matriz_confusion_v11 = pd.DataFrame(
    matriz_confusion_v11,
    index=clases_v11,
    columns=clases_v11,
)


matriz_confusion_normalizada_v11 = (
    matriz_confusion_v11
    / matriz_confusion_v11.sum(
        axis=1,
        keepdims=True
    )
)


df_matriz_confusion_normalizada_v11 = (
    pd.DataFrame(
        matriz_confusion_normalizada_v11,
        index=clases_v11,
        columns=clases_v11,
    )
)


print("=" * 80)
print("MATRIZ DE CONFUSIÓN")
print("=" * 80)

print(
    df_matriz_confusion_v11
)


print("\nMatriz normalizada:")

print(
    df_matriz_confusion_normalizada_v11
    .round(4)
)

MATRIZ DE CONFUSIÓN
             backend  cloud  datascience  frontend
backend          192     13            9        18
cloud             20    197            6         9
datascience       12      9          193         6
frontend          22     16            4       191

Matriz normalizada:
             backend   cloud  datascience  frontend
backend       0.8276  0.0560       0.0388    0.0776
cloud         0.0862  0.8491       0.0259    0.0388
datascience   0.0545  0.0409       0.8773    0.0273
frontend      0.0944  0.0687       0.0172    0.8197


In [54]:
# ==============================================================================
# CELDA 11.4 — EVALUACIÓN OPERACIONAL SOBRE TEST
# ==============================================================================

feature_union_final = (
    modelo_final_v11.named_steps[
        "features"
    ]
)


transformadores_final = dict(
    feature_union_final.transformer_list
)


vectorizador_word_final = (
    transformadores_final[
        "word"
    ]
)


vectorizador_char_final = (
    transformadores_final[
        "char"
    ]
)


matriz_word_test = (
    vectorizador_word_final.transform(
        X_test
    )
)


matriz_char_test = (
    vectorizador_char_final.transform(
        X_test
    )
)


registros_test_v11 = []


for i in range(
    len(X_test)
):

    fila_scores = (
        scores_test_v11[i]
    )

    orden = np.argsort(
        fila_scores
    )[::-1]

    ganador = int(
        orden[0]
    )

    segundo = int(
        orden[1]
    )


    categoria_predicha = str(
        clases_v11[ganador]
    )

    segunda_categoria = str(
        clases_v11[segundo]
    )


    categoria_real = str(
        y_test.iloc[i]
    )


    margen = float(
        fila_scores[ganador]
        - fila_scores[segundo]
    )


    word_activas = int(
        matriz_word_test[
            i
        ].getnnz()
    )

    char_activas = int(
        matriz_char_test[
            i
        ].getnnz()
    )

    features_total = (
        word_activas
        + char_activas
    )


    estado, motivo = (
        asignar_estado_operacional_v11(
            margen=margen,
            features_activas=features_total,
        )
    )


    registros_test_v11.append({
        "indice_original": int(
            X_test.index[i]
        ),

        "texto": str(
            X_test.iloc[i]
        ),

        "categoria_real": (
            categoria_real
        ),

        "categoria_predicha": (
            categoria_predicha
        ),

        "segunda_categoria": (
            segunda_categoria
        ),

        "prediccion_correcta": (
            categoria_real
            == categoria_predicha
        ),

        "puntuacion_ganadora": float(
            fila_scores[ganador]
        ),

        "puntuacion_segunda": float(
            fila_scores[segundo]
        ),

        "margen_decision": (
            margen
        ),

        "word_features_activas": (
            word_activas
        ),

        "char_features_activas": (
            char_activas
        ),

        "features_activas_total": (
            features_total
        ),

        "estado_operacional": (
            estado
        ),

        "motivo_operacional": (
            motivo
        ),
    })


df_predicciones_test_v11 = (
    pd.DataFrame(
        registros_test_v11
    )
)


print(
    "Predicciones procesadas:",
    len(
        df_predicciones_test_v11
    )
)

Predicciones procesadas: 917


In [55]:
# ==============================================================================
# CELDA 11.5 — CALIDAD POR ESTADO
# ==============================================================================

df_estado_test_v11 = (
    df_predicciones_test_v11
    .groupby(
        "estado_operacional"
    )
    .agg(
        documentos=(
            "prediccion_correcta",
            "size"
        ),

        correctas=(
            "prediccion_correcta",
            "sum"
        ),

        accuracy=(
            "prediccion_correcta",
            "mean"
        ),

        margen_promedio=(
            "margen_decision",
            "mean"
        ),

        features_promedio=(
            "features_activas_total",
            "mean"
        ),
    )
    .reset_index()
)


df_estado_test_v11[
    "porcentaje"
] = (
    df_estado_test_v11[
        "documentos"
    ]
    / len(
        df_predicciones_test_v11
    )
)


print("=" * 80)
print("CALIDAD OPERACIONAL — TEST")
print("=" * 80)

print(
    df_estado_test_v11
    .round(4)
)

CALIDAD OPERACIONAL — TEST
  estado_operacional  documentos  correctas  accuracy  margen_promedio  \
0           aceptada         671        638    0.9508           2.0348   
1           revision         246        135    0.5488           0.5319   

   features_promedio  porcentaje  
0           315.8301      0.7317  
1           292.2033      0.2683  


In [56]:
# ==============================================================================
# CELDA 11.6 — EFECTIVIDAD DE LA REVISIÓN
# ==============================================================================

mask_aceptadas_test = (
    df_predicciones_test_v11[
        "estado_operacional"
    ] == "aceptada"
)


mask_revision_test = (
    df_predicciones_test_v11[
        "estado_operacional"
    ].isin([
        "revision",
        "rechazada",
    ])
)


errores_test = (
    ~df_predicciones_test_v11[
        "prediccion_correcta"
    ]
)


TASA_REVISION_TEST_V11 = float(
    mask_revision_test.mean()
)


if mask_aceptadas_test.sum() > 0:

    ACCURACY_ACEPTADAS_TEST_V11 = float(
        df_predicciones_test_v11.loc[
            mask_aceptadas_test,
            "prediccion_correcta",
        ].mean()
    )

else:

    ACCURACY_ACEPTADAS_TEST_V11 = 0.0


TOTAL_ERRORES_TEST_V11 = int(
    errores_test.sum()
)


ERRORES_CAPTURADOS_TEST_V11 = int(
    (
        errores_test
        & mask_revision_test
    ).sum()
)


if TOTAL_ERRORES_TEST_V11 > 0:

    TASA_CAPTURA_ERRORES_TEST_V11 = (
        ERRORES_CAPTURADOS_TEST_V11
        / TOTAL_ERRORES_TEST_V11
    )

else:

    TASA_CAPTURA_ERRORES_TEST_V11 = 0.0


DOCUMENTOS_SIN_COBERTURA_TEST_V11 = int(
    (
        df_predicciones_test_v11[
            "features_activas_total"
        ] == 0
    ).sum()
)


print("=" * 80)
print("EFECTIVIDAD OPERACIONAL — TEST")
print("=" * 80)

print(
    f"Accuracy global: "
    f"{ACCURACY_TEST_V11:.2%}"
)

print(
    f"Accuracy aceptadas: "
    f"{ACCURACY_ACEPTADAS_TEST_V11:.2%}"
)

print(
    f"Tasa revisión/rechazo: "
    f"{TASA_REVISION_TEST_V11:.2%}"
)

print(
    "Errores totales:",
    TOTAL_ERRORES_TEST_V11
)

print(
    "Errores capturados por revisión:",
    ERRORES_CAPTURADOS_TEST_V11
)

print(
    f"Tasa captura de errores: "
    f"{TASA_CAPTURA_ERRORES_TEST_V11:.2%}"
)

print(
    "Documentos sin cobertura:",
    DOCUMENTOS_SIN_COBERTURA_TEST_V11
)

EFECTIVIDAD OPERACIONAL — TEST
Accuracy global: 84.30%
Accuracy aceptadas: 95.08%
Tasa revisión/rechazo: 26.83%
Errores totales: 144
Errores capturados por revisión: 111
Tasa captura de errores: 77.08%
Documentos sin cobertura: 0


In [57]:
# ==============================================================================
# CELDA 11.7 — COMPARACIÓN v1.0.0 vs v1.1.0
# ==============================================================================

y_pred_v1 = modelo_v1.predict(
    X_test
)


ACCURACY_TEST_V1 = float(
    accuracy_score(
        y_test,
        y_pred_v1
    )
)


PRECISION_MACRO_TEST_V1 = float(
    precision_score(
        y_test,
        y_pred_v1,
        average="macro",
    )
)


RECALL_MACRO_TEST_V1 = float(
    recall_score(
        y_test,
        y_pred_v1,
        average="macro",
    )
)


F1_MACRO_TEST_V1 = float(
    f1_score(
        y_test,
        y_pred_v1,
        average="macro",
    )
)


F1_WEIGHTED_TEST_V1 = float(
    f1_score(
        y_test,
        y_pred_v1,
        average="weighted",
    )
)


df_comparacion_final_versiones = (
    pd.DataFrame([
        {
            "version": "1.0.0",
            "accuracy": (
                ACCURACY_TEST_V1
            ),
            "precision_macro": (
                PRECISION_MACRO_TEST_V1
            ),
            "recall_macro": (
                RECALL_MACRO_TEST_V1
            ),
            "f1_macro": (
                F1_MACRO_TEST_V1
            ),
            "f1_weighted": (
                F1_WEIGHTED_TEST_V1
            ),
        },
        {
            "version": "1.1.0",
            "accuracy": (
                ACCURACY_TEST_V11
            ),
            "precision_macro": (
                PRECISION_MACRO_TEST_V11
            ),
            "recall_macro": (
                RECALL_MACRO_TEST_V11
            ),
            "f1_macro": (
                F1_MACRO_TEST_V11
            ),
            "f1_weighted": (
                F1_WEIGHTED_TEST_V11
            ),
        },
    ])
)


print("=" * 80)
print("COMPARACIÓN FINAL DE VERSIONES")
print("=" * 80)

print(
    df_comparacion_final_versiones
    .round(4)
)


print("\nCambios v1.1.0 vs v1.0.0:")

print(
    f"Accuracy: "
    f"{ACCURACY_TEST_V11 - ACCURACY_TEST_V1:+.4f}"
)

print(
    f"Precision Macro: "
    f"{PRECISION_MACRO_TEST_V11 - PRECISION_MACRO_TEST_V1:+.4f}"
)

print(
    f"Recall Macro: "
    f"{RECALL_MACRO_TEST_V11 - RECALL_MACRO_TEST_V1:+.4f}"
)

print(
    f"F1 Macro: "
    f"{F1_MACRO_TEST_V11 - F1_MACRO_TEST_V1:+.4f}"
)

COMPARACIÓN FINAL DE VERSIONES
  version  accuracy  precision_macro  recall_macro  f1_macro  f1_weighted
0   1.0.0    0.8386           0.8445        0.8385    0.8401       0.8397
1   1.1.0    0.8430           0.8455        0.8434    0.8441       0.8435

Cambios v1.1.0 vs v1.0.0:
Accuracy: +0.0044
Precision Macro: +0.0010
Recall Macro: +0.0049
F1 Macro: +0.0040


In [58]:
# ==============================================================================
# CELDA 11.8 — CAMBIOS DE PREDICCIÓN ENTRE VERSIONES
# ==============================================================================

df_cambios_versiones = pd.DataFrame({
    "indice_original": (
        X_test.index
    ),

    "categoria_real": (
        y_test.values
    ),

    "prediccion_v1_0": (
        y_pred_v1
    ),

    "prediccion_v1_1": (
        y_pred_v11
    ),
})


df_cambios_versiones[
    "correcta_v1_0"
] = (
    df_cambios_versiones[
        "prediccion_v1_0"
    ]
    == df_cambios_versiones[
        "categoria_real"
    ]
)


df_cambios_versiones[
    "correcta_v1_1"
] = (
    df_cambios_versiones[
        "prediccion_v1_1"
    ]
    == df_cambios_versiones[
        "categoria_real"
    ]
)


df_cambios_versiones[
    "prediccion_cambio"
] = (
    df_cambios_versiones[
        "prediccion_v1_0"
    ]
    != df_cambios_versiones[
        "prediccion_v1_1"
    ]
)


df_cambios_versiones[
    "corregido_por_v1_1"
] = (
    ~df_cambios_versiones[
        "correcta_v1_0"
    ]
    & df_cambios_versiones[
        "correcta_v1_1"
    ]
)


df_cambios_versiones[
    "empeorado_por_v1_1"
] = (
    df_cambios_versiones[
        "correcta_v1_0"
    ]
    & ~df_cambios_versiones[
        "correcta_v1_1"
    ]
)


CAMBIOS_TOTALES_V11 = int(
    df_cambios_versiones[
        "prediccion_cambio"
    ].sum()
)


CORREGIDOS_V11 = int(
    df_cambios_versiones[
        "corregido_por_v1_1"
    ].sum()
)


EMPEORADOS_V11 = int(
    df_cambios_versiones[
        "empeorado_por_v1_1"
    ].sum()
)


print("=" * 80)
print("IMPACTO DE LA NUEVA VERSIÓN")
print("=" * 80)

print(
    "Predicciones diferentes:",
    CAMBIOS_TOTALES_V11
)

print(
    "Errores v1.0 corregidos por v1.1:",
    CORREGIDOS_V11
)

print(
    "Aciertos v1.0 perdidos por v1.1:",
    EMPEORADOS_V11
)

print(
    "Balance neto:",
    CORREGIDOS_V11
    - EMPEORADOS_V11
)

IMPACTO DE LA NUEVA VERSIÓN
Predicciones diferentes: 72
Errores v1.0 corregidos por v1.1: 31
Aciertos v1.0 perdidos por v1.1: 27
Balance neto: 4


In [59]:
# ==============================================================================
# CELDA 11.9 — F1 POR CATEGORÍA v1.0 vs v1.1
# ==============================================================================

reporte_v1 = classification_report(
    y_test,
    y_pred_v1,
    labels=clases_v11,
    output_dict=True,
    zero_division=0,
)


registros_categoria_comparacion = []


for categoria in clases_v11:

    f1_v1_categoria = float(
        reporte_v1[
            categoria
        ][
            "f1-score"
        ]
    )

    f1_v11_categoria = float(
        reporte_v11[
            categoria
        ][
            "f1-score"
        ]
    )


    registros_categoria_comparacion.append({
        "categoria": categoria,
        "f1_v1_0": (
            f1_v1_categoria
        ),
        "f1_v1_1": (
            f1_v11_categoria
        ),
        "diferencia": (
            f1_v11_categoria
            - f1_v1_categoria
        ),
    })


df_comparacion_categoria = (
    pd.DataFrame(
        registros_categoria_comparacion
    )
)


print("=" * 80)
print("EVOLUCIÓN POR CATEGORÍA")
print("=" * 80)

print(
    df_comparacion_categoria
    .round(4)
)

EVOLUCIÓN POR CATEGORÍA
     categoria  f1_v1_0  f1_v1_1  diferencia
0      backend   0.8000   0.8033      0.0033
1        cloud   0.8460   0.8437     -0.0023
2  datascience   0.8735   0.8935      0.0200
3     frontend   0.8410   0.8359     -0.0051


In [60]:
# ==============================================================================
# CELDA 11.10 — APROBACIÓN FINAL DEL MODELO
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. GENERALIZACIÓN
# ------------------------------------------------------------------------------

CRITERIO_GENERALIZACION_TEST_V11 = (
    abs(
        DIFERENCIA_TEST_CV_V11
    )
    <= 0.03
)


# ------------------------------------------------------------------------------
# 2. SIN DEGRADACIÓN MATERIAL VS v1.0
#
# Permitimos como máximo -0.005 de F1 debido a que v1.1 aporta
# mejoras importantes de robustez y cobertura.
# ------------------------------------------------------------------------------

CRITERIO_F1_TEST_V11 = (
    F1_MACRO_TEST_V11
    >= F1_MACRO_TEST_V1 - 0.005
)


# ------------------------------------------------------------------------------
# 3. FILTRADO OPERACIONAL ÚTIL
# ------------------------------------------------------------------------------

CRITERIO_ACEPTADAS_TEST_V11 = (
    ACCURACY_ACEPTADAS_TEST_V11
    >= ACCURACY_TEST_V11
)


# ------------------------------------------------------------------------------
# 4. COBERTURA
# ------------------------------------------------------------------------------

CRITERIO_COBERTURA_TEST_V11 = (
    DOCUMENTOS_SIN_COBERTURA_TEST_V11
    <= max(
        1,
        int(
            len(X_test)
            * 0.01
        )
    )
)


# ------------------------------------------------------------------------------
# 5. TASA DE REVISIÓN
# ------------------------------------------------------------------------------

CRITERIO_REVISION_TEST_V11 = (
    TASA_REVISION_TEST_V11
    <= 0.35
)


# ------------------------------------------------------------------------------
# 6. PRUEBAS EXTERNAS YA APROBADAS
# ------------------------------------------------------------------------------

CRITERIO_EXTERNOS_V11 = (
    CASOS_CORRECTOS_OPT_V11
    == len(
        df_diagnostico_optimizado_v11
    )
    and
    CASOS_SIN_COBERTURA_OPT_V11
    == 0
)


# ------------------------------------------------------------------------------
# APROBACIÓN
# ------------------------------------------------------------------------------

MODELO_FINAL_APROBADO_V11 = all([
    CRITERIO_GENERALIZACION_TEST_V11,
    CRITERIO_F1_TEST_V11,
    CRITERIO_ACEPTADAS_TEST_V11,
    CRITERIO_COBERTURA_TEST_V11,
    CRITERIO_REVISION_TEST_V11,
    CRITERIO_EXTERNOS_V11,
])


print("=" * 80)
print("APROBACIÓN FINAL — TECHMIND v1.1.0")
print("=" * 80)

print(
    "Generalización consistente:",
    CRITERIO_GENERALIZACION_TEST_V11
)

print(
    "F1 test aprobado:",
    CRITERIO_F1_TEST_V11
)

print(
    "Accuracy aceptadas mejora:",
    CRITERIO_ACEPTADAS_TEST_V11
)

print(
    "Cobertura test aprobada:",
    CRITERIO_COBERTURA_TEST_V11
)

print(
    "Tasa revisión aprobada:",
    CRITERIO_REVISION_TEST_V11
)

print(
    "Pruebas externas aprobadas:",
    CRITERIO_EXTERNOS_V11
)


print(
    "\nMODELO FINAL v1.1.0 APROBADO:",
    MODELO_FINAL_APROBADO_V11
)

APROBACIÓN FINAL — TECHMIND v1.1.0
Generalización consistente: True
F1 test aprobado: True
Accuracy aceptadas mejora: True
Cobertura test aprobada: True
Tasa revisión aprobada: True
Pruebas externas aprobadas: True

MODELO FINAL v1.1.0 APROBADO: True


In [61]:
# ==============================================================================
# CELDA 11.11 — EXPORTACIÓN DE RESULTADOS
# ==============================================================================

REPORTS_FINAL_V11 = (
    REPORTS_V11
    / "final"
)


REPORTS_FINAL_V11.mkdir(
    parents=True,
    exist_ok=True
)


PATH_METRICAS_TEST_V11 = (
    REPORTS_FINAL_V11
    / "metricas_finales_test_v1_1.json"
)


PATH_REPORTE_CATEGORIAS_V11 = (
    REPORTS_FINAL_V11
    / "metricas_por_categoria_v1_1.csv"
)


PATH_MATRIZ_V11 = (
    REPORTS_FINAL_V11
    / "matriz_confusion_v1_1.csv"
)


PATH_MATRIZ_NORMALIZADA_V11 = (
    REPORTS_FINAL_V11
    / "matriz_confusion_normalizada_v1_1.csv"
)


PATH_PREDICCIONES_TEST_V11 = (
    REPORTS_FINAL_V11
    / "predicciones_test_v1_1.csv"
)


PATH_COMPARACION_VERSIONES = (
    REPORTS_FINAL_V11
    / "comparacion_v1_0_vs_v1_1.csv"
)


PATH_COMPARACION_CATEGORIAS = (
    REPORTS_FINAL_V11
    / "comparacion_categorias_v1_0_vs_v1_1.csv"
)


PATH_CAMBIOS_VERSIONES = (
    REPORTS_FINAL_V11
    / "cambios_predicciones_v1_0_vs_v1_1.csv"
)


metricas_finales_v11 = {
    "version": "1.1.0",

    "model_name": (
        NOMBRE_MODELO_FINAL_V11
    ),

    "test_documents": int(
        len(X_test)
    ),

    "accuracy": (
        ACCURACY_TEST_V11
    ),

    "precision_macro": (
        PRECISION_MACRO_TEST_V11
    ),

    "recall_macro": (
        RECALL_MACRO_TEST_V11
    ),

    "f1_macro": (
        F1_MACRO_TEST_V11
    ),

    "f1_weighted": (
        F1_WEIGHTED_TEST_V11
    ),

    "f1_macro_cv": (
        float(
            MEJOR_F1_OPT_V11
        )
    ),

    "test_cv_difference": (
        float(
            DIFERENCIA_TEST_CV_V11
        )
    ),

    "review_rate": (
        TASA_REVISION_TEST_V11
    ),

    "accepted_accuracy": (
        ACCURACY_ACEPTADAS_TEST_V11
    ),

    "error_capture_rate": (
        TASA_CAPTURA_ERRORES_TEST_V11
    ),

    "documents_without_coverage": (
        DOCUMENTOS_SIN_COBERTURA_TEST_V11
    ),

    "approved": bool(
        MODELO_FINAL_APROBADO_V11
    ),
}


with open(
    PATH_METRICAS_TEST_V11,
    "w",
    encoding="utf-8",
) as archivo:

    json.dump(
        metricas_finales_v11,
        archivo,
        ensure_ascii=False,
        indent=2,
    )


df_metricas_categoria_v11.to_csv(
    PATH_REPORTE_CATEGORIAS_V11,
    index=False,
    encoding="utf-8-sig",
)


df_matriz_confusion_v11.to_csv(
    PATH_MATRIZ_V11,
    encoding="utf-8-sig",
)


df_matriz_confusion_normalizada_v11.to_csv(
    PATH_MATRIZ_NORMALIZADA_V11,
    encoding="utf-8-sig",
)


df_predicciones_test_v11.to_csv(
    PATH_PREDICCIONES_TEST_V11,
    index=False,
    encoding="utf-8-sig",
)


df_comparacion_final_versiones.to_csv(
    PATH_COMPARACION_VERSIONES,
    index=False,
    encoding="utf-8-sig",
)


df_comparacion_categoria.to_csv(
    PATH_COMPARACION_CATEGORIAS,
    index=False,
    encoding="utf-8-sig",
)


df_cambios_versiones.to_csv(
    PATH_CAMBIOS_VERSIONES,
    index=False,
    encoding="utf-8-sig",
)


print("Resultados finales exportados.")

Resultados finales exportados.


In [66]:
# ==============================================================================
# CELDA 11.12 — EXPORTACIÓN DEL MODELO FINAL v1.1.0
# ==============================================================================

PATH_MODELO_FINAL_V11 = (
    MODELS_V11
    / "techmind_modelo_final_v1_1_0.joblib"
)


PATH_METADATA_FINAL_V11 = (
    MODELS_V11
    / "techmind_modelo_final_v1_1_0_metadata.json"
)


def calcular_sha256_local(
    ruta,
    chunk_size=1024 * 1024,
):

    sha256 = hashlib.sha256()

    with open(
        ruta,
        "rb"
    ) as archivo:

        while True:

            bloque = archivo.read(
                chunk_size
            )

            if not bloque:
                break

            sha256.update(
                bloque
            )

    return sha256.hexdigest()


if MODELO_FINAL_APROBADO_V11:

    joblib.dump(
        modelo_final_v11,
        PATH_MODELO_FINAL_V11,
    )


    SHA256_MODELO_FINAL_V11 = (
        calcular_sha256_local(
            PATH_MODELO_FINAL_V11
        )
    )


    metadata_final_v11 = {
        "version": "1.1.0",

        "status": (
            "Modelo final evaluado y aprobado"
        ),

        "model_name": (
            NOMBRE_MODELO_FINAL_V11
        ),

        "architecture": {
            "word_tfidf": True,
            "char_tfidf": True,
            "char_analyzer": "char_wb",
            "char_ngram_range": [
                3,
                6,
            ],
            "stopwords_spanish": True,
            "strip_accents": "unicode",
            "classifier": (
                "SGDClassifier"
            ),
            "alpha": float(
                MEJOR_ALPHA_V11
            ),
        },

        "classes": [
            str(x)
            for x in clases_v11
        ],

        "training_documents": int(
            len(X_train)
        ),

        "test_documents": int(
            len(X_test)
        ),

        "metrics": (
            metricas_finales_v11
        ),

        "operational_thresholds": {
            "review_margin": float(
                UMBRAL_MARGEN_REVISION_V11
            ),

            "few_features": int(
                UMBRAL_FEATURES_POCAS_V11
            ),

            "reject_if_total_features": 0,
        },

        "sha256": (
            SHA256_MODELO_FINAL_V11
        ),
    }


    with open(
        PATH_METADATA_FINAL_V11,
        "w",
        encoding="utf-8",
    ) as archivo:

        json.dump(
            metadata_final_v11,
            archivo,
            ensure_ascii=False,
            indent=2,
        )


    print("=" * 80)
    print("MODELO v1.1.0 EXPORTADO")
    print("=" * 80)

    print(
        "Modelo:"
    )

    print(
        PATH_MODELO_FINAL_V11
    )

    print(
        "\nMetadata:"
    )

    print(
        PATH_METADATA_FINAL_V11
    )

    print(
        "\nSHA-256:"
    )

    print(
        SHA256_MODELO_FINAL_V11
    )

else:

    print(
        "El modelo NO será exportado "
        "como versión final porque no "
        "superó todos los criterios."
    )

MODELO v1.1.0 EXPORTADO
Modelo:
C:\Users\MAMÁ\Downloads\models\v1.1.0\techmind_modelo_final_v1_1_0.joblib

Metadata:
C:\Users\MAMÁ\Downloads\models\v1.1.0\techmind_modelo_final_v1_1_0_metadata.json

SHA-256:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6


In [67]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 11.13 — VERIFICACIÓN DEFINITIVA DEL ARTEFACTO
# ==============================================================================

from pathlib import Path

import hashlib
import json
import joblib
import numpy as np


# ==============================================================================
# 1. LOCALIZAR ARTEFACTOS
# ==============================================================================

if "PATH_MODELO_FINAL_V11" not in globals():

    PATH_MODELO_FINAL_V11 = (
        MODELS_V11
        / "techmind_modelo_final_v1_1_0.joblib"
    )


if "PATH_METADATA_FINAL_V11" not in globals():

    PATH_METADATA_FINAL_V11 = (
        MODELS_V11
        / "techmind_modelo_final_v1_1_0_metadata.json"
    )


print("=" * 80)
print("VERIFICACIÓN DEL ARTEFACTO — TECHMIND v1.1.0")
print("=" * 80)

print("\nModelo:")
print(PATH_MODELO_FINAL_V11)

print("\nMetadata:")
print(PATH_METADATA_FINAL_V11)


# ==============================================================================
# 2. COMPROBAR EXISTENCIA
# ==============================================================================

MODELO_EXISTE_V11 = (
    Path(PATH_MODELO_FINAL_V11).exists()
)

METADATA_EXISTE_V11 = (
    Path(PATH_METADATA_FINAL_V11).exists()
)


print(
    "\nModelo existe:",
    MODELO_EXISTE_V11
)

print(
    "Metadata existe:",
    METADATA_EXISTE_V11
)


if not MODELO_EXISTE_V11:

    raise FileNotFoundError(
        "No existe el modelo final v1.1.0.\n"
        "Ejecuta primero la Celda 11.12."
    )


if not METADATA_EXISTE_V11:

    raise FileNotFoundError(
        "No existe el metadata del modelo v1.1.0.\n"
        "Ejecuta primero la Celda 11.12."
    )


# ==============================================================================
# 3. FUNCIÓN SHA-256
# ==============================================================================

def calcular_sha256_verificacion(
    ruta,
    chunk_size=1024 * 1024,
):

    sha256 = hashlib.sha256()

    with open(
        ruta,
        "rb"
    ) as archivo:

        while True:

            bloque = archivo.read(
                chunk_size
            )

            if not bloque:
                break

            sha256.update(
                bloque
            )

    return sha256.hexdigest()


# ==============================================================================
# 4. LEER METADATA
# ==============================================================================

with open(
    PATH_METADATA_FINAL_V11,
    "r",
    encoding="utf-8",
) as archivo:

    metadata_verificacion_v11 = (
        json.load(
            archivo
        )
    )


SHA256_ESPERADO_V11 = (
    metadata_verificacion_v11.get(
        "sha256"
    )
)


print(
    "\nSHA esperado:"
)

print(
    SHA256_ESPERADO_V11
)


# ==============================================================================
# 5. CALCULAR SHA ACTUAL
# ==============================================================================

SHA256_ACTUAL_V11 = (
    calcular_sha256_verificacion(
        PATH_MODELO_FINAL_V11
    )
)


print(
    "\nSHA actual:"
)

print(
    SHA256_ACTUAL_V11
)


VERIFICACION_HASH_V11 = bool(
    SHA256_ESPERADO_V11
    and
    SHA256_ACTUAL_V11
    == SHA256_ESPERADO_V11
)


# ==============================================================================
# 6. RECARGAR MODELO
# ==============================================================================

modelo_recargado_v11 = (
    joblib.load(
        PATH_MODELO_FINAL_V11
    )
)


print(
    "\nModelo recargado correctamente."
)


# ==============================================================================
# 7. INSPECCIONAR PIPELINE
# ==============================================================================

PASOS_PIPELINE_RECARGADO_V11 = (
    list(
        modelo_recargado_v11
        .named_steps
        .keys()
    )
)


print(
    "\nPasos pipeline:"
)

print(
    PASOS_PIPELINE_RECARGADO_V11
)


PIPELINE_CORRECTO_V11 = (
    "features"
    in PASOS_PIPELINE_RECARGADO_V11
    and
    "clasificador"
    in PASOS_PIPELINE_RECARGADO_V11
)


# ==============================================================================
# 8. INSPECCIONAR FEATURE UNION
# ==============================================================================

if PIPELINE_CORRECTO_V11:

    feature_union_recargado = (
        modelo_recargado_v11
        .named_steps[
            "features"
        ]
    )

    nombres_transformadores = [
        nombre
        for nombre, _
        in feature_union_recargado.transformer_list
    ]

else:

    nombres_transformadores = []


print(
    "Transformadores:"
)

print(
    nombres_transformadores
)


FEATURE_UNION_CORRECTA_V11 = (
    "word"
    in nombres_transformadores
    and
    "char"
    in nombres_transformadores
)


# ==============================================================================
# 9. VERIFICAR CLASES
# ==============================================================================

clases_recargadas_v11 = np.asarray(
    modelo_recargado_v11
    .named_steps[
        "clasificador"
    ]
    .classes_
)


CLASES_ESPERADAS_V11 = np.array([
    "backend",
    "cloud",
    "datascience",
    "frontend",
])


VERIFICACION_CLASES_V11 = bool(
    np.array_equal(
        clases_recargadas_v11,
        CLASES_ESPERADAS_V11
    )
)


print(
    "\nClases:"
)

print(
    clases_recargadas_v11
)


# ==============================================================================
# 10. REPRODUCIBILIDAD SOBRE LOS 917 DOCUMENTOS
# ==============================================================================

predicciones_recarga_v11 = (
    modelo_recargado_v11.predict(
        X_test
    )
)


# Comparar contra las predicciones originales
# de la evaluación final.
VERIFICACION_RECARGA_V11 = bool(
    np.array_equal(
        predicciones_recarga_v11,
        y_pred_v11
    )
)


NUMERO_DIFERENCIAS_RECARGA = int(
    np.sum(
        predicciones_recarga_v11
        != y_pred_v11
    )
)


# ==============================================================================
# 11. VOLVER A CALCULAR F1
# ==============================================================================

from sklearn.metrics import f1_score


F1_RECARGADO_V11 = float(
    f1_score(
        y_test,
        predicciones_recarga_v11,
        average="macro",
    )
)


VERIFICACION_F1_V11 = bool(
    np.isclose(
        F1_RECARGADO_V11,
        F1_MACRO_TEST_V11,
        atol=1e-12,
    )
)


# ==============================================================================
# 12. APROBACIÓN INTEGRAL
# ==============================================================================

ARTEFACTO_VERIFICADO_V11 = all([
    MODELO_EXISTE_V11,
    METADATA_EXISTE_V11,
    VERIFICACION_HASH_V11,
    PIPELINE_CORRECTO_V11,
    FEATURE_UNION_CORRECTA_V11,
    VERIFICACION_CLASES_V11,
    VERIFICACION_RECARGA_V11,
    VERIFICACION_F1_V11,
])


# ==============================================================================
# 13. RESULTADO
# ==============================================================================

print("\n" + "=" * 80)
print("RESULTADO DE VERIFICACIÓN")
print("=" * 80)

print(
    "SHA-256 consistente:",
    VERIFICACION_HASH_V11
)

print(
    "Pipeline correcto:",
    PIPELINE_CORRECTO_V11
)

print(
    "Word + Char presentes:",
    FEATURE_UNION_CORRECTA_V11
)

print(
    "Clases correctas:",
    VERIFICACION_CLASES_V11
)

print(
    "Predicciones reproducibles:",
    VERIFICACION_RECARGA_V11
)

print(
    "Diferencias de predicción:",
    NUMERO_DIFERENCIAS_RECARGA
)

print(
    f"F1 original: "
    f"{F1_MACRO_TEST_V11:.4f}"
)

print(
    f"F1 recargado: "
    f"{F1_RECARGADO_V11:.4f}"
)

print(
    "F1 reproducible:",
    VERIFICACION_F1_V11
)

print(
    "\nARTEFACTO v1.1.0 VERIFICADO:",
    ARTEFACTO_VERIFICADO_V11
)

VERIFICACIÓN DEL ARTEFACTO — TECHMIND v1.1.0

Modelo:
C:\Users\MAMÁ\Downloads\models\v1.1.0\techmind_modelo_final_v1_1_0.joblib

Metadata:
C:\Users\MAMÁ\Downloads\models\v1.1.0\techmind_modelo_final_v1_1_0_metadata.json

Modelo existe: True
Metadata existe: True

SHA esperado:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6

SHA actual:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6

Modelo recargado correctamente.

Pasos pipeline:
['features', 'clasificador']
Transformadores:
['word', 'char']

Clases:
['backend' 'cloud' 'datascience' 'frontend']

RESULTADO DE VERIFICACIÓN
SHA-256 consistente: True
Pipeline correcto: True
Word + Char presentes: True
Clases correctas: True
Predicciones reproducibles: True
Diferencias de predicción: 0
F1 original: 0.8441
F1 recargado: 0.8441
F1 reproducible: True

ARTEFACTO v1.1.0 VERIFICADO: True


In [68]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 11.14 — CONCLUSIÓN FINAL ROBUSTA
# ==============================================================================

# ------------------------------------------------------------------------------
# RECUPERAR VALIDACIONES
# ------------------------------------------------------------------------------

verificacion_recarga = bool(
    globals().get(
        "VERIFICACION_RECARGA_V11",
        False
    )
)

verificacion_hash = bool(
    globals().get(
        "VERIFICACION_HASH_V11",
        False
    )
)


# ==============================================================================
# APROBACIÓN INTEGRAL
# ==============================================================================

ARTEFACTO_FINAL_APROBADO_V11 = (
    MODELO_FINAL_APROBADO_V11
    and verificacion_recarga
    and verificacion_hash
)


# ==============================================================================
# CONCLUSIÓN
# ==============================================================================

print("=" * 80)
print("CONCLUSIÓN — TECHMIND MODEL v1.1.0")
print("=" * 80)

print(
    "Modelo:",
    NOMBRE_MODELO_FINAL_V11
)

print(
    f"\nF1 Macro CV: "
    f"{MEJOR_F1_OPT_V11:.4f}"
)

print(
    f"F1 Macro Test: "
    f"{F1_MACRO_TEST_V11:.4f}"
)

print(
    f"Diferencia test-CV: "
    f"{DIFERENCIA_TEST_CV_V11:+.4f}"
)

print(
    f"\nAccuracy Test: "
    f"{ACCURACY_TEST_V11:.4f}"
)

print(
    f"Precision Macro: "
    f"{PRECISION_MACRO_TEST_V11:.4f}"
)

print(
    f"Recall Macro: "
    f"{RECALL_MACRO_TEST_V11:.4f}"
)

print(
    f"F1 Weighted: "
    f"{F1_WEIGHTED_TEST_V11:.4f}"
)


# ==============================================================================
# OPERACIÓN
# ==============================================================================

print(
    f"\nAccuracy predicciones aceptadas: "
    f"{ACCURACY_ACEPTADAS_TEST_V11:.4f}"
)

print(
    f"Tasa revisión/rechazo: "
    f"{TASA_REVISION_TEST_V11:.2%}"
)

print(
    f"Captura de errores: "
    f"{TASA_CAPTURA_ERRORES_TEST_V11:.2%}"
)

print(
    "Documentos sin cobertura:",
    DOCUMENTOS_SIN_COBERTURA_TEST_V11
)


# ==============================================================================
# PRUEBAS EXTERNAS
# ==============================================================================

print(
    "\nCasos externos correctos:",
    CASOS_CORRECTOS_OPT_V11,
    "/",
    len(
        df_diagnostico_optimizado_v11
    )
)

print(
    "Casos externos sin cobertura:",
    CASOS_SIN_COBERTURA_OPT_V11
)


# ==============================================================================
# COMPARACIÓN CONTRA v1.0.0
# ==============================================================================

print(
    f"\nCambio F1 vs v1.0.0: "
    f"{F1_MACRO_TEST_V11 - F1_MACRO_TEST_V1:+.4f}"
)


# ==============================================================================
# VERIFICACIÓN DEL ARTEFACTO
# ==============================================================================

print(
    "\nModelo reproducible:",
    verificacion_recarga
)

print(
    "Integridad SHA-256:",
    verificacion_hash
)


# ==============================================================================
# ESTADO
# ==============================================================================

print(
    "\nModelo estadísticamente aprobado:",
    MODELO_FINAL_APROBADO_V11
)

print(
    "Artefacto final aprobado:",
    ARTEFACTO_FINAL_APROBADO_V11
)


if ARTEFACTO_FINAL_APROBADO_V11:

    print(
        "\nEstado final: APROBADO."
    )

else:

    print(
        "\nEstado final: PENDIENTE DE "
        "VERIFICACIÓN DEL ARTEFACTO."
    )

CONCLUSIÓN — TECHMIND MODEL v1.1.0
Modelo: TF-IDF Word + Char 3-6 + SGDClassifier optimizado

F1 Macro CV: 0.8493
F1 Macro Test: 0.8441
Diferencia test-CV: -0.0052

Accuracy Test: 0.8430
Precision Macro: 0.8455
Recall Macro: 0.8434
F1 Weighted: 0.8435

Accuracy predicciones aceptadas: 0.9508
Tasa revisión/rechazo: 26.83%
Captura de errores: 77.08%
Documentos sin cobertura: 0

Casos externos correctos: 9 / 9
Casos externos sin cobertura: 0

Cambio F1 vs v1.0.0: +0.0040

Modelo reproducible: True
Integridad SHA-256: True

Modelo estadísticamente aprobado: True
Artefacto final aprobado: True

Estado final: APROBADO.


In [69]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.1 — CLONAR PAQUETE v1.0.0
# ==============================================================================

from pathlib import Path

import datetime
import hashlib
import json
import shutil


# ==============================================================================
# VALIDACIÓN
# ==============================================================================

if not MODELO_FINAL_APROBADO_V11:
    raise RuntimeError(
        "El modelo v1.1.0 no está aprobado."
    )

if not VERIFICACION_RECARGA_V11:
    raise RuntimeError(
        "La recarga del modelo v1.1.0 no fue validada."
    )

if not VERIFICACION_HASH_V11:
    raise RuntimeError(
        "El hash del modelo v1.1.0 no fue validado."
    )


# ==============================================================================
# DIRECTORIO EXPORT
# ==============================================================================

EXPORT_ROOT = (
    Path.home()
    / "Downloads"
    / "export"
)


# ==============================================================================
# POSIBLES UBICACIONES DEL PAQUETE v1.0.0
# ==============================================================================

candidatos_paquete_v1 = [
    EXPORT_ROOT / "techmind_model",
    EXPORT_ROOT / "techmind_model_v1.0.0",
]


PACKAGE_V1_ROOT = None


for candidato in candidatos_paquete_v1:

    if (
        candidato.exists()
        and candidato.is_dir()
    ):

        if (
            candidato
            / "techmind"
            / "predictor.py"
        ).exists():

            PACKAGE_V1_ROOT = candidato

            break


if PACKAGE_V1_ROOT is None:

    raise FileNotFoundError(
        "No se encontró el directorio del paquete "
        "TechMind v1.0.0.\n\n"
        f"Se buscó en:\n"
        f"{EXPORT_ROOT}"
    )


print("Paquete fuente v1.0.0:")
print(PACKAGE_V1_ROOT)

Paquete fuente v1.0.0:
C:\Users\MAMÁ\Downloads\export\techmind_model


In [70]:
# ==============================================================================
# CREAR PAQUETE v1.1.0
# ==============================================================================

PACKAGE_V11_ROOT = (
    EXPORT_ROOT
    / "techmind_model_v1.1.0"
)


# ------------------------------------------------------------------------------
# RESPALDAR SI YA EXISTE
# ------------------------------------------------------------------------------

if PACKAGE_V11_ROOT.exists():

    timestamp = datetime.datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    backup = (
        EXPORT_ROOT
        / f"techmind_model_v1.1.0_backup_{timestamp}"
    )

    PACKAGE_V11_ROOT.rename(
        backup
    )

    print(
        "Versión anterior respaldada en:"
    )

    print(
        backup
    )


# ------------------------------------------------------------------------------
# COPIAR v1.0.0
# ------------------------------------------------------------------------------

shutil.copytree(
    PACKAGE_V1_ROOT,
    PACKAGE_V11_ROOT,
)


print("\nNuevo paquete:")
print(PACKAGE_V11_ROOT)


Nuevo paquete:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0


In [71]:
# ==============================================================================
# CELDA 12.2 — ACTUALIZAR ARTEFACTOS
# ==============================================================================

ARTIFACTS_V11 = (
    PACKAGE_V11_ROOT
    / "artifacts"
)


ARTIFACTS_V11.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# DESTINOS COMPATIBLES CON v1.0.0
# ==============================================================================

PATH_MODELO_PAQUETE_V11 = (
    ARTIFACTS_V11
    / "techmind_modelo_final.joblib"
)


PATH_METADATA_PAQUETE_V11 = (
    ARTIFACTS_V11
    / "techmind_modelo_final_metadata.json"
)


PATH_CONFIG_PAQUETE_V11 = (
    ARTIFACTS_V11
    / "configuracion_inferencia.json"
)


# ==============================================================================
# COPIAR MODELO
# ==============================================================================

shutil.copy2(
    PATH_MODELO_FINAL_V11,
    PATH_MODELO_PAQUETE_V11,
)


# ==============================================================================
# COPIAR METADATA
# ==============================================================================

shutil.copy2(
    PATH_METADATA_FINAL_V11,
    PATH_METADATA_PAQUETE_V11,
)


# ==============================================================================
# COPIAR NUEVA CONFIGURACIÓN OPERACIONAL
# ==============================================================================

shutil.copy2(
    PATH_CONFIG_INFERENCIA_V11,
    PATH_CONFIG_PAQUETE_V11,
)


# ==============================================================================
# VERSION
# ==============================================================================

PATH_VERSION_V11 = (
    PACKAGE_V11_ROOT
    / "VERSION"
)


PATH_VERSION_V11.write_text(
    "1.1.0\n",
    encoding="utf-8",
)


# ==============================================================================
# VALIDAR HASH
# ==============================================================================

SHA256_PAQUETE_V11 = (
    calcular_sha256_local(
        PATH_MODELO_PAQUETE_V11
    )
)


MODELO_COPIADO_CORRECTAMENTE = (
    SHA256_PAQUETE_V11
    == SHA256_MODELO_FINAL_V11
)


print("=" * 80)
print("ARTEFACTOS v1.1.0")
print("=" * 80)

print(
    "Modelo:"
)

print(
    PATH_MODELO_PAQUETE_V11
)

print(
    "\nMetadata:"
)

print(
    PATH_METADATA_PAQUETE_V11
)

print(
    "\nConfiguración:"
)

print(
    PATH_CONFIG_PAQUETE_V11
)

print(
    "\nSHA-256:"
)

print(
    SHA256_PAQUETE_V11
)

print(
    "\nModelo copiado correctamente:",
    MODELO_COPIADO_CORRECTAMENTE
)

ARTEFACTOS v1.1.0
Modelo:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\artifacts\techmind_modelo_final.joblib

Metadata:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\artifacts\techmind_modelo_final_metadata.json

Configuración:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\artifacts\configuracion_inferencia.json

SHA-256:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6

Modelo copiado correctamente: True


In [72]:
# ==============================================================================
# CELDA 12.3 — AUDITORÍA DEL PREDICTOR v1.0.0
# ==============================================================================

PATH_PREDICTOR_V11 = (
    PACKAGE_V11_ROOT
    / "techmind"
    / "predictor.py"
)


if not PATH_PREDICTOR_V11.exists():

    raise FileNotFoundError(
        PATH_PREDICTOR_V11
    )


# ==============================================================================
# LEER ARCHIVO
# ==============================================================================

try:

    codigo_predictor = (
        PATH_PREDICTOR_V11
        .read_text(
            encoding="utf-8"
        )
    )

except UnicodeDecodeError:

    codigo_predictor = (
        PATH_PREDICTOR_V11
        .read_text(
            encoding="cp1252"
        )
    )


lineas_predictor = (
    codigo_predictor
    .splitlines()
)


print(
    "Líneas predictor.py:",
    len(lineas_predictor)
)

Líneas predictor.py: 1251


In [73]:
# ==============================================================================
# PATRONES IMPORTANTES
# ==============================================================================

PATRONES_PREDICTOR = [
    "named_steps",
    "tfidf",
    "vectorizador",
    "terminos_activos",
    "few_terms",
    "positive_terms",
    "negative_terms",
    "differential_terms",
    "coef_",
    "get_feature_names_out",
    "decision_function",
    "health",
    "pipeline_steps",
]


coincidencias_predictor = []


for numero, linea in enumerate(
    lineas_predictor,
    start=1,
):

    linea_lower = linea.lower()

    if any(
        patron.lower()
        in linea_lower
        for patron
        in PATRONES_PREDICTOR
    ):

        coincidencias_predictor.append({
            "linea": numero,
            "codigo": linea,
        })


df_auditoria_predictor = pd.DataFrame(
    coincidencias_predictor
)


print("=" * 100)
print("REFERENCIAS RELEVANTES — predictor.py")
print("=" * 100)


pd.set_option(
    "display.max_colwidth",
    None
)


print(
    df_auditoria_predictor
    .to_string(
        index=False
    )
)

REFERENCIAS RELEVANTES — predictor.py
 linea                                        codigo
   169                                 "named_steps"
   178                                      "tfidf",
   186                                  .named_steps
   200                         .named_steps["tfidf"]
   205                  .named_steps["clasificador"]
   214                      .get_feature_names_out()
   226                                       "coef_"
   299               self.few_terms_threshold = int(
   307           def health(self) -> dict[str, Any]:
   323                        "tfidf_features": int(
   327                       "pipeline_steps": list(
   329                                  .named_steps
   543                   <= self.few_terms_threshold
   640                    tfidf_values = vector.data
   645                         "positive_terms": [],
   646                         "negative_terms": [],
   647                     "differential_terms": [],
   658  

In [74]:
# ==============================================================================
# CELDA 12.4 — COMPATIBILIDAD DEL PREDICTOR
# ==============================================================================

USA_TFIDF_DIRECTO = (
    'named_steps["tfidf"]'
    in codigo_predictor

    or

    "named_steps['tfidf']"
    in codigo_predictor
)


USA_FEATURES_V11 = (
    'named_steps["features"]'
    in codigo_predictor

    or

    "named_steps['features']"
    in codigo_predictor
)


USA_EXPLICABILIDAD = any(
    termino in codigo_predictor
    for termino in [
        "positive_terms",
        "differential_terms",
        "coef_",
    ]
)


USA_TERMINOS_ACTIVOS = (
    "terminos_activos"
    in codigo_predictor
)


print("=" * 80)
print("DIAGNÓSTICO DE COMPATIBILIDAD")
print("=" * 80)

print(
    "Acceso directo a tfidf:",
    USA_TFIDF_DIRECTO
)

print(
    "Soporte actual para features:",
    USA_FEATURES_V11
)

print(
    "Tiene explicabilidad:",
    USA_EXPLICABILIDAD
)

print(
    "Calcula términos activos:",
    USA_TERMINOS_ACTIVOS
)


PREDICTOR_REQUIERE_ADAPTACION = (
    USA_TFIDF_DIRECTO
    or not USA_FEATURES_V11
)


print(
    "\nPredictor requiere adaptación:",
    PREDICTOR_REQUIERE_ADAPTACION
)

DIAGNÓSTICO DE COMPATIBILIDAD
Acceso directo a tfidf: True
Soporte actual para features: False
Tiene explicabilidad: True
Calcula términos activos: True

Predictor requiere adaptación: True


In [75]:
# ==============================================================================
# CELDA 12.5 — AUDITORÍA DE LA API
# ==============================================================================

PATH_API_MAIN_V11 = (
    PACKAGE_V11_ROOT
    / "techmind_api"
    / "main.py"
)


PATH_API_SCHEMAS_V11 = (
    PACKAGE_V11_ROOT
    / "techmind_api"
    / "schemas.py"
)


for ruta in [
    PATH_API_MAIN_V11,
    PATH_API_SCHEMAS_V11,
]:

    if not ruta.exists():
        raise FileNotFoundError(
            ruta
        )


codigo_api = (
    PATH_API_MAIN_V11
    .read_text(
        encoding="utf-8"
    )
)


lineas_api = (
    codigo_api
    .splitlines()
)


PATRONES_API = [
    "TechMindPredictor",
    ".predict(",
    "predictor.",
    "health",
    "model-info",
    "include",
    "top_n",
    "top_k",
]


registros_api = []


for numero, linea in enumerate(
    lineas_api,
    start=1,
):

    if any(
        patron.lower()
        in linea.lower()
        for patron
        in PATRONES_API
    ):

        registros_api.append({
            "linea": numero,
            "codigo": linea,
        })


df_auditoria_api = pd.DataFrame(
    registros_api
)


print("=" * 100)
print("LLAMADAS DEL MICROSERVICIO")
print("=" * 100)

print(
    df_auditoria_api
    .to_string(
        index=False
    )
)

LLAMADAS DEL MICROSERVICIO
 linea                                    codigo
    30    from techmind import TechMindPredictor
    33                           HealthResponse,
    73            predictor = TechMindPredictor(
   113                   ) -> TechMindPredictor:
   142                      "health": "/health",
   143              "model_info": "/model-info",
   152                                "/health",
   153            response_model=HealthResponse,
   156                               def health(
   163                          model_health = (
   164                        predictor.health()
   171           "model_name": model_health.get(
   174         "model_status": model_health.get(
   177             "model_sha256": model_health[
   180                  "classes": model_health[
   183           "tfidf_features": model_health[
   186           "pipeline_steps": model_health[
   198                            "/model-info",
   209                          model_heal

In [76]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.6 — AUDITORÍA ESTRUCTURAL COMPLETA DE predictor.py
# ==============================================================================

from pathlib import Path
import re
import pandas as pd


PATH_PREDICTOR_V11 = (
    PACKAGE_V11_ROOT
    / "techmind"
    / "predictor.py"
)


if not PATH_PREDICTOR_V11.exists():
    raise FileNotFoundError(
        PATH_PREDICTOR_V11
    )


codigo_predictor = (
    PATH_PREDICTOR_V11
    .read_text(
        encoding="utf-8"
    )
)


lineas_predictor = (
    codigo_predictor
    .splitlines()
)


print("=" * 100)
print("PREDICTOR")
print("=" * 100)

print("Archivo:")
print(PATH_PREDICTOR_V11)

print(
    "\nTotal de líneas:",
    len(lineas_predictor)
)

PREDICTOR
Archivo:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind\predictor.py

Total de líneas: 1251


In [77]:
# ==============================================================================
# CLASES Y MÉTODOS
# ==============================================================================

PATRON_ESTRUCTURA = re.compile(
    r"^\s*(class\s+\w+|def\s+\w+|async\s+def\s+\w+)"
)


estructura = []


for numero, linea in enumerate(
    lineas_predictor,
    start=1,
):

    if PATRON_ESTRUCTURA.search(
        linea
    ):

        estructura.append({
            "linea": numero,
            "codigo": linea.strip(),
        })


df_estructura_predictor = (
    pd.DataFrame(
        estructura
    )
)


print("\n" + "=" * 100)
print("ESTRUCTURA DE predictor.py")
print("=" * 100)

print(
    df_estructura_predictor
    .to_string(
        index=False
    )
)


ESTRUCTURA DE predictor.py
 linea                                        codigo
    21               def _sha256(path: Path) -> str:
    37 def _load_json(path: Path) -> dict[str, Any]:
    48                      class TechMindPredictor:
    51                                 def __init__(
   307           def health(self) -> dict[str, Any]:
   335                          def _prepare_inputs(
   392                           def _validate_text(
   492                            def _margin_level(
   513                    def _operational_decision(
   628                          def _explain_vector(
   823                                  def predict(


In [78]:
# ==============================================================================
# REFERENCIAS SENSIBLES A LA ARQUITECTURA
# ==============================================================================

PATRONES_SENSIBLES = [
    "named_steps",
    '["tfidf"]',
    "['tfidf']",
    "tfidf",
    "vectorizer",
    "vectorizador",
    "get_feature_names_out",
    "transform(",
    "coef_",
    "decision_function",
    "terminos_activos",
    "features",
    "positive_terms",
    "negative_terms",
    "differential_terms",
    "few_terms",
    "margin",
    "margen",
    "health",
    "pipeline_steps",
]


registros_sensibles = []


for numero, linea in enumerate(
    lineas_predictor,
    start=1,
):

    if any(
        patron.lower()
        in linea.lower()
        for patron
        in PATRONES_SENSIBLES
    ):

        registros_sensibles.append({
            "linea": numero,
            "codigo": linea.rstrip(),
        })


df_referencias_predictor = (
    pd.DataFrame(
        registros_sensibles
    )
)


print("\n" + "=" * 100)
print("REFERENCIAS RELEVANTES — predictor.py")
print("=" * 100)

print(
    df_referencias_predictor
    .to_string(
        index=False
    )
)


REFERENCIAS RELEVANTES — predictor.py
 linea                                               codigo
   169                                        "named_steps"
   178                                             "tfidf",
   186                                         .named_steps
   198                                  self.vectorizer = (
   200                                .named_steps["tfidf"]
   205                         .named_steps["clasificador"]
   213                                      self.vectorizer
   214                             .get_feature_names_out()
   226                                              "coef_"
   234                      "umbrales_margen_descriptivos",
   238                             self.margin_p10 = float(
   245                             self.margin_p25 = float(
   248                                      self.margin_p10
   252                             self.margin_p50 = float(
   255                                      self.margin_p25
 

In [79]:
# ==============================================================================
# MOSTRAR BLOQUES DE CÓDIGO ALREDEDOR DE MÉTODOS IMPORTANTES
# ==============================================================================

def mostrar_contexto_metodo(
    nombre,
    lineas_antes=3,
    lineas_despues=80,
):

    patron = re.compile(
        rf"^\s*def\s+{re.escape(nombre)}\s*\("
    )

    for indice, linea in enumerate(
        lineas_predictor
    ):

        if patron.search(linea):

            inicio = max(
                0,
                indice - lineas_antes
            )

            fin = min(
                len(lineas_predictor),
                indice
                + lineas_despues
            )

            print("\n" + "=" * 100)
            print(
                f"MÉTODO: {nombre}"
            )
            print("=" * 100)

            for i in range(
                inicio,
                fin
            ):

                print(
                    f"{i + 1:04d}: "
                    f"{lineas_predictor[i]}"
                )

            return

    print(
        f"\nMétodo '{nombre}' no encontrado."
    )


for metodo in [
    "__init__",
    "health",
    "predict",
]:

    mostrar_contexto_metodo(
        metodo
    )


MÉTODO: __init__
0048: class TechMindPredictor:
0049:     """Clasificador independiente de contenido tecnológico."""
0050: 
0051:     def __init__(
0052:         self,
0053:         package_root: str | Path | None = None,
0054:         verify_hash: bool = True
0055:     ) -> None:
0056:         """
0057:         Carga el modelo y la configuración del paquete.
0058: 
0059:         Parameters
0060:         ----------
0061:         package_root:
0062:             Directorio raíz del paquete exportado.
0063: 
0064:         verify_hash:
0065:             Comprueba que la firma del modelo coincide
0066:             con la registrada en la configuración.
0067:         """
0068: 
0069:         if package_root is None:
0070: 
0071:             self.package_root = (
0072:                 Path(__file__)
0073:                 .resolve()
0074:                 .parents[1]
0075:             )
0076: 
0077:         else:
0078: 
0079:             self.package_root = Path(
0080:                 package_

In [80]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.7 — CONFIGURACIÓN RETROCOMPATIBLE
# ==============================================================================

import json


with open(
    PATH_CONFIG_PAQUETE_V11,
    "r",
    encoding="utf-8",
) as archivo:

    config_package_v11 = json.load(
        archivo
    )


# ==============================================================================
# VERSIONES
# ==============================================================================

config_package_v11[
    "interface_version"
] = "1.0.0"

config_package_v11[
    "model_version"
] = "1.1.0"

config_package_v11[
    "margin_is_probability"
] = False


# ==============================================================================
# LÍMITES — COMPATIBILIDAD API
# ==============================================================================

limits = config_package_v11.get(
    "limits",
    {}
)


limits.setdefault(
    "max_batch_size",
    500
)

limits.setdefault(
    "max_characters_per_document",
    50000
)


config_package_v11[
    "limits"
] = limits


# ==============================================================================
# INFORMACIÓN OPERACIONAL
# ==============================================================================

config_package_v11[
    "operational"
] = {
    "states": [
        "aceptada",
        "revision",
        "rechazada",
    ],

    "review_margin": float(
        UMBRAL_MARGEN_REVISION_V11
    ),

    "few_features_threshold": int(
        UMBRAL_FEATURES_POCAS_V11
    ),

    "reject_if_total_features": 0,

    "coverage_definition": (
        "word_features + char_features"
    ),
}


# ==============================================================================
# GUARDAR
# ==============================================================================

with open(
    PATH_CONFIG_PAQUETE_V11,
    "w",
    encoding="utf-8",
) as archivo:

    json.dump(
        config_package_v11,
        archivo,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 80)
print("CONFIGURACIÓN v1.1.0 ACTUALIZADA")
print("=" * 80)

print(
    "Interface version:",
    config_package_v11[
        "interface_version"
    ]
)

print(
    "Model version:",
    config_package_v11[
        "model_version"
    ]
)

print(
    "Max batch:",
    config_package_v11[
        "limits"
    ][
        "max_batch_size"
    ]
)

print(
    "Review margin:",
    config_package_v11[
        "operational"
    ][
        "review_margin"
    ]
)

print(
    "Few features:",
    config_package_v11[
        "operational"
    ][
        "few_features_threshold"
    ]
)

CONFIGURACIÓN v1.1.0 ACTUALIZADA
Interface version: 1.0.0
Model version: 1.1.0
Max batch: 500
Review margin: 0.64346894252355
Few features: 178


In [81]:
# ==============================================================================
# CELDA 12.8 — RESPALDO DE predictor.py
# ==============================================================================

import shutil
from pathlib import Path


PATH_PREDICTOR_BACKUP = (
    PATH_PREDICTOR_V11
    .with_name(
        "predictor_v1_0_backup.py"
    )
)


if not PATH_PREDICTOR_BACKUP.exists():

    shutil.copy2(
        PATH_PREDICTOR_V11,
        PATH_PREDICTOR_BACKUP,
    )


print("Predictor original:")
print(PATH_PREDICTOR_V11)

print("\nRespaldo:")
print(PATH_PREDICTOR_BACKUP)

print(
    "\nRespaldo existe:",
    PATH_PREDICTOR_BACKUP.exists()
)

Predictor original:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind\predictor.py

Respaldo:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind\predictor_v1_0_backup.py

Respaldo existe: True


In [82]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.9 — GENERAR TechMindPredictor v1.1.0
# ==============================================================================

from pathlib import Path
import textwrap


codigo_predictor_v11 = r'''
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
from uuid import uuid4

import hashlib
import json
import time

import joblib
import numpy as np


INTERFACE_VERSION = "1.0.0"


# ==============================================================================
# UTILIDADES
# ==============================================================================

def _sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """Calcula SHA-256 de un archivo."""

    digest = hashlib.sha256()

    with path.open("rb") as archivo:

        while True:

            bloque = archivo.read(
                chunk_size
            )

            if not bloque:
                break

            digest.update(
                bloque
            )

    return digest.hexdigest()


def _load_json(
    path: Path,
) -> dict[str, Any]:
    """Carga un JSON UTF-8."""

    with path.open(
        "r",
        encoding="utf-8",
    ) as archivo:

        return json.load(
            archivo
        )


# ==============================================================================
# PREDICTOR
# ==============================================================================

class TechMindPredictor:
    """Clasificador independiente de contenido tecnológico."""

    def __init__(
        self,
        package_root: str | Path | None = None,
        verify_hash: bool = True,
    ) -> None:

        # ----------------------------------------------------------------------
        # PACKAGE ROOT
        # ----------------------------------------------------------------------

        if package_root is None:

            self.package_root = (
                Path(__file__)
                .resolve()
                .parents[1]
            )

        else:

            self.package_root = Path(
                package_root
            ).resolve()


        self.artifacts_dir = (
            self.package_root
            / "artifacts"
        )


        self.model_path = (
            self.artifacts_dir
            / "techmind_modelo_final.joblib"
        )


        self.metadata_path = (
            self.artifacts_dir
            / "techmind_modelo_final_metadata.json"
        )


        self.config_path = (
            self.artifacts_dir
            / "configuracion_inferencia.json"
        )


        self.contract_path = (
            self.artifacts_dir
            / "contrato_inferencia.json"
        )


        required_files = [
            self.model_path,
            self.metadata_path,
            self.config_path,
            self.contract_path,
        ]


        missing_files = [
            str(path)
            for path in required_files
            if not path.exists()
        ]


        if missing_files:

            raise FileNotFoundError(
                "Faltan archivos del paquete:\n"
                + "\n".join(
                    f"- {path}"
                    for path
                    in missing_files
                )
            )


        # ----------------------------------------------------------------------
        # METADATA / CONFIG / CONTRACT
        # ----------------------------------------------------------------------

        self.metadata = _load_json(
            self.metadata_path
        )

        self.config = _load_json(
            self.config_path
        )

        self.contract = _load_json(
            self.contract_path
        )


        self.interface_version = str(
            self.config.get(
                "interface_version",
                INTERFACE_VERSION,
            )
        )


        self.model_version = str(
            self.metadata.get(
                "version",
                self.config.get(
                    "model_version",
                    "1.1.0",
                ),
            )
        )


        self.model_name = str(
            self.metadata.get(
                "model_name",
                self.metadata.get(
                    "modelo",
                    (
                        "TF-IDF Word + Char 3-6 "
                        "+ SGDClassifier optimizado"
                    ),
                ),
            )
        )


        self.model_status = str(
            self.metadata.get(
                "status",
                self.metadata.get(
                    "estado",
                    "ready",
                ),
            )
        )


        # ----------------------------------------------------------------------
        # HASH
        # ----------------------------------------------------------------------

        self.model_sha256 = _sha256(
            self.model_path
        )


        expected_sha256 = (
            self.metadata.get(
                "sha256"
            )
            or
            self.metadata.get(
                "model_sha256"
            )
            or
            self.metadata.get(
                "sha256_modelo"
            )
            or
            self.config.get(
                "model_sha256"
            )
        )


        if (
            verify_hash
            and expected_sha256
            and self.model_sha256
            != str(expected_sha256)
        ):

            raise RuntimeError(
                "La firma SHA-256 del modelo "
                "no coincide con la registrada."
            )


        # ----------------------------------------------------------------------
        # CARGAR MODELO
        # ----------------------------------------------------------------------

        self.model = joblib.load(
            self.model_path
        )


        if not hasattr(
            self.model,
            "named_steps",
        ):

            raise TypeError(
                "El artefacto cargado no es "
                "un Pipeline compatible."
            )


        pipeline_steps = list(
            self.model.named_steps.keys()
        )


        required_steps = {
            "features",
            "clasificador",
        }


        if not required_steps.issubset(
            set(pipeline_steps)
        ):

            raise RuntimeError(
                "Pipeline v1.1.0 incompatible. "
                "Se requieren los pasos "
                "'features' y 'clasificador'. "
                f"Encontrados: {pipeline_steps}"
            )


        # ----------------------------------------------------------------------
        # FEATURE UNION
        # ----------------------------------------------------------------------

        self.feature_union = (
            self.model
            .named_steps[
                "features"
            ]
        )


        transformadores = dict(
            self.feature_union
            .transformer_list
        )


        if (
            "word" not in transformadores
            or
            "char" not in transformadores
        ):

            raise RuntimeError(
                "FeatureUnion incompatible. "
                "Se requieren transformadores "
                "'word' y 'char'."
            )


        self.word_vectorizer = (
            transformadores[
                "word"
            ]
        )


        self.char_vectorizer = (
            transformadores[
                "char"
            ]
        )


        self.classifier = (
            self.model
            .named_steps[
                "clasificador"
            ]
        )


        # ----------------------------------------------------------------------
        # VOCABULARIOS
        # ----------------------------------------------------------------------

        self.word_feature_names = np.asarray(
            self.word_vectorizer
            .get_feature_names_out()
        )


        self.char_feature_names = np.asarray(
            self.char_vectorizer
            .get_feature_names_out()
        )


        self.word_feature_count = int(
            len(
                self.word_feature_names
            )
        )


        self.char_feature_count = int(
            len(
                self.char_feature_names
            )
        )


        self.total_feature_count = int(
            self.word_feature_count
            + self.char_feature_count
        )


        # Alias retrocompatible:
        # main.py todavía solicita tfidf_features.
        self.feature_names = np.concatenate([
            self.word_feature_names,
            self.char_feature_names,
        ])


        self.feature_types = np.asarray(
            (
                ["word"]
                * self.word_feature_count
            )
            +
            (
                ["char"]
                * self.char_feature_count
            )
        )


        # ----------------------------------------------------------------------
        # CLASES
        # ----------------------------------------------------------------------

        self.classes = np.asarray(
            self.classifier.classes_
        )


        if len(
            self.classes
        ) < 2:

            raise RuntimeError(
                "El modelo debe contener "
                "al menos dos clases."
            )


        # ----------------------------------------------------------------------
        # VALIDACIÓN DE COEFICIENTES
        # ----------------------------------------------------------------------

        if not hasattr(
            self.classifier,
            "coef_",
        ):

            raise RuntimeError(
                "El clasificador no expone coef_. "
                "La explicabilidad lineal no "
                "está disponible."
            )


        coef_shape = (
            self.classifier
            .coef_
            .shape
        )


        if (
            coef_shape[1]
            != self.total_feature_count
        ):

            raise RuntimeError(
                "El número de coeficientes "
                "no coincide con Word + Char. "
                f"Coeficientes={coef_shape[1]}, "
                f"features={self.total_feature_count}."
            )


        # ----------------------------------------------------------------------
        # MÁRGENES
        # ----------------------------------------------------------------------

        margin_thresholds = (
            self.config.get(
                "margin_thresholds",
                {},
            )
        )


        operational_config = (
            self.config.get(
                "operational",
                {},
            )
        )


        self.review_margin = float(
            margin_thresholds.get(
                "review",
                operational_config.get(
                    "review_margin",
                    0.0,
                ),
            )
        )


        self.margin_p10 = float(
            margin_thresholds.get(
                "p10",
                self.review_margin,
            )
        )


        self.margin_p25 = float(
            margin_thresholds.get(
                "p25",
                self.margin_p10,
            )
        )


        self.margin_p50 = float(
            margin_thresholds.get(
                "p50",
                self.margin_p25,
            )
        )


        self.margin_p90 = float(
            margin_thresholds.get(
                "p90",
                self.margin_p50,
            )
        )


        # ----------------------------------------------------------------------
        # COBERTURA
        # ----------------------------------------------------------------------

        coverage = self.config.get(
            "coverage",
            {}
        )


        self.few_terms_threshold = int(
            coverage.get(
                "few_features_threshold",
                operational_config.get(
                    "few_features_threshold",
                    1,
                ),
            )
        )


        self.reject_if_total_features = int(
            coverage.get(
                "reject_if_total_features",
                operational_config.get(
                    "reject_if_total_features",
                    0,
                ),
            )
        )


        # ----------------------------------------------------------------------
        # LÍMITES
        # ----------------------------------------------------------------------

        limits = self.config.get(
            "limits",
            {}
        )


        self.max_batch_size = int(
            limits.get(
                "max_batch_size",
                500,
            )
        )


        self.max_characters_per_document = int(
            limits.get(
                "max_characters_per_document",
                limits.get(
                    "max_characters",
                    50000,
                ),
            )
        )


    # ==========================================================================
    # HEALTH
    # ==========================================================================

    def health(
        self,
    ) -> dict[str, Any]:
        """Devuelve información técnica del modelo."""

        return {
            "status": "ok",

            "interface_version": (
                self.interface_version
            ),

            "model_version": (
                self.model_version
            ),

            "model_name": (
                self.model_name
            ),

            "model_status": (
                self.model_status
            ),

            "classes": [
                str(value)
                for value
                in self.classes
            ],

            # --------------------------------------------------------------
            # Alias requerido por FastAPI v1.0.
            # Ahora representa Word + Char.
            # --------------------------------------------------------------

            "tfidf_features": int(
                self.total_feature_count
            ),

            # --------------------------------------------------------------
            # Información ampliada v1.1.
            # --------------------------------------------------------------

            "word_features": int(
                self.word_feature_count
            ),

            "char_features": int(
                self.char_feature_count
            ),

            "total_features": int(
                self.total_feature_count
            ),

            "model_sha256": (
                self.model_sha256
            ),

            "pipeline_steps": list(
                self.model
                .named_steps
                .keys()
            ),

            "architecture": (
                "TF-IDF Word + "
                "TF-IDF Char 3-6 + "
                "SGDClassifier"
            ),

            "margin_is_probability": False,
        }


    # ==========================================================================
    # INPUTS
    # ==========================================================================

    def _prepare_inputs(
        self,
        texts: str | Iterable[str],
    ) -> list[Any]:
        """Normaliza el contenedor de entrada."""

        if isinstance(
            texts,
            str,
        ):

            documents = [
                texts
            ]


        elif isinstance(
            texts,
            np.ndarray,
        ):

            if texts.ndim != 1:

                raise ValueError(
                    "El arreglo NumPy debe "
                    "ser unidimensional."
                )

            documents = (
                texts.tolist()
            )


        elif isinstance(
            texts,
            (
                list,
                tuple,
            ),
        ):

            documents = list(
                texts
            )


        else:

            raise TypeError(
                "La entrada debe ser un texto, "
                "lista, tupla o arreglo "
                "unidimensional."
            )


        if not documents:

            raise ValueError(
                "La colección de textos "
                "está vacía."
            )


        if (
            len(documents)
            > self.max_batch_size
        ):

            raise ValueError(
                "El lote excede el máximo de "
                f"{self.max_batch_size} documentos."
            )


        return documents


    # ==========================================================================
    # VALIDACIÓN
    # ==========================================================================

    def _validate_text(
        self,
        value: Any,
    ) -> dict[str, Any]:
        """Valida un documento individual."""

        input_type = type(
            value
        ).__name__


        if not isinstance(
            value,
            str,
        ):

            return {
                "valid": False,
                "text": None,
                "input_type": input_type,
                "message": (
                    "El documento debe ser texto."
                ),
            }


        text = value.strip()


        if not text:

            return {
                "valid": False,
                "text": text,
                "input_type": input_type,
                "message": (
                    "El texto está vacío."
                ),
            }


        if (
            len(text)
            > self.max_characters_per_document
        ):

            return {
                "valid": False,
                "text": text,
                "input_type": input_type,
                "message": (
                    "El documento excede el "
                    "máximo de caracteres permitido."
                ),
            }


        return {
            "valid": True,
            "text": text,
            "input_type": input_type,
            "message": (
                "Inferencia completada."
            ),
        }


    # ==========================================================================
    # MARGEN
    # ==========================================================================

    def _margin_level(
        self,
        margin: float,
    ) -> str:
        """Asigna un nivel descriptivo al margen."""

        if margin <= self.margin_p10:
            return "Crítica"

        if margin <= self.margin_p25:
            return "Baja"

        if margin <= self.margin_p50:
            return "Media"

        if margin <= self.margin_p90:
            return "Alta"

        return "Muy alta"


    # ==========================================================================
    # DECISIÓN OPERACIONAL
    # ==========================================================================

    def _operational_decision(
        self,
        margin: float,
        total_features: int,
        word_features: int,
        char_features: int,
    ) -> dict[str, Any]:
        """Aplica la política calibrada de v1.1.0."""

        warnings: list[str] = []


        # ------------------------------------------------------------------
        # SIN COBERTURA
        # ------------------------------------------------------------------

        if (
            total_features
            <= self.reject_if_total_features
        ):

            return {
                "estado": "rechazada",
                "accion": (
                    "No utilizar la predicción"
                ),
                "requiere_revision": False,
                "prediccion_utilizable": False,
                "advertencias": [
                    (
                        "Sin cobertura de "
                        "características."
                    )
                ],
            }


        # ------------------------------------------------------------------
        # INFORMACIÓN DE COBERTURA
        # ------------------------------------------------------------------

        if word_features == 0:

            warnings.append(
                "La cobertura depende únicamente "
                "de n-gramas de caracteres."
            )


        if (
            total_features
            <= self.few_terms_threshold
        ):

            warnings.append(
                "Cobertura reducida de "
                "características."
            )


        # ------------------------------------------------------------------
        # MARGEN
        # ------------------------------------------------------------------

        if (
            margin
            < self.review_margin
        ):

            warnings.append(
                "El margen de decisión "
                "es reducido."
            )


        # ------------------------------------------------------------------
        # REVISIÓN
        # ------------------------------------------------------------------

        if (
            margin
            < self.review_margin
            or
            total_features
            <= self.few_terms_threshold
        ):

            return {
                "estado": "revision",
                "accion": (
                    "Revisión humana recomendada"
                ),
                "requiere_revision": True,
                "prediccion_utilizable": True,
                "advertencias": warnings,
            }


        # ------------------------------------------------------------------
        # ACEPTADA
        # ------------------------------------------------------------------

        return {
            "estado": "aceptada",
            "accion": (
                "Predicción utilizable "
                "automáticamente"
            ),
            "requiere_revision": False,
            "prediccion_utilizable": True,
            "advertencias": warnings,
        }


    # ==========================================================================
    # SCORES
    # ==========================================================================

    def _normalize_scores(
        self,
        scores: np.ndarray,
    ) -> np.ndarray:
        """Normaliza decision_function a matriz 2D."""

        scores = np.asarray(
            scores
        )


        if scores.ndim == 1:

            if len(
                self.classes
            ) == 2:

                scores = np.column_stack([
                    -scores,
                    scores,
                ])

            else:

                scores = scores.reshape(
                    1,
                    -1,
                )


        return scores


    # ==========================================================================
    # RANKING
    # ==========================================================================

    def _ranking(
        self,
        scores: np.ndarray,
        top_k: int,
    ) -> list[dict[str, Any]]:
        """Genera ranking de categorías."""

        orden = np.argsort(
            scores
        )[::-1][
            :top_k
        ]


        return [
            {
                "position": int(
                    posicion
                ),

                "category": str(
                    self.classes[
                        indice
                    ]
                ),

                "score": float(
                    scores[
                        indice
                    ]
                ),
            }

            for posicion, indice
            in enumerate(
                orden,
                start=1,
            )
        ]


    # ==========================================================================
    # EXPLICABILIDAD
    # ==========================================================================

    def _explain_vector(
        self,
        vector,
        winner_index: int,
        second_index: int,
        top_n: int,
    ) -> dict[str, Any]:
        """
        Explica una predicción lineal sobre
        FeatureUnion Word + Char.
        """

        vector = vector.getrow(
            0
        )


        indices = (
            vector.indices
        )

        values = (
            vector.data
        )


        if len(indices) == 0:

            return {
                "positive_terms": [],
                "negative_terms": [],
                "differential_terms": [],
                "warning": (
                    "No existen características "
                    "activas para explicar."
                ),
            }


        winner_coef = (
            self.classifier
            .coef_[
                winner_index,
                indices,
            ]
        )


        second_coef = (
            self.classifier
            .coef_[
                second_index,
                indices,
            ]
        )


        contributions = (
            values
            * winner_coef
        )


        differential = (
            values
            * (
                winner_coef
                - second_coef
            )
        )


        records = []


        for local_index, feature_index in enumerate(
            indices
        ):

            records.append({
                "feature_index": int(
                    feature_index
                ),

                "term": str(
                    self.feature_names[
                        feature_index
                    ]
                ),

                "feature_type": str(
                    self.feature_types[
                        feature_index
                    ]
                ),

                "tfidf": float(
                    values[
                        local_index
                    ]
                ),

                "coefficient": float(
                    winner_coef[
                        local_index
                    ]
                ),

                "contribution": float(
                    contributions[
                        local_index
                    ]
                ),

                "differential": float(
                    differential[
                        local_index
                    ]
                ),
            })


        # ------------------------------------------------------------------
        # POSITIVAS
        # ------------------------------------------------------------------

        positive = sorted(
            (
                item
                for item
                in records
                if item[
                    "contribution"
                ] > 0
            ),
            key=lambda item: (
                item[
                    "contribution"
                ]
            ),
            reverse=True,
        )[:top_n]


        positive_terms = [
            {
                "term": item["term"],
                "feature_type": (
                    item[
                        "feature_type"
                    ]
                ),
                "tfidf": item["tfidf"],
                "coefficient": (
                    item[
                        "coefficient"
                    ]
                ),
                "contribution": (
                    item[
                        "contribution"
                    ]
                ),
            }

            for item
            in positive
        ]


        # ------------------------------------------------------------------
        # NEGATIVAS
        # ------------------------------------------------------------------

        negative = sorted(
            (
                item
                for item
                in records
                if item[
                    "contribution"
                ] < 0
            ),
            key=lambda item: (
                item[
                    "contribution"
                ]
            ),
        )[:top_n]


        negative_terms = [
            {
                "term": item["term"],
                "feature_type": (
                    item[
                        "feature_type"
                    ]
                ),
                "tfidf": item["tfidf"],
                "coefficient": (
                    item[
                        "coefficient"
                    ]
                ),
                "contribution": (
                    item[
                        "contribution"
                    ]
                ),
            }

            for item
            in negative
        ]


        # ------------------------------------------------------------------
        # DIFERENCIALES
        # ------------------------------------------------------------------

        differential_sorted = sorted(
            records,
            key=lambda item: abs(
                item[
                    "differential"
                ]
            ),
            reverse=True,
        )[:top_n]


        winner_class = str(
            self.classes[
                winner_index
            ]
        )


        second_class = str(
            self.classes[
                second_index
            ]
        )


        differential_terms = [
            {
                "term": item["term"],

                "feature_type": (
                    item[
                        "feature_type"
                    ]
                ),

                "contribution": (
                    item[
                        "differential"
                    ]
                ),

                "favours": (
                    winner_class
                    if item[
                        "differential"
                    ] >= 0
                    else second_class
                ),
            }

            for item
            in differential_sorted
        ]


        return {
            "positive_terms": (
                positive_terms
            ),

            "negative_terms": (
                negative_terms
            ),

            "differential_terms": (
                differential_terms
            ),

            "warning": (
                "Las contribuciones describen "
                "el comportamiento matemático "
                "del modelo, no causalidad."
            ),
        }


    # ==========================================================================
    # PREDICT
    # ==========================================================================

    def predict(
        self,
        texts: str | Iterable[str],
        include_explanation: bool = False,
        explanation_top_n: int = 8,
        top_k: int | None = None,
    ) -> dict[str, Any]:
        """Clasifica uno o varios documentos."""

        start_time = (
            time.perf_counter()
        )


        documents = (
            self._prepare_inputs(
                texts
            )
        )


        if not isinstance(
            include_explanation,
            bool,
        ):

            raise TypeError(
                "include_explanation "
                "debe ser booleano."
            )


        explanation_top_n = int(
            explanation_top_n
        )


        if explanation_top_n <= 0:

            raise ValueError(
                "explanation_top_n debe "
                "ser mayor que cero."
            )


        if top_k is None:

            top_k = len(
                self.classes
            )


        top_k = int(
            top_k
        )


        if not (
            1
            <= top_k
            <= len(
                self.classes
            )
        ):

            raise ValueError(
                "top_k está fuera del "
                "rango permitido."
            )


        request_id = str(
            uuid4()
        )


        timestamp = datetime.now(
            timezone.utc
        ).isoformat(
            timespec="milliseconds"
        )


        results: list[
            dict[str, Any]
        ] = []


        valid_positions: list[
            int
        ] = []


        valid_texts: list[
            str
        ] = []


        # ------------------------------------------------------------------
        # VALIDACIÓN
        # ------------------------------------------------------------------

        for index, value in enumerate(
            documents
        ):

            validation = (
                self._validate_text(
                    value
                )
            )


            text_for_result = (
                validation[
                    "text"
                ]
            )


            result = {
                "request_id": request_id,
                "record_id": int(
                    index
                ),
                "timestamp_utc": timestamp,

                "input_type": (
                    validation[
                        "input_type"
                    ]
                ),

                "text": (
                    text_for_result
                ),

                "characters": (
                    len(
                        text_for_result
                    )
                    if isinstance(
                        text_for_result,
                        str,
                    )
                    else 0
                ),

                "words": (
                    len(
                        text_for_result
                        .split()
                    )
                    if isinstance(
                        text_for_result,
                        str,
                    )
                    else 0
                ),

                "valid_input": bool(
                    validation[
                        "valid"
                    ]
                ),

                "validation_message": (
                    validation[
                        "message"
                    ]
                ),

                "estado": "rechazada",

                "categoria_predicha": None,

                "segunda_categoria": None,

                "puntuacion_ganadora": None,

                "puntuacion_segunda": None,

                "margen_decision": None,

                "nivel_margen": None,

                # ------------------------------------------------------
                # COMPATIBILIDAD v1.0
                # ------------------------------------------------------

                "terminos_activos": 0,

                # ------------------------------------------------------
                # NUEVO v1.1
                # ------------------------------------------------------

                "word_features_activas": 0,

                "char_features_activas": 0,

                "features_activas_total": 0,

                "accion_recomendada": (
                    "No utilizar la predicción"
                ),

                "requiere_revision": False,

                "prediccion_utilizable": False,

                "advertencias": [],

                "ranking_categorias": [],

                "explicacion": (
                    None
                    if not include_explanation
                    else {
                        "positive_terms": [],
                        "negative_terms": [],
                        "differential_terms": [],
                        "warning": (
                            "No existe una predicción "
                            "utilizable para explicar."
                        ),
                    }
                ),
            }


            results.append(
                result
            )


            if validation[
                "valid"
            ]:

                valid_positions.append(
                    index
                )

                valid_texts.append(
                    validation[
                        "text"
                    ]
                )


        # ------------------------------------------------------------------
        # INFERENCIA
        # ------------------------------------------------------------------

        if valid_texts:

            combined_matrix = (
                self.feature_union
                .transform(
                    valid_texts
                )
            )


            word_matrix = (
                self.word_vectorizer
                .transform(
                    valid_texts
                )
            )


            char_matrix = (
                self.char_vectorizer
                .transform(
                    valid_texts
                )
            )


            scores_matrix = (
                self._normalize_scores(
                    self.model
                    .decision_function(
                        valid_texts
                    )
                )
            )


            for local_position, result_position in enumerate(
                valid_positions
            ):

                combined_vector = (
                    combined_matrix[
                        local_position
                    ]
                )


                word_features = int(
                    word_matrix[
                        local_position
                    ].getnnz()
                )


                char_features = int(
                    char_matrix[
                        local_position
                    ].getnnz()
                )


                total_features = int(
                    combined_vector.getnnz()
                )


                # ------------------------------------------------------
                # SIN COBERTURA
                # ------------------------------------------------------

                if (
                    total_features
                    <= self.reject_if_total_features
                ):

                    results[
                        result_position
                    ].update({
                        "estado": (
                            "rechazada"
                        ),

                        "validation_message": (
                            "Texto válido, pero "
                            "sin cobertura de "
                            "características."
                        ),

                        "word_features_activas": (
                            word_features
                        ),

                        "char_features_activas": (
                            char_features
                        ),

                        "features_activas_total": (
                            total_features
                        ),

                        "terminos_activos": (
                            total_features
                        ),

                        "accion_recomendada": (
                            "No utilizar "
                            "la predicción"
                        ),

                        "requiere_revision": False,

                        "prediccion_utilizable": False,

                        "advertencias": [
                            (
                                "Sin cobertura de "
                                "características."
                            )
                        ],
                    })

                    continue


                # ------------------------------------------------------
                # SCORES
                # ------------------------------------------------------

                scores = (
                    scores_matrix[
                        local_position
                    ]
                )


                order = np.argsort(
                    scores
                )[::-1]


                winner_index = int(
                    order[
                        0
                    ]
                )


                second_index = int(
                    order[
                        1
                    ]
                )


                category = str(
                    self.classes[
                        winner_index
                    ]
                )


                second_category = str(
                    self.classes[
                        second_index
                    ]
                )


                winner_score = float(
                    scores[
                        winner_index
                    ]
                )


                second_score = float(
                    scores[
                        second_index
                    ]
                )


                margin = float(
                    winner_score
                    - second_score
                )


                level = (
                    self._margin_level(
                        margin
                    )
                )


                decision = (
                    self._operational_decision(
                        margin=margin,
                        total_features=(
                            total_features
                        ),
                        word_features=(
                            word_features
                        ),
                        char_features=(
                            char_features
                        ),
                    )
                )


                ranking = (
                    self._ranking(
                        scores,
                        top_k=top_k,
                    )
                )


                explanation = None


                if include_explanation:

                    explanation = (
                        self._explain_vector(
                            vector=(
                                combined_vector
                            ),
                            winner_index=(
                                winner_index
                            ),
                            second_index=(
                                second_index
                            ),
                            top_n=(
                                explanation_top_n
                            ),
                        )
                    )


                results[
                    result_position
                ].update({
                    "validation_message": (
                        "Inferencia completada."
                    ),

                    "estado": (
                        decision[
                            "estado"
                        ]
                    ),

                    "categoria_predicha": (
                        category
                    ),

                    "segunda_categoria": (
                        second_category
                    ),

                    "puntuacion_ganadora": (
                        winner_score
                    ),

                    "puntuacion_segunda": (
                        second_score
                    ),

                    "margen_decision": (
                        margin
                    ),

                    "nivel_margen": (
                        level
                    ),

                    # --------------------------------------------------
                    # COMPATIBILIDAD:
                    # ahora significa features Word + Char activas.
                    # --------------------------------------------------

                    "terminos_activos": (
                        total_features
                    ),

                    # --------------------------------------------------
                    # v1.1
                    # --------------------------------------------------

                    "word_features_activas": (
                        word_features
                    ),

                    "char_features_activas": (
                        char_features
                    ),

                    "features_activas_total": (
                        total_features
                    ),

                    "accion_recomendada": (
                        decision[
                            "accion"
                        ]
                    ),

                    "requiere_revision": (
                        decision[
                            "requiere_revision"
                        ]
                    ),

                    "prediccion_utilizable": (
                        decision[
                            "prediccion_utilizable"
                        ]
                    ),

                    "advertencias": (
                        decision[
                            "advertencias"
                        ]
                    ),

                    "ranking_categorias": (
                        ranking
                    ),

                    "explicacion": (
                        explanation
                    ),
                })


        # ------------------------------------------------------------------
        # RESUMEN
        # ------------------------------------------------------------------

        duration_seconds = float(
            time.perf_counter()
            - start_time
        )


        accepted = sum(
            result[
                "estado"
            ] == "aceptada"
            for result
            in results
        )


        review = sum(
            result[
                "estado"
            ] == "revision"
            for result
            in results
        )


        rejected = sum(
            result[
                "estado"
            ] == "rechazada"
            for result
            in results
        )


        document_count = len(
            results
        )


        milliseconds_per_document = (
            duration_seconds
            * 1000
            / document_count
            if document_count
            else 0.0
        )


        return {
            "resumen": {
                "request_id": request_id,

                "timestamp_utc": timestamp,

                "interface_version": (
                    self.interface_version
                ),

                "model_version": (
                    self.model_version
                ),

                "model_name": (
                    self.model_name
                ),

                "documents_received": (
                    document_count
                ),

                "documents_accepted": int(
                    accepted
                ),

                "documents_review": int(
                    review
                ),

                "documents_rejected": int(
                    rejected
                ),

                "duration_seconds": (
                    duration_seconds
                ),

                "milliseconds_per_document": (
                    milliseconds_per_document
                ),

                "explanations_included": (
                    include_explanation
                ),

                "margin_is_probability": False,
            },

            "resultados": results,
        }
'''


codigo_predictor_v11 = (
    textwrap.dedent(
        codigo_predictor_v11
    )
    .lstrip()
)


PATH_PREDICTOR_V11.write_text(
    codigo_predictor_v11,
    encoding="utf-8",
)


print("=" * 80)
print("TechMindPredictor v1.1.0 GENERADO")
print("=" * 80)

print(
    "Archivo:"
)

print(
    PATH_PREDICTOR_V11
)

print(
    "\nLíneas:",
    len(
        codigo_predictor_v11
        .splitlines()
    )
)

print(
    "\nRespaldo v1.0:"
)

print(
    PATH_PREDICTOR_BACKUP
)

TechMindPredictor v1.1.0 GENERADO
Archivo:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind\predictor.py

Líneas: 2041

Respaldo v1.0:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind\predictor_v1_0_backup.py


In [83]:
# ==============================================================================
# CELDA 12.10 — VALIDACIÓN SINTÁCTICA
# ==============================================================================

import py_compile


py_compile.compile(
    str(
        PATH_PREDICTOR_V11
    ),
    doraise=True,
)


print(
    "Sintaxis predictor.py: OK"
)

Sintaxis predictor.py: OK


In [84]:
# ==============================================================================
# CELDA 12.11 — CARGA AISLADA DEL PREDICTOR v1.1
# ==============================================================================

import importlib.util
import sys


module_name_v11 = (
    "techmind_predictor_v11_test"
)


spec_v11 = (
    importlib.util
    .spec_from_file_location(
        module_name_v11,
        PATH_PREDICTOR_V11,
    )
)


if (
    spec_v11 is None
    or spec_v11.loader is None
):

    raise RuntimeError(
        "No fue posible crear "
        "el módulo de prueba."
    )


module_v11 = (
    importlib.util
    .module_from_spec(
        spec_v11
    )
)


sys.modules[
    module_name_v11
] = module_v11


spec_v11.loader.exec_module(
    module_v11
)


TechMindPredictorV11 = (
    module_v11
    .TechMindPredictor
)


predictor_v11 = (
    TechMindPredictorV11(
        package_root=(
            PACKAGE_V11_ROOT
        ),
        verify_hash=True,
    )
)


print(
    "Predictor cargado correctamente."
)

Predictor cargado correctamente.


In [85]:
# ==============================================================================
# CELDA 12.12 — HEALTH CHECK
# ==============================================================================

health_v11 = (
    predictor_v11.health()
)


print("=" * 80)
print("HEALTH — TECHMIND v1.1.0")
print("=" * 80)


for clave, valor in (
    health_v11.items()
):

    print(
        f"{clave}: {valor}"
    )

HEALTH — TECHMIND v1.1.0
status: ok
interface_version: 1.0.0
model_version: 1.1.0
model_name: TF-IDF Word + Char 3-6 + SGDClassifier optimizado
model_status: Modelo final evaluado y aprobado
classes: ['backend', 'cloud', 'datascience', 'frontend']
tfidf_features: 60000
word_features: 30000
char_features: 30000
total_features: 60000
model_sha256: 756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6
pipeline_steps: ['features', 'clasificador']
architecture: TF-IDF Word + TF-IDF Char 3-6 + SGDClassifier
margin_is_probability: False


In [86]:
assert (
    health_v11[
        "interface_version"
    ]
    == "1.0.0"
)

assert (
    health_v11[
        "model_version"
    ]
    == "1.1.0"
)

assert (
    health_v11[
        "pipeline_steps"
    ]
    == [
        "features",
        "clasificador",
    ]
)

assert (
    health_v11[
        "tfidf_features"
    ]
    ==
    health_v11[
        "total_features"
    ]
)

assert (
    health_v11[
        "word_features"
    ]
    +
    health_v11[
        "char_features"
    ]
    ==
    health_v11[
        "total_features"
    ]
)


print(
    "\nHealth compatible: True"
)


Health compatible: True


In [87]:
# ==============================================================================
# CELDA 12.13 — REGRESIÓN BACKEND EN PAQUETE DE PRODUCCIÓN
# ==============================================================================

CASOS_BACKEND_PRODUCCION = [
    (
        "Spring Boot es un framework de Java para desarrollar "
        "aplicaciones backend. Permite crear servicios REST, "
        "controladores, inyección de dependencias, seguridad con "
        "Spring Security, acceso a bases de datos mediante JPA y "
        "Hibernate, utilizando Maven como gestor de dependencias."
    ),

    (
        "Este contenido explica cómo crear una API REST con "
        "Spring Boot y Java, incluyendo el uso de controladores, "
        "servicios y repositorios."
    ),
]


respuesta_backend_v11 = (
    predictor_v11.predict(
        CASOS_BACKEND_PRODUCCION,
        include_explanation=True,
        explanation_top_n=8,
        top_k=4,
    )
)


for resultado in (
    respuesta_backend_v11[
        "resultados"
    ]
):

    print("\n" + "=" * 80)

    print(
        "Record:",
        resultado[
            "record_id"
        ]
    )

    print(
        "Estado:",
        resultado[
            "estado"
        ]
    )

    print(
        "Categoría:",
        resultado[
            "categoria_predicha"
        ]
    )

    print(
        "Segunda:",
        resultado[
            "segunda_categoria"
        ]
    )

    print(
        "Margen:",
        round(
            resultado[
                "margen_decision"
            ],
            4,
        )
    )

    print(
        "Word features:",
        resultado[
            "word_features_activas"
        ]
    )

    print(
        "Char features:",
        resultado[
            "char_features_activas"
        ]
    )

    print(
        "Total features:",
        resultado[
            "features_activas_total"
        ]
    )

    print(
        "Predicción utilizable:",
        resultado[
            "prediccion_utilizable"
        ]
    )


Record: 0
Estado: aceptada
Categoría: backend
Segunda: cloud
Margen: 1.8771
Word features: 5
Char features: 437
Total features: 442
Predicción utilizable: True

Record: 1
Estado: aceptada
Categoría: backend
Segunda: cloud
Margen: 0.9828
Word features: 0
Char features: 215
Total features: 215
Predicción utilizable: True


In [88]:
# ==============================================================================
# CELDA 12.14 — EXPLICABILIDAD WORD + CHAR
# ==============================================================================

for resultado in (
    respuesta_backend_v11[
        "resultados"
    ]
):

    print("\n" + "=" * 80)

    print(
        "CASO:",
        resultado[
            "record_id"
        ]
    )

    print("=" * 80)


    explicacion = (
        resultado[
            "explicacion"
        ]
    )


    print(
        "\nPOSITIVE TERMS"
    )

    for termino in (
        explicacion[
            "positive_terms"
        ]
    ):

        print(
            termino[
                "feature_type"
            ],
            "|",
            repr(
                termino[
                    "term"
                ]
            ),
            "| contrib:",
            round(
                termino[
                    "contribution"
                ],
                6,
            ),
        )


    print(
        "\nDIFFERENTIAL TERMS"
    )

    for termino in (
        explicacion[
            "differential_terms"
        ]
    ):

        print(
            termino[
                "feature_type"
            ],
            "|",
            repr(
                termino[
                    "term"
                ]
            ),
            "| contrib:",
            round(
                termino[
                    "contribution"
                ],
                6,
            ),
            "| favorece:",
            termino[
                "favours"
            ],
        )


CASO: 0

POSITIVE TERMS
word | 'backend' | contrib: 0.388609
word | 'framework' | contrib: 0.091901
char | ' ba' | contrib: 0.038447
char | 'pring' | contrib: 0.037787
char | 'spring' | contrib: 0.037787
char | 'pring ' | contrib: 0.037536
char | 'sprin' | contrib: 0.029075
char | ' spri' | contrib: 0.026748

DIFFERENTIAL TERMS
word | 'backend' | contrib: 0.527504 | favorece: backend
word | 'seguridad' | contrib: -0.140602 | favorece: cloud
word | 'framework' | contrib: 0.080497 | favorece: backend
char | ' ba' | contrib: 0.056969 | favorece: backend
char | 'pring' | contrib: 0.054533 | favorece: backend
char | 'spring' | contrib: 0.054533 | favorece: backend
char | 'pring ' | contrib: 0.053288 | favorece: backend
word | 'datos' | contrib: -0.046864 | favorece: cloud

CASO: 1

POSITIVE TERMS
char | ' con' | contrib: 0.040504
char | 'pring' | contrib: 0.036007
char | 'spring' | contrib: 0.036007
char | 'pring ' | contrib: 0.035768
char | 'api' | contrib: 0.034352
char | 'con' | contrib

In [89]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.15 — REGRESIÓN DE CASOS BACKEND
# ==============================================================================

CASOS_BACKEND_PRODUCCION = [
    (
        "Spring Boot es un framework de Java para desarrollar "
        "aplicaciones backend. Permite crear servicios REST, "
        "controladores, inyección de dependencias, seguridad con "
        "Spring Security, acceso a bases de datos mediante JPA y "
        "Hibernate, utilizando Maven como gestor de dependencias."
    ),

    (
        "Este contenido explica cómo crear una API REST con "
        "Spring Boot y Java, incluyendo el uso de controladores, "
        "servicios y repositorios."
    ),
]


respuesta_backend_v11 = (
    predictor_v11.predict(
        CASOS_BACKEND_PRODUCCION,
        include_explanation=True,
        explanation_top_n=8,
        top_k=4,
    )
)


print("=" * 80)
print("REGRESIÓN BACKEND — PAQUETE v1.1.0")
print("=" * 80)


for resultado in respuesta_backend_v11["resultados"]:

    print("\n" + "-" * 80)

    print(
        "Caso:",
        resultado["record_id"]
    )

    print(
        "Estado:",
        resultado["estado"]
    )

    print(
        "Categoría:",
        resultado["categoria_predicha"]
    )

    print(
        "Segunda:",
        resultado["segunda_categoria"]
    )

    print(
        "Margen:",
        round(
            resultado["margen_decision"],
            4,
        )
    )

    print(
        "Word features:",
        resultado[
            "word_features_activas"
        ]
    )

    print(
        "Char features:",
        resultado[
            "char_features_activas"
        ]
    )

    print(
        "Total features:",
        resultado[
            "features_activas_total"
        ]
    )

    print(
        "Utilizable:",
        resultado[
            "prediccion_utilizable"
        ]
    )

    print(
        "Advertencias:",
        resultado[
            "advertencias"
        ]
    )

REGRESIÓN BACKEND — PAQUETE v1.1.0

--------------------------------------------------------------------------------
Caso: 0
Estado: aceptada
Categoría: backend
Segunda: cloud
Margen: 1.8771
Word features: 5
Char features: 437
Total features: 442
Utilizable: True
Advertencias: []

--------------------------------------------------------------------------------
Caso: 1
Estado: aceptada
Categoría: backend
Segunda: cloud
Margen: 0.9828
Word features: 0
Char features: 215
Total features: 215
Utilizable: True
Advertencias: ['La cobertura depende únicamente de n-gramas de caracteres.']


In [90]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.16 — APROBACIÓN DE REGRESIÓN BACKEND
# ==============================================================================

resultados_backend = (
    respuesta_backend_v11[
        "resultados"
    ]
)


REGRESION_BACKEND_CATEGORIA_OK = all(
    resultado[
        "categoria_predicha"
    ] == "backend"

    for resultado
    in resultados_backend
)


REGRESION_BACKEND_COBERTURA_OK = all(
    resultado[
        "features_activas_total"
    ] > 0

    for resultado
    in resultados_backend
)


REGRESION_BACKEND_UTILIZABLE_OK = all(
    resultado[
        "prediccion_utilizable"
    ]

    for resultado
    in resultados_backend
)


REGRESION_BACKEND_ESTADO_OK = all(
    resultado[
        "estado"
    ] in {
        "aceptada",
        "revision",
    }

    for resultado
    in resultados_backend
)


# Caso especial que queremos garantizar:
# Word = 0 pero Char > 0 debe seguir siendo utilizable.

CASO_CHAR_ONLY_OK = any(
    resultado[
        "word_features_activas"
    ] == 0

    and

    resultado[
        "char_features_activas"
    ] > 0

    and

    resultado[
        "prediccion_utilizable"
    ]

    for resultado
    in resultados_backend
)


REGRESION_BACKEND_OK = all([
    REGRESION_BACKEND_CATEGORIA_OK,
    REGRESION_BACKEND_COBERTURA_OK,
    REGRESION_BACKEND_UTILIZABLE_OK,
    REGRESION_BACKEND_ESTADO_OK,
    CASO_CHAR_ONLY_OK,
])


print("=" * 80)
print("VALIDACIÓN DE REGRESIÓN — TECHMIND v1.1.0")
print("=" * 80)

print(
    "Ambos predicen backend:",
    REGRESION_BACKEND_CATEGORIA_OK
)

print(
    "Ambos tienen cobertura:",
    REGRESION_BACKEND_COBERTURA_OK
)

print(
    "Ambos son utilizables:",
    REGRESION_BACKEND_UTILIZABLE_OK
)

print(
    "Estados operativos válidos:",
    REGRESION_BACKEND_ESTADO_OK
)

print(
    "Caso Word=0 / Char>0 soportado:",
    CASO_CHAR_ONLY_OK
)

print(
    "\nREGRESIÓN BACKEND APROBADA:",
    REGRESION_BACKEND_OK
)

VALIDACIÓN DE REGRESIÓN — TECHMIND v1.1.0
Ambos predicen backend: True
Ambos tienen cobertura: True
Ambos son utilizables: True
Estados operativos válidos: True
Caso Word=0 / Char>0 soportado: True

REGRESIÓN BACKEND APROBADA: True


In [91]:
# ==============================================================================
# CELDA 12.17 — SMOKE TEST DEL PREDICTOR
# ==============================================================================

CASOS_SMOKE_V11 = [
    {
        "id": "backend",
        "esperada": "backend",
        "texto": (
            "Desarrollo de una API REST con Java, Spring Boot, "
            "controladores, servicios, repositorios JPA y Hibernate."
        ),
    },

    {
        "id": "cloud",
        "esperada": "cloud",
        "texto": (
            "Despliegue de contenedores Docker en Kubernetes "
            "sobre AWS con balanceadores y auto scaling."
        ),
    },

    {
        "id": "datascience",
        "esperada": "datascience",
        "texto": (
            "Entrenamiento de modelos de machine learning con "
            "Python, pandas, scikit-learn y validación cruzada."
        ),
    },

    {
        "id": "frontend",
        "esperada": "frontend",
        "texto": (
            "Creación de interfaces web con React, JavaScript, "
            "componentes, estado y estilos CSS responsivos."
        ),
    },
]


respuesta_smoke_v11 = (
    predictor_v11.predict(
        [
            caso["texto"]
            for caso
            in CASOS_SMOKE_V11
        ],
        include_explanation=True,
        explanation_top_n=5,
        top_k=4,
    )
)


registros_smoke_v11 = []


for caso, resultado in zip(
    CASOS_SMOKE_V11,
    respuesta_smoke_v11[
        "resultados"
    ],
):

    registros_smoke_v11.append({
        "caso": caso["id"],
        "esperada": caso["esperada"],

        "predicha": (
            resultado[
                "categoria_predicha"
            ]
        ),

        "estado": (
            resultado[
                "estado"
            ]
        ),

        "correcta": (
            resultado[
                "categoria_predicha"
            ]
            == caso["esperada"]
        ),

        "word_features": (
            resultado[
                "word_features_activas"
            ]
        ),

        "char_features": (
            resultado[
                "char_features_activas"
            ]
        ),

        "total_features": (
            resultado[
                "features_activas_total"
            ]
        ),

        "margen": (
            resultado[
                "margen_decision"
            ]
        ),
    })


df_smoke_predictor_v11 = pd.DataFrame(
    registros_smoke_v11
)


print("=" * 80)
print("SMOKE TEST — PREDICTOR")
print("=" * 80)

print(
    df_smoke_predictor_v11
    .round(4)
    .to_string(
        index=False
    )
)

SMOKE TEST — PREDICTOR
       caso    esperada    predicha   estado  correcta  word_features  char_features  total_features  margen
    backend     backend     backend revision      True              0            168             168  0.8223
      cloud       cloud       cloud aceptada      True              3            176             179  1.9413
datascience datascience datascience revision      True              5            172             177  2.1776
   frontend    frontend    frontend aceptada      True              2            185             187  3.7295


In [92]:
SMOKE_PREDICTOR_OK = bool(
    df_smoke_predictor_v11[
        "correcta"
    ].all()
    and
    (
        df_smoke_predictor_v11[
            "total_features"
        ] > 0
    ).all()
)


print(
    "\nSmoke predictor aprobado:",
    SMOKE_PREDICTOR_OK
)


Smoke predictor aprobado: True


In [93]:
# ==============================================================================
# CELDA 12.18 — VALIDACIÓN DE ENTRADAS
# ==============================================================================

CASOS_INVALIDOS_V11 = [
    "",
    "   ",
    None,
    12345,
]


respuesta_invalidos_v11 = (
    predictor_v11.predict(
        CASOS_INVALIDOS_V11,
        include_explanation=True,
    )
)


df_invalidos_v11 = pd.DataFrame([
    {
        "record_id": resultado[
            "record_id"
        ],

        "input_type": resultado[
            "input_type"
        ],

        "valid_input": resultado[
            "valid_input"
        ],

        "estado": resultado[
            "estado"
        ],

        "utilizable": resultado[
            "prediccion_utilizable"
        ],

        "mensaje": resultado[
            "validation_message"
        ],
    }

    for resultado
    in respuesta_invalidos_v11[
        "resultados"
    ]
])


print(
    df_invalidos_v11
    .to_string(
        index=False
    )
)


INVALIDOS_OK = bool(
    (
        ~df_invalidos_v11[
            "valid_input"
        ]
    ).all()
    and
    (
        df_invalidos_v11[
            "estado"
        ] == "rechazada"
    ).all()
    and
    (
        ~df_invalidos_v11[
            "utilizable"
        ]
    ).all()
)


print(
    "\nEntradas inválidas controladas:",
    INVALIDOS_OK
)

 record_id input_type  valid_input    estado  utilizable                      mensaje
         0        str        False rechazada       False         El texto está vacío.
         1        str        False rechazada       False         El texto está vacío.
         2   NoneType        False rechazada       False El documento debe ser texto.
         3        int        False rechazada       False El documento debe ser texto.

Entradas inválidas controladas: True


In [94]:
# ==============================================================================
# CELDA 12.19 — AUDITORÍA DE EXPLICABILIDAD
# ==============================================================================

tipos_features_explicacion = set()


for resultado in respuesta_smoke_v11[
    "resultados"
]:

    explicacion = (
        resultado[
            "explicacion"
        ]
    )

    for grupo in [
        "positive_terms",
        "negative_terms",
        "differential_terms",
    ]:

        for item in explicacion[
            grupo
        ]:

            if (
                "feature_type"
                in item
            ):

                tipos_features_explicacion.add(
                    item[
                        "feature_type"
                    ]
                )


print("=" * 80)
print("EXPLICABILIDAD v1.1")
print("=" * 80)

print(
    "Tipos encontrados:",
    sorted(
        tipos_features_explicacion
    )
)


EXPLICABILIDAD_WORD_OK = (
    "word"
    in tipos_features_explicacion
)


EXPLICABILIDAD_CHAR_OK = (
    "char"
    in tipos_features_explicacion
)


print(
    "Explicaciones Word:",
    EXPLICABILIDAD_WORD_OK
)

print(
    "Explicaciones Char:",
    EXPLICABILIDAD_CHAR_OK
)

EXPLICABILIDAD v1.1
Tipos encontrados: ['char', 'word']
Explicaciones Word: True
Explicaciones Char: True


In [95]:
# ==============================================================================
# CELDA 12.20 — IMPORTACIÓN REAL DEL PAQUETE
# ==============================================================================

import importlib
import sys


package_root_str = str(
    PACKAGE_V11_ROOT
)


if package_root_str not in sys.path:

    sys.path.insert(
        0,
        package_root_str,
    )


# Eliminar módulos antiguos de memoria.
for nombre_modulo in list(
    sys.modules.keys()
):

    if (
        nombre_modulo == "techmind"
        or
        nombre_modulo.startswith(
            "techmind."
        )
    ):

        del sys.modules[
            nombre_modulo
        ]


techmind_module = (
    importlib.import_module(
        "techmind"
    )
)


TechMindPredictorPackage = (
    techmind_module
    .TechMindPredictor
)


predictor_package_v11 = (
    TechMindPredictorPackage(
        package_root=(
            PACKAGE_V11_ROOT
        ),
        verify_hash=True,
    )
)


print(
    "Importación real del paquete: OK"
)


print(
    predictor_package_v11
    .health()
)

Importación real del paquete: OK
{'status': 'ok', 'interface_version': '1.0.0', 'model_version': '1.1.0', 'model_name': 'TF-IDF Word + Char 3-6 + SGDClassifier optimizado', 'model_status': 'Modelo final evaluado y aprobado', 'classes': ['backend', 'cloud', 'datascience', 'frontend'], 'tfidf_features': 60000, 'word_features': 30000, 'char_features': 30000, 'total_features': 60000, 'model_sha256': '756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6', 'pipeline_steps': ['features', 'clasificador'], 'architecture': 'TF-IDF Word + TF-IDF Char 3-6 + SGDClassifier', 'margin_is_probability': False}


In [98]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.21 CORREGIDA — SMOKE TEST FASTAPI CON LIFESPAN
# ==============================================================================

import importlib
import json
import sys

from fastapi.testclient import TestClient


# ==============================================================================
# 1. ASEGURAR QUE v1.1.0 SEA LA PRIMERA RUTA
# ==============================================================================

package_root_str = str(
    PACKAGE_V11_ROOT.resolve()
)


# Eliminar la ruta si ya estaba en otra posición.
sys.path = [
    ruta
    for ruta in sys.path
    if str(ruta) != package_root_str
]


# Insertarla explícitamente al principio.
sys.path.insert(
    0,
    package_root_str,
)


print(
    "Package root activo:"
)

print(
    sys.path[0]
)


# ==============================================================================
# 2. LIMPIAR MÓDULOS CACHEADOS
# ==============================================================================

for nombre_modulo in list(
    sys.modules.keys()
):

    if (
        nombre_modulo == "techmind"
        or nombre_modulo.startswith(
            "techmind."
        )
        or nombre_modulo == "techmind_api"
        or nombre_modulo.startswith(
            "techmind_api."
        )
    ):

        del sys.modules[
            nombre_modulo
        ]


importlib.invalidate_caches()


# ==============================================================================
# 3. IMPORTAR API v1.1.0
# ==============================================================================

api_main_v11 = importlib.import_module(
    "techmind_api.main"
)


print("\nmain.py cargado desde:")

print(
    api_main_v11.__file__
)


# Validar que realmente venga del paquete nuevo.
API_PATH_CORRECTO = (
    str(
        PACKAGE_V11_ROOT.resolve()
    ).lower()
    in
    str(
        api_main_v11.__file__
    ).lower()
)


print(
    "\nAPI pertenece al paquete v1.1.0:",
    API_PATH_CORRECTO
)


assert API_PATH_CORRECTO, (
    "Se está importando techmind_api "
    "desde otro paquete."
)


app_v11 = (
    api_main_v11.app
)

Package root activo:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0

main.py cargado desde:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind_api\main.py

API pertenece al paquete v1.1.0: True


In [103]:
# ==============================================================================
# CELDA 12.22 — FASTAPI LIFESPAN + GET /health
# ==============================================================================

with TestClient(
    app_v11
) as client_v11:

    print("=" * 80)
    print("FASTAPI LIFESPAN INICIADO")
    print("=" * 80)


    # ==========================================================================
    # /health
    # ==========================================================================

    response_health_api = (
        client_v11.get(
            "/health"
        )
    )


    print(
        "\nGET /health"
    )

    print(
        "HTTP:",
        response_health_api.status_code
    )

    print(
        json.dumps(
            response_health_api.json(),
            indent=2,
            ensure_ascii=False,
        )
    )


    assert (
        response_health_api.status_code
        == 200
    )


    # ==========================================================================
    # /model-info
    # ==========================================================================

    response_model_info = (
        client_v11.get(
            "/model-info"
        )
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "GET /model-info"
    )

    print(
        "HTTP:",
        response_model_info.status_code
    )

    print(
        json.dumps(
            response_model_info.json(),
            indent=2,
            ensure_ascii=False,
        )
    )


    assert (
        response_model_info.status_code
        == 200
    )


    # ==========================================================================
    # /predict
    # ==========================================================================

    payload_predict_v11 = {
        "textos": [
            (
                "Este contenido explica cómo crear una API REST "
                "con Spring Boot y Java, incluyendo el uso de "
                "controladores, servicios y repositorios."
            )
        ],

        "incluir_explicacion": True,

        "top_n_explicacion": 8,

        "top_k": 4,
    }


    response_predict_api = (
        client_v11.post(
            "/predict",
            json=payload_predict_v11,
        )
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "POST /predict"
    )

    print(
        "HTTP:",
        response_predict_api.status_code
    )


    respuesta_api_v11 = (
        response_predict_api.json()
    )


    print(
        json.dumps(
            respuesta_api_v11,
            indent=2,
            ensure_ascii=False,
        )
    )


    assert (
        response_predict_api.status_code
        == 200
    )


    resultado_api = (
        respuesta_api_v11[
            "resultados"
        ][0]
    )


    assert (
        resultado_api[
            "categoria_predicha"
        ]
        == "backend"
    )


    assert (
        resultado_api[
            "terminos_activos"
        ]
        > 0
    )


    assert (
        resultado_api[
            "prediccion_utilizable"
        ]
    )


    assert (
        respuesta_api_v11[
            "resumen"
        ][
            "interface_version"
        ]
        == "1.0.0"
    )


    # ==========================================================================
    # RESULTADO
    # ==========================================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "SMOKE TEST FASTAPI — RESULTADO"
    )

    print("=" * 80)

    print(
        "GET /health:     OK"
    )

    print(
        "GET /model-info: OK"
    )

    print(
        "POST /predict:   OK"
    )

    print(
        "\nCategoría:",
        resultado_api[
            "categoria_predicha"
        ]
    )

    print(
        "Estado:",
        resultado_api[
            "estado"
        ]
    )

    print(
        "Features activas:",
        resultado_api[
            "terminos_activos"
        ]
    )


print(
    "\nFastAPI TestClient cerrado correctamente."
)

FASTAPI LIFESPAN INICIADO

GET /health
HTTP: 200
{
  "status": "ok",
  "ready": true,
  "api_version": "1.0.0",
  "model_name": "TF-IDF Word + Char 3-6 + SGDClassifier optimizado",
  "model_status": "Modelo final evaluado y aprobado",
  "model_sha256": "756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6",
  "classes": [
    "backend",
    "cloud",
    "datascience",
    "frontend"
  ],
  "tfidf_features": 60000,
  "pipeline_steps": [
    "features",
    "clasificador"
  ],
  "started_at_utc": "2026-08-07T22:48:11.707+00:00",
  "checked_at_utc": "2026-08-07T22:48:11.710+00:00"
}

GET /model-info
HTTP: 200
{
  "api_version": "1.0.0",
  "model_name": "TF-IDF Word + Char 3-6 + SGDClassifier optimizado",
  "model_status": "Modelo final evaluado y aprobado",
  "model_version": null,
  "model_sha256": "756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6",
  "classes": [
    "backend",
    "cloud",
    "datascience",
    "frontend"
  ],
  "tfidf_features": 60000,
  "

In [104]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.23 — AUDITORÍA FINAL DE API
# ==============================================================================

from pathlib import Path
import json


# ==============================================================================
# 1. RUTAS
# ==============================================================================

PATH_API_MAIN_V11 = (
    PACKAGE_V11_ROOT
    / "techmind_api"
    / "main.py"
)

PATH_API_SCHEMAS_V11 = (
    PACKAGE_V11_ROOT
    / "techmind_api"
    / "schemas.py"
)

PATH_CONFIG_PAQUETE_V11 = (
    PACKAGE_V11_ROOT
    / "artifacts"
    / "configuracion_inferencia.json"
)


for ruta in [
    PATH_API_MAIN_V11,
    PATH_API_SCHEMAS_V11,
    PATH_CONFIG_PAQUETE_V11,
]:

    if not ruta.exists():

        raise FileNotFoundError(
            ruta
        )


# ==============================================================================
# 2. SCHEMAS.PY
# ==============================================================================

codigo_schemas = (
    PATH_API_SCHEMAS_V11
    .read_text(
        encoding="utf-8"
    )
)

lineas_schemas = (
    codigo_schemas
    .splitlines()
)


print("=" * 100)
print("SCHEMAS.PY — CAMPOS RELEVANTES")
print("=" * 100)


patrones_schema = [
    "class ",
    "model_version",
    "interface_version",
    "terminos_activos",
    "word_features",
    "char_features",
    "features_activas",
    "tfidf_features",
    "limits",
    "margin_thresholds",
    "positive_terms",
    "feature_type",
]


for numero, linea in enumerate(
    lineas_schemas,
    start=1,
):

    if any(
        patron.lower()
        in linea.lower()
        for patron
        in patrones_schema
    ):

        print(
            f"{numero:04d}: "
            f"{linea}"
        )


# ==============================================================================
# 3. MAIN.PY — HEALTH Y MODEL-INFO
# ==============================================================================

codigo_main = (
    PATH_API_MAIN_V11
    .read_text(
        encoding="utf-8"
    )
)

lineas_main = (
    codigo_main
    .splitlines()
)


print("\n" + "=" * 100)
print("MAIN.PY — LÍNEAS 150–250")
print("=" * 100)


for numero in range(
    150,
    min(
        250,
        len(lineas_main)
    ) + 1,
):

    print(
        f"{numero:04d}: "
        f"{lineas_main[numero - 1]}"
    )


# ==============================================================================
# 4. CONFIGURACIÓN REAL
# ==============================================================================

with open(
    PATH_CONFIG_PAQUETE_V11,
    "r",
    encoding="utf-8",
) as archivo:

    config_actual_v11 = (
        json.load(
            archivo
        )
    )


print("\n" + "=" * 100)
print("CONFIGURACIÓN DE INFERENCIA — CLAVES")
print("=" * 100)


print(
    json.dumps(
        config_actual_v11,
        indent=2,
        ensure_ascii=False,
    )
)

SCHEMAS.PY — CAMPOS RELEVANTES
0016: class PredictionRequest(BaseModel):
0057: class RankingItem(BaseModel):
0063: class PredictionResult(BaseModel):
0080:     terminos_activos: int
0089: class PredictionSummary(BaseModel):
0092:     interface_version: str
0104: class PredictionResponse(BaseModel):
0109: class HealthResponse(BaseModel):
0117:     tfidf_features: int
0123: class ModelInfoResponse(BaseModel):
0127:     model_version: str | None
0130:     tfidf_features: int
0132:     limits: dict[str, Any]
0133:     margin_thresholds: dict[str, Any]
0139: class RootResponse(BaseModel):

MAIN.PY — LÍNEAS 150–250
0150: 
0151: @app.get(
0152:     "/health",
0153:     response_model=HealthResponse,
0154:     tags=["Monitoring"]
0155: )
0156: def health(
0157:     request: Request
0158: ) -> dict[str, Any]:
0159:     predictor = _get_predictor(
0160:         request
0161:     )
0162: 
0163:     model_health = (
0164:         predictor.health()
0165:     )
0166: 
0167:     return {
0168:      

In [105]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.24 — ACTUALIZAR main.py
# ==============================================================================

from pathlib import Path
import shutil


# ------------------------------------------------------------------------------
# RESPALDO
# ------------------------------------------------------------------------------

PATH_MAIN_BACKUP_V11 = (
    PATH_API_MAIN_V11
    .with_name(
        "main_v1_0_backup.py"
    )
)


if not PATH_MAIN_BACKUP_V11.exists():

    shutil.copy2(
        PATH_API_MAIN_V11,
        PATH_MAIN_BACKUP_V11,
    )


codigo_main = (
    PATH_API_MAIN_V11
    .read_text(
        encoding="utf-8"
    )
)


# ==============================================================================
# 1. MODEL VERSION
# ==============================================================================

bloque_antiguo = '''        "model_version": (
            predictor.config.get(
                "version_modelo"
            )
        ),'''

bloque_nuevo = '''        "model_version": (
            model_health.get(
                "model_version"
            )
            or predictor.config.get(
                "model_version"
            )
            or predictor.config.get(
                "version_modelo"
            )
        ),'''


if bloque_antiguo not in codigo_main:

    raise RuntimeError(
        "No se encontró el bloque antiguo "
        "de model_version."
    )


codigo_main = codigo_main.replace(
    bloque_antiguo,
    bloque_nuevo,
    1,
)


# ==============================================================================
# 2. LIMITS
# ==============================================================================

bloque_antiguo = '''        "limits": predictor.config.get(
            "limites_entrada",
            {}
        ),'''

bloque_nuevo = '''        "limits": (
            predictor.config.get(
                "limits"
            )
            or predictor.config.get(
                "limites_entrada",
                {}
            )
        ),'''


if bloque_antiguo not in codigo_main:

    raise RuntimeError(
        "No se encontró el bloque antiguo "
        "de limits."
    )


codigo_main = codigo_main.replace(
    bloque_antiguo,
    bloque_nuevo,
    1,
)


# ==============================================================================
# 3. MARGIN THRESHOLDS
# ==============================================================================

bloque_antiguo = '''        "margin_thresholds": (
            predictor.config.get(
                "umbrales_margen_descriptivos",
                {}
            )
        ),'''

bloque_nuevo = '''        "margin_thresholds": (
            predictor.config.get(
                "margin_thresholds"
            )
            or predictor.config.get(
                "umbrales_margen_descriptivos",
                {}
            )
        ),'''


if bloque_antiguo not in codigo_main:

    raise RuntimeError(
        "No se encontró el bloque antiguo "
        "de margin_thresholds."
    )


codigo_main = codigo_main.replace(
    bloque_antiguo,
    bloque_nuevo,
    1,
)


# ==============================================================================
# 4. CAMPOS WORD + CHAR EN /health
# ==============================================================================

ancla_health = '''        "tfidf_features": model_health[
            "tfidf_features"
        ],'''

nuevo_health = '''        "tfidf_features": model_health[
            "tfidf_features"
        ],
        "word_features": model_health.get(
            "word_features"
        ),
        "char_features": model_health.get(
            "char_features"
        ),
        "total_features": model_health.get(
            "total_features"
        ),'''


if ancla_health not in codigo_main:

    raise RuntimeError(
        "No se encontró tfidf_features "
        "en /health."
    )


codigo_main = codigo_main.replace(
    ancla_health,
    nuevo_health,
    1,
)


# ==============================================================================
# 5. CAMPOS WORD + CHAR EN /model-info
# ==============================================================================

# Queda una segunda aparición de tfidf_features.
if ancla_health not in codigo_main:

    raise RuntimeError(
        "No se encontró tfidf_features "
        "en /model-info."
    )


codigo_main = codigo_main.replace(
    ancla_health,
    nuevo_health,
    1,
)


# ==============================================================================
# GUARDAR
# ==============================================================================

PATH_API_MAIN_V11.write_text(
    codigo_main,
    encoding="utf-8",
)


print("=" * 80)
print("main.py ACTUALIZADO")
print("=" * 80)

print(
    "Archivo:",
    PATH_API_MAIN_V11
)

print(
    "Backup:",
    PATH_MAIN_BACKUP_V11
)

main.py ACTUALIZADO
Archivo: C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind_api\main.py
Backup: C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind_api\main_v1_0_backup.py


In [106]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.25 — ACTUALIZAR schemas.py
# ==============================================================================

import shutil
import re


PATH_SCHEMAS_BACKUP_V11 = (
    PATH_API_SCHEMAS_V11
    .with_name(
        "schemas_v1_0_backup.py"
    )
)


if not PATH_SCHEMAS_BACKUP_V11.exists():

    shutil.copy2(
        PATH_API_SCHEMAS_V11,
        PATH_SCHEMAS_BACKUP_V11,
    )


codigo_schemas = (
    PATH_API_SCHEMAS_V11
    .read_text(
        encoding="utf-8"
    )
)


# ==============================================================================
# HELPER — INSERTAR CAMPO DENTRO DE UNA CLASE
# ==============================================================================

def insertar_despues_en_clase(
    codigo,
    nombre_clase,
    ancla,
    texto_nuevo,
):

    patron_clase = re.compile(
        rf"(^class {re.escape(nombre_clase)}"
        rf"\(BaseModel\):.*?)(?=^class |\Z)",
        flags=re.MULTILINE | re.DOTALL,
    )

    match = patron_clase.search(
        codigo
    )


    if not match:

        raise RuntimeError(
            f"No se encontró la clase "
            f"{nombre_clase}."
        )


    bloque = match.group(1)


    if texto_nuevo.strip() in bloque:

        # Ya aplicado.
        return codigo


    if ancla not in bloque:

        raise RuntimeError(
            f"No se encontró el ancla "
            f"en {nombre_clase}: {ancla!r}"
        )


    bloque_nuevo = bloque.replace(
        ancla,
        ancla + texto_nuevo,
        1,
    )


    return (
        codigo[
            :match.start(1)
        ]
        + bloque_nuevo
        + codigo[
            match.end(1):
        ]
    )


# ==============================================================================
# 1. PredictionResult
# ==============================================================================

codigo_schemas = insertar_despues_en_clase(
    codigo_schemas,
    "PredictionResult",
    "    terminos_activos: int\n",
    (
        "    word_features_activas: int | None = None\n"
        "    char_features_activas: int | None = None\n"
        "    features_activas_total: int | None = None\n"
    ),
)


# ==============================================================================
# 2. PredictionSummary
# ==============================================================================

codigo_schemas = insertar_despues_en_clase(
    codigo_schemas,
    "PredictionSummary",
    "    interface_version: str\n",
    (
        "    model_version: str | None = None\n"
    ),
)


# ==============================================================================
# 3. HealthResponse
# ==============================================================================

codigo_schemas = insertar_despues_en_clase(
    codigo_schemas,
    "HealthResponse",
    "    tfidf_features: int\n",
    (
        "    word_features: int | None = None\n"
        "    char_features: int | None = None\n"
        "    total_features: int | None = None\n"
    ),
)


# ==============================================================================
# 4. ModelInfoResponse
# ==============================================================================

codigo_schemas = insertar_despues_en_clase(
    codigo_schemas,
    "ModelInfoResponse",
    "    tfidf_features: int\n",
    (
        "    word_features: int | None = None\n"
        "    char_features: int | None = None\n"
        "    total_features: int | None = None\n"
    ),
)


# ==============================================================================
# GUARDAR
# ==============================================================================

PATH_API_SCHEMAS_V11.write_text(
    codigo_schemas,
    encoding="utf-8",
)


print("=" * 80)
print("schemas.py ACTUALIZADO")
print("=" * 80)

print(
    "Archivo:",
    PATH_API_SCHEMAS_V11
)

print(
    "Backup:",
    PATH_SCHEMAS_BACKUP_V11
)

schemas.py ACTUALIZADO
Archivo: C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind_api\schemas.py
Backup: C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\techmind_api\schemas_v1_0_backup.py


In [107]:
# ==============================================================================
# CELDA 12.26 — VALIDACIÓN SINTÁCTICA DE API
# ==============================================================================

import py_compile


py_compile.compile(
    str(PATH_API_MAIN_V11),
    doraise=True,
)

py_compile.compile(
    str(PATH_API_SCHEMAS_V11),
    doraise=True,
)


print("=" * 80)
print("VALIDACIÓN SINTÁCTICA")
print("=" * 80)

print("main.py:    OK")
print("schemas.py: OK")

VALIDACIÓN SINTÁCTICA
main.py:    OK
schemas.py: OK


In [108]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.27 — SMOKE TEST FINAL DE API ACTUALIZADA
# ==============================================================================

import importlib
import sys
import json

from fastapi.testclient import TestClient


# ==============================================================================
# 1. LIMPIAR CACHE
# ==============================================================================

for nombre in list(
    sys.modules.keys()
):

    if (
        nombre == "techmind"
        or nombre.startswith(
            "techmind."
        )
        or nombre == "techmind_api"
        or nombre.startswith(
            "techmind_api."
        )
    ):

        del sys.modules[
            nombre
        ]


importlib.invalidate_caches()


# ==============================================================================
# 2. ASEGURAR RUTA v1.1
# ==============================================================================

package_root_str = str(
    PACKAGE_V11_ROOT.resolve()
)


sys.path = [
    ruta
    for ruta in sys.path
    if str(ruta) != package_root_str
]


sys.path.insert(
    0,
    package_root_str,
)


# ==============================================================================
# 3. IMPORTAR API
# ==============================================================================

api_main_final_v11 = (
    importlib.import_module(
        "techmind_api.main"
    )
)


app_final_v11 = (
    api_main_final_v11.app
)


# ==============================================================================
# 4. TEST
# ==============================================================================

with TestClient(
    app_final_v11
) as client:

    # --------------------------------------------------------------------------
    # HEALTH
    # --------------------------------------------------------------------------

    response_health = client.get(
        "/health"
    )

    health_json = (
        response_health.json()
    )


    # --------------------------------------------------------------------------
    # MODEL INFO
    # --------------------------------------------------------------------------

    response_info = client.get(
        "/model-info"
    )

    info_json = (
        response_info.json()
    )


    # --------------------------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------------------------

    response_predict = client.post(
        "/predict",
        json={
            "textos": [
                (
                    "Este contenido explica cómo crear "
                    "una API REST con Spring Boot y Java, "
                    "incluyendo el uso de controladores, "
                    "servicios y repositorios."
                )
            ],
            "incluir_explicacion": True,
            "top_n_explicacion": 8,
            "top_k": 4,
        },
    )

    predict_json = (
        response_predict.json()
    )


# ==============================================================================
# 5. MOSTRAR RESULTADOS
# ==============================================================================

print("=" * 80)
print("GET /health")
print("=" * 80)

print(
    json.dumps(
        health_json,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 80)
print("GET /model-info")
print("=" * 80)

print(
    json.dumps(
        info_json,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 80)
print("POST /predict — RESUMEN")
print("=" * 80)


resultado = (
    predict_json[
        "resultados"
    ][0]
)


print(
    "HTTP health:",
    response_health.status_code
)

print(
    "HTTP model-info:",
    response_info.status_code
)

print(
    "HTTP predict:",
    response_predict.status_code
)

print(
    "\nModel version:",
    info_json.get(
        "model_version"
    )
)

print(
    "Word features:",
    info_json.get(
        "word_features"
    )
)

print(
    "Char features:",
    info_json.get(
        "char_features"
    )
)

print(
    "Total features:",
    info_json.get(
        "total_features"
    )
)

print(
    "\nCategoría:",
    resultado.get(
        "categoria_predicha"
    )
)

print(
    "Word activas:",
    resultado.get(
        "word_features_activas"
    )
)

print(
    "Char activas:",
    resultado.get(
        "char_features_activas"
    )
)

print(
    "Total activas:",
    resultado.get(
        "features_activas_total"
    )
)


# ==============================================================================
# 6. APROBACIÓN
# ==============================================================================

API_FINAL_V11_OK = all([
    response_health.status_code == 200,
    response_info.status_code == 200,
    response_predict.status_code == 200,

    info_json.get(
        "model_version"
    ) == "1.1.0",

    info_json.get(
        "word_features"
    ) == 30000,

    info_json.get(
        "char_features"
    ) == 30000,

    info_json.get(
        "total_features"
    ) == 60000,

    bool(
        info_json.get(
            "limits"
        )
    ),

    bool(
        info_json.get(
            "margin_thresholds"
        )
    ),

    resultado.get(
        "categoria_predicha"
    ) == "backend",

    resultado.get(
        "word_features_activas"
    ) == 0,

    (
        resultado.get(
            "char_features_activas",
            0
        )
        > 0
    ),

    (
        resultado.get(
            "features_activas_total",
            0
        )
        > 0
    ),

    resultado.get(
        "prediccion_utilizable"
    ) is True,
])


print("\n" + "=" * 80)
print("APROBACIÓN FINAL DE API")
print("=" * 80)

print(
    "API v1.1.0 aprobada:",
    API_FINAL_V11_OK
)

GET /health
{
  "status": "ok",
  "ready": true,
  "api_version": "1.0.0",
  "model_name": "TF-IDF Word + Char 3-6 + SGDClassifier optimizado",
  "model_status": "Modelo final evaluado y aprobado",
  "model_sha256": "756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6",
  "classes": [
    "backend",
    "cloud",
    "datascience",
    "frontend"
  ],
  "tfidf_features": 60000,
  "word_features": 30000,
  "char_features": 30000,
  "total_features": 60000,
  "pipeline_steps": [
    "features",
    "clasificador"
  ],
  "started_at_utc": "2026-08-07T22:52:22.366+00:00",
  "checked_at_utc": "2026-08-07T22:52:22.403+00:00"
}

GET /model-info
{
  "api_version": "1.0.0",
  "model_name": "TF-IDF Word + Char 3-6 + SGDClassifier optimizado",
  "model_status": "Modelo final evaluado y aprobado",
  "model_version": "1.1.0",
  "model_sha256": "756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6",
  "classes": [
    "backend",
    "cloud",
    "datascience",
    "frontend"


In [109]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 12.28 — COMPLETAR FEATURES EN /model-info
# ==============================================================================

codigo_main = (
    PATH_API_MAIN_V11
    .read_text(
        encoding="utf-8"
    )
)


# ==============================================================================
# LOCALIZAR EL BLOQUE DE /model-info
# ==============================================================================

ancla = '''        "tfidf_features": model_health[
            "tfidf_features"
        ],
        "pipeline_steps": model_health[
            "pipeline_steps"
        ],'''


reemplazo = '''        "tfidf_features": model_health[
            "tfidf_features"
        ],
        "word_features": model_health.get(
            "word_features"
        ),
        "char_features": model_health.get(
            "char_features"
        ),
        "total_features": model_health.get(
            "total_features"
        ),
        "pipeline_steps": model_health[
            "pipeline_steps"
        ],'''


if ancla not in codigo_main:

    raise RuntimeError(
        "No se encontró el bloque esperado "
        "de /model-info."
    )


codigo_main = codigo_main.replace(
    ancla,
    reemplazo,
    1,
)


PATH_API_MAIN_V11.write_text(
    codigo_main,
    encoding="utf-8",
)


print("=" * 80)
print("/model-info ACTUALIZADO")
print("=" * 80)

print(
    "Word/Char/Total agregados: True"
)

/model-info ACTUALIZADO
Word/Char/Total agregados: True


In [110]:
# ==============================================================================
# CELDA 12.29 — VALIDACIÓN SINTÁCTICA
# ==============================================================================

import py_compile


py_compile.compile(
    str(PATH_API_MAIN_V11),
    doraise=True,
)


print(
    "main.py: OK"
)

main.py: OK


In [111]:
# ==============================================================================
# CELDA 12.30 — VALIDACIÓN FINAL FASTAPI v1.1.0
# ==============================================================================

import importlib
import sys

from fastapi.testclient import TestClient


# ==============================================================================
# LIMPIAR MÓDULOS
# ==============================================================================

for nombre in list(
    sys.modules.keys()
):

    if (
        nombre == "techmind"
        or nombre.startswith("techmind.")
        or nombre == "techmind_api"
        or nombre.startswith("techmind_api.")
    ):

        del sys.modules[nombre]


importlib.invalidate_caches()


# ==============================================================================
# ASEGURAR PAQUETE v1.1.0
# ==============================================================================

package_root_str = str(
    PACKAGE_V11_ROOT.resolve()
)


sys.path = [
    ruta
    for ruta in sys.path
    if str(ruta) != package_root_str
]


sys.path.insert(
    0,
    package_root_str,
)


# ==============================================================================
# IMPORTAR API
# ==============================================================================

api_main_final_v11 = (
    importlib.import_module(
        "techmind_api.main"
    )
)


app_final_v11 = (
    api_main_final_v11.app
)


# ==============================================================================
# EJECUTAR PRUEBAS
# ==============================================================================

with TestClient(
    app_final_v11
) as client:

    response_health = client.get(
        "/health"
    )

    response_info = client.get(
        "/model-info"
    )

    response_predict = client.post(
        "/predict",
        json={
            "textos": [
                (
                    "Este contenido explica cómo crear "
                    "una API REST con Spring Boot y Java, "
                    "incluyendo el uso de controladores, "
                    "servicios y repositorios."
                )
            ],
            "incluir_explicacion": False,
            "top_n_explicacion": 8,
            "top_k": 4,
        },
    )


health_json = (
    response_health.json()
)

info_json = (
    response_info.json()
)

predict_json = (
    response_predict.json()
)

resultado = (
    predict_json[
        "resultados"
    ][0]
)


# ==============================================================================
# MOSTRAR
# ==============================================================================

print("=" * 80)
print("VALIDACIÓN FINAL — FASTAPI v1.1.0")
print("=" * 80)

print(
    "HTTP /health:",
    response_health.status_code
)

print(
    "HTTP /model-info:",
    response_info.status_code
)

print(
    "HTTP /predict:",
    response_predict.status_code
)


print("\n--- MODELO ---")

print(
    "Model version:",
    info_json.get(
        "model_version"
    )
)

print(
    "Word features:",
    info_json.get(
        "word_features"
    )
)

print(
    "Char features:",
    info_json.get(
        "char_features"
    )
)

print(
    "Total features:",
    info_json.get(
        "total_features"
    )
)


print("\n--- CONFIGURACIÓN ---")

print(
    "Limits:",
    info_json.get(
        "limits"
    )
)

print(
    "Margin thresholds:",
    info_json.get(
        "margin_thresholds"
    )
)


print("\n--- INFERENCIA ---")

print(
    "Categoría:",
    resultado.get(
        "categoria_predicha"
    )
)

print(
    "Estado:",
    resultado.get(
        "estado"
    )
)

print(
    "Word activas:",
    resultado.get(
        "word_features_activas"
    )
)

print(
    "Char activas:",
    resultado.get(
        "char_features_activas"
    )
)

print(
    "Total activas:",
    resultado.get(
        "features_activas_total"
    )
)


# ==============================================================================
# APROBACIÓN
# ==============================================================================

API_FINAL_V11_OK = all([

    response_health.status_code == 200,

    response_info.status_code == 200,

    response_predict.status_code == 200,

    info_json.get(
        "api_version"
    ) == "1.0.0",

    info_json.get(
        "model_version"
    ) == "1.1.0",

    info_json.get(
        "word_features"
    ) == 30000,

    info_json.get(
        "char_features"
    ) == 30000,

    info_json.get(
        "total_features"
    ) == 60000,

    bool(
        info_json.get(
            "limits"
        )
    ),

    bool(
        info_json.get(
            "margin_thresholds"
        )
    ),

    resultado.get(
        "categoria_predicha"
    ) == "backend",

    resultado.get(
        "word_features_activas"
    ) == 0,

    resultado.get(
        "char_features_activas"
    ) == 215,

    resultado.get(
        "features_activas_total"
    ) == 215,

    resultado.get(
        "prediccion_utilizable"
    ) is True,
])


print("\n" + "=" * 80)
print("APROBACIÓN FINAL DE API")
print("=" * 80)

print(
    "API v1.1.0 aprobada:",
    API_FINAL_V11_OK
)

VALIDACIÓN FINAL — FASTAPI v1.1.0
HTTP /health: 200
HTTP /model-info: 200
HTTP /predict: 200

--- MODELO ---
Model version: 1.1.0
Word features: 30000
Char features: 30000
Total features: 60000

--- CONFIGURACIÓN ---
Limits: {'max_batch_size': 500, 'max_characters_per_document': 50000}
Margin thresholds: {'review': 0.64346894252355, 'p10': 0.25069918298912164, 'p25': 0.6753474254201497, 'p50': 1.4896972811173064, 'p90': 2.91889408125449}

--- INFERENCIA ---
Categoría: backend
Estado: aceptada
Word activas: 0
Char activas: 215
Total activas: 215

APROBACIÓN FINAL DE API
API v1.1.0 aprobada: True


In [112]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 13.1 — LIMPIEZA DEL PAQUETE DE PRODUCCIÓN
# ==============================================================================

from pathlib import Path
import shutil


ARCHIVOS_DESARROLLO = [
    PACKAGE_V11_ROOT
    / "techmind"
    / "predictor_v1_0_backup.py",

    PACKAGE_V11_ROOT
    / "techmind_api"
    / "main_v1_0_backup.py",

    PACKAGE_V11_ROOT
    / "techmind_api"
    / "schemas_v1_0_backup.py",
]


eliminados = []


for ruta in ARCHIVOS_DESARROLLO:

    if ruta.exists():

        ruta.unlink()

        eliminados.append(
            str(ruta)
        )


# __pycache__
for cache in PACKAGE_V11_ROOT.rglob(
    "__pycache__"
):

    shutil.rmtree(
        cache,
        ignore_errors=True,
    )


# .pyc
for archivo in PACKAGE_V11_ROOT.rglob(
    "*.pyc"
):

    archivo.unlink(
        missing_ok=True
    )


print("=" * 80)
print("LIMPIEZA DE PRODUCCIÓN")
print("=" * 80)

print(
    "Backups eliminados:",
    len(eliminados)
)

print(
    "Caches eliminadas: OK"
)

LIMPIEZA DE PRODUCCIÓN
Backups eliminados: 3
Caches eliminadas: OK


In [113]:
# ==============================================================================
# CELDA 13.2 — PACKAGE METADATA v1.1.0
# ==============================================================================

import json
from datetime import datetime, timezone


PATH_PACKAGE_METADATA_V11 = (
    PACKAGE_V11_ROOT
    / "package_metadata.json"
)


package_metadata_v11 = {
    "package_name": "TechMind Model Package",

    "package_version": "1.1.0",

    "model_version": "1.1.0",

    "interface_version": "1.0.0",

    "model_name": (
        "TF-IDF Word + Char 3-6 "
        "+ SGDClassifier optimizado"
    ),

    "architecture": {
        "features": {
            "word_tfidf": 30000,
            "char_tfidf": 30000,
            "char_ngram_range": [
                3,
                6,
            ],
            "total_features": 60000,
        },
        "classifier": "SGDClassifier",
    },

    "classes": [
        "backend",
        "cloud",
        "datascience",
        "frontend",
    ],

    "model_sha256": (
        SHA256_MODELO_FINAL_V11
    ),

    "evaluation": {
        "f1_macro_cv": 0.8493,
        "f1_macro_test": 0.8441,
        "accuracy_test": 0.8430,
        "precision_macro_test": 0.8455,
        "recall_macro_test": 0.8434,
        "f1_weighted_test": 0.8435,
        "accepted_accuracy": 0.9508,
        "review_reject_rate": 0.2683,
        "error_capture_rate": 0.7708,
        "external_cases_correct": "9/9",
        "external_cases_without_coverage": 0,
    },

    "operational_policy": {
        "review_margin": float(
            UMBRAL_MARGEN_REVISION_V11
        ),

        "few_features_threshold": int(
            UMBRAL_FEATURES_POCAS_V11
        ),

        "coverage_definition": (
            "word_features + char_features"
        ),

        "margin_is_probability": False,
    },

    "api": {
        "framework": "FastAPI",
        "interface_version": "1.0.0",
        "backward_compatible_with": "1.0.0",
        "endpoints": [
            "/",
            "/health",
            "/model-info",
            "/predict",
            "/docs",
            "/redoc",
            "/openapi.json",
        ],
    },

    "validation": {
        "model_approved": True,
        "artifact_reproducible": True,
        "sha256_verified": True,
        "api_smoke_test": True,
        "backend_regression_test": True,
    },

    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(
        timespec="seconds"
    ),
}


with open(
    PATH_PACKAGE_METADATA_V11,
    "w",
    encoding="utf-8",
) as archivo:

    json.dump(
        package_metadata_v11,
        archivo,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 80)
print("PACKAGE METADATA")
print("=" * 80)

print(
    "Version:",
    package_metadata_v11[
        "package_version"
    ]
)

print(
    "Modelo:",
    package_metadata_v11[
        "model_version"
    ]
)

print(
    "API:",
    package_metadata_v11[
        "interface_version"
    ]
)

print(
    "Guardado:",
    PATH_PACKAGE_METADATA_V11
)

PACKAGE METADATA
Version: 1.1.0
Modelo: 1.1.0
API: 1.0.0
Guardado: C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\package_metadata.json


In [114]:
# ==============================================================================
# CELDA 13.3 — VERSION
# ==============================================================================

PATH_VERSION_V11 = (
    PACKAGE_V11_ROOT
    / "VERSION"
)


PATH_VERSION_V11.write_text(
    "1.1.0\n",
    encoding="utf-8",
)


assert (
    PATH_VERSION_V11
    .read_text(
        encoding="utf-8"
    )
    .strip()
    == "1.1.0"
)


print(
    "VERSION:",
    PATH_VERSION_V11
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

VERSION: 1.1.0


In [115]:
# ==============================================================================
# CELDA 13.4 — CREAR SMOKE TEST INDEPENDIENTE v1.1.0
# ==============================================================================

PATH_SMOKE_FINAL_V11 = (
    PACKAGE_V11_ROOT
    / "tests"
    / "smoke_test_v1_1.py"
)


codigo_smoke_final = r'''
from pathlib import Path
import sys

PACKAGE_ROOT = (
    Path(__file__)
    .resolve()
    .parents[1]
)

sys.path.insert(
    0,
    str(PACKAGE_ROOT),
)

from techmind import TechMindPredictor


def main():

    predictor = TechMindPredictor(
        package_root=PACKAGE_ROOT,
        verify_hash=True,
    )

    health = predictor.health()

    assert (
        health["interface_version"]
        == "1.0.0"
    )

    assert (
        health["model_version"]
        == "1.1.0"
    )

    assert (
        health["word_features"]
        == 30000
    )

    assert (
        health["char_features"]
        == 30000
    )

    assert (
        health["total_features"]
        == 60000
    )

    texts = [
        (
            "Este contenido explica cómo crear "
            "una API REST con Spring Boot y Java, "
            "incluyendo el uso de controladores, "
            "servicios y repositorios."
        ),

        (
            "Despliegue de contenedores Docker "
            "en Kubernetes sobre AWS."
        ),

        (
            "Entrenamiento de modelos de machine "
            "learning usando Python y scikit-learn."
        ),

        (
            "Interfaz web creada con React, "
            "JavaScript y componentes CSS."
        ),
    ]

    expected = [
        "backend",
        "cloud",
        "datascience",
        "frontend",
    ]

    response = predictor.predict(
        texts,
        include_explanation=True,
        explanation_top_n=5,
        top_k=4,
    )

    obtained = [
        item["categoria_predicha"]
        for item
        in response["resultados"]
    ]

    assert obtained == expected

    backend = response[
        "resultados"
    ][0]

    assert (
        backend[
            "categoria_predicha"
        ]
        == "backend"
    )

    assert (
        backend[
            "features_activas_total"
        ] > 0
    )

    assert (
        backend[
            "prediccion_utilizable"
        ]
    )

    print(
        "========================================"
    )
    print(
        "TECHMIND v1.1.0 — SMOKE TEST"
    )
    print(
        "========================================"
    )

    print(
        "Interface version:",
        health["interface_version"]
    )

    print(
        "Model version:",
        health["model_version"]
    )

    print(
        "Word features:",
        health["word_features"]
    )

    print(
        "Char features:",
        health["char_features"]
    )

    print(
        "Total features:",
        health["total_features"]
    )

    print(
        "Predicciones esperadas:",
        expected
    )

    print(
        "Predicciones obtenidas:",
        obtained
    )

    print(
        "\nSMOKE TEST: APROBADO"
    )


if __name__ == "__main__":
    main()
'''


PATH_SMOKE_FINAL_V11.parent.mkdir(
    parents=True,
    exist_ok=True,
)


PATH_SMOKE_FINAL_V11.write_text(
    codigo_smoke_final,
    encoding="utf-8",
)


print(
    "Smoke test creado:"
)

print(
    PATH_SMOKE_FINAL_V11
)

Smoke test creado:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\tests\smoke_test_v1_1.py


In [116]:
# ==============================================================================
# CELDA 13.5 — EJECUTAR SMOKE TEST INDEPENDIENTE
# ==============================================================================

import os
import subprocess
import sys


env_smoke = os.environ.copy()

env_smoke[
    "PYTHONIOENCODING"
] = "utf-8"

env_smoke[
    "PYTHONUTF8"
] = "1"


proceso_smoke = subprocess.run(
    [
        sys.executable,
        str(
            PATH_SMOKE_FINAL_V11
        ),
    ],
    cwd=str(
        PACKAGE_V11_ROOT
    ),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=False,
    env=env_smoke,
)


stdout_smoke = (
    proceso_smoke.stdout
    .decode(
        "utf-8",
        errors="replace",
    )
)


stderr_smoke = (
    proceso_smoke.stderr
    .decode(
        "utf-8",
        errors="replace",
    )
)


print("=" * 80)
print("SMOKE TEST INDEPENDIENTE")
print("=" * 80)

print(
    stdout_smoke
)


if stderr_smoke.strip():

    print(
        "\nSTDERR:"
    )

    print(
        stderr_smoke
    )


SMOKE_INDEPENDIENTE_V11_OK = (
    proceso_smoke.returncode
    == 0
)


print(
    "\nCódigo de salida:",
    proceso_smoke.returncode
)

print(
    "Smoke independiente aprobado:",
    SMOKE_INDEPENDIENTE_V11_OK
)

SMOKE TEST INDEPENDIENTE
TECHMIND v1.1.0 — SMOKE TEST
Interface version: 1.0.0
Model version: 1.1.0
Word features: 30000
Char features: 30000
Total features: 60000
Predicciones esperadas: ['backend', 'cloud', 'datascience', 'frontend']
Predicciones obtenidas: ['backend', 'cloud', 'datascience', 'frontend']

SMOKE TEST: APROBADO


Código de salida: 0
Smoke independiente aprobado: True


In [117]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 13.6 — GENERAR MANIFEST DE INTEGRIDAD
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json


# ==============================================================================
# FUNCIÓN SHA-256
# ==============================================================================

def sha256_archivo(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as archivo:

        while True:

            bloque = archivo.read(
                chunk_size
            )

            if not bloque:
                break

            digest.update(
                bloque
            )

    return digest.hexdigest()


# ==============================================================================
# RUTA MANIFEST
# ==============================================================================

PATH_MANIFEST_V11 = (
    PACKAGE_V11_ROOT
    / "manifest.json"
)


# ==============================================================================
# ARCHIVOS A REGISTRAR
# ==============================================================================

archivos_manifest = []


for path in sorted(
    PACKAGE_V11_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    # El manifest no se firma a sí mismo.
    if path == PATH_MANIFEST_V11:
        continue

    # Excluir archivos temporales.
    if "__pycache__" in path.parts:
        continue

    if path.suffix == ".pyc":
        continue

    relative_path = (
        path.relative_to(
            PACKAGE_V11_ROOT
        )
        .as_posix()
    )

    archivos_manifest.append({
        "path": relative_path,

        "size_bytes": int(
            path.stat().st_size
        ),

        "sha256": sha256_archivo(
            path
        ),
    })


# ==============================================================================
# MANIFEST
# ==============================================================================

manifest_v11 = {
    "package_name": (
        "TechMind Model Package"
    ),

    "package_version": "1.1.0",

    "model_version": "1.1.0",

    "interface_version": "1.0.0",

    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(
        timespec="seconds"
    ),

    "hash_algorithm": "SHA-256",

    "files_count": len(
        archivos_manifest
    ),

    "model_artifact": (
        "artifacts/"
        "techmind_modelo_final.joblib"
    ),

    "model_sha256": (
        SHA256_MODELO_FINAL_V11
    ),

    "files": archivos_manifest,
}


PATH_MANIFEST_V11.write_text(
    json.dumps(
        manifest_v11,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


print("=" * 80)
print("MANIFEST — TECHMIND v1.1.0")
print("=" * 80)

print(
    "Archivos registrados:",
    manifest_v11[
        "files_count"
    ]
)

print(
    "Algoritmo:",
    manifest_v11[
        "hash_algorithm"
    ]
)

print(
    "Modelo SHA-256:"
)

print(
    manifest_v11[
        "model_sha256"
    ]
)

print(
    "\nManifest:"
)

print(
    PATH_MANIFEST_V11
)

MANIFEST — TECHMIND v1.1.0
Archivos registrados: 61
Algoritmo: SHA-256
Modelo SHA-256:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6

Manifest:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0\manifest.json


In [118]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 13.7 — VALIDACIÓN INTEGRAL DEL PAQUETE
# ==============================================================================

import json


# ==============================================================================
# ARCHIVOS OBLIGATORIOS
# ==============================================================================

ARCHIVOS_OBLIGATORIOS_V11 = [
    "VERSION",

    "package_metadata.json",

    "manifest.json",

    "requirements.txt",

    "artifacts/techmind_modelo_final.joblib",

    "artifacts/techmind_modelo_final_metadata.json",

    "artifacts/configuracion_inferencia.json",

    "artifacts/contrato_inferencia.json",

    "techmind/__init__.py",

    "techmind/predictor.py",

    "techmind_api/__init__.py",

    "techmind_api/main.py",

    "techmind_api/schemas.py",

    "tests/smoke_test_v1_1.py",
]


estado_archivos = {}


for relative_path in (
    ARCHIVOS_OBLIGATORIOS_V11
):

    full_path = (
        PACKAGE_V11_ROOT
        / relative_path
    )

    estado_archivos[
        relative_path
    ] = full_path.exists()


ARCHIVOS_OBLIGATORIOS_OK = all(
    estado_archivos.values()
)


# ==============================================================================
# VERSION
# ==============================================================================

VERSION_PACKAGE_OK = (
    (
        PACKAGE_V11_ROOT
        / "VERSION"
    )
    .read_text(
        encoding="utf-8"
    )
    .strip()
    == "1.1.0"
)


# ==============================================================================
# LEER MANIFEST
# ==============================================================================

with PATH_MANIFEST_V11.open(
    "r",
    encoding="utf-8",
) as archivo:

    manifest_validacion = (
        json.load(
            archivo
        )
    )


# ==============================================================================
# VALIDAR HASH DE CADA ARCHIVO
# ==============================================================================

errores_manifest = []


for item in (
    manifest_validacion[
        "files"
    ]
):

    relative_path = (
        item[
            "path"
        ]
    )

    expected_hash = (
        item[
            "sha256"
        ]
    )

    full_path = (
        PACKAGE_V11_ROOT
        / relative_path
    )


    if not full_path.exists():

        errores_manifest.append({
            "path": relative_path,
            "error": (
                "archivo_no_encontrado"
            ),
        })

        continue


    actual_hash = (
        sha256_archivo(
            full_path
        )
    )


    if (
        actual_hash
        != expected_hash
    ):

        errores_manifest.append({
            "path": relative_path,
            "error": (
                "sha256_no_coincide"
            ),
            "esperado": expected_hash,
            "obtenido": actual_hash,
        })


MANIFEST_HASHES_OK = (
    len(
        errores_manifest
    )
    == 0
)


# ==============================================================================
# VALIDAR MODELO
# ==============================================================================

PATH_MODELO_PAQUETE_V11 = (
    PACKAGE_V11_ROOT
    / "artifacts"
    / "techmind_modelo_final.joblib"
)


SHA_MODELO_PAQUETE_V11 = (
    sha256_archivo(
        PATH_MODELO_PAQUETE_V11
    )
)


MODELO_SHA_OK = (
    SHA_MODELO_PAQUETE_V11
    ==
    SHA256_MODELO_FINAL_V11
)


# ==============================================================================
# METADATA
# ==============================================================================

with (
    PACKAGE_V11_ROOT
    / "package_metadata.json"
).open(
    "r",
    encoding="utf-8",
) as archivo:

    metadata_package_check = (
        json.load(
            archivo
        )
    )


METADATA_VERSION_OK = (
    metadata_package_check.get(
        "package_version"
    )
    == "1.1.0"

    and

    metadata_package_check.get(
        "model_version"
    )
    == "1.1.0"

    and

    metadata_package_check.get(
        "interface_version"
    )
    == "1.0.0"
)


METADATA_SHA_OK = (
    metadata_package_check.get(
        "model_sha256"
    )
    ==
    SHA_MODELO_PAQUETE_V11
)


# ==============================================================================
# APROBACIÓN
# ==============================================================================

PAQUETE_V11_VALIDADO = all([
    ARCHIVOS_OBLIGATORIOS_OK,

    VERSION_PACKAGE_OK,

    MANIFEST_HASHES_OK,

    MODELO_SHA_OK,

    METADATA_VERSION_OK,

    METADATA_SHA_OK,

    SMOKE_INDEPENDIENTE_V11_OK,

    API_FINAL_V11_OK,
])


# ==============================================================================
# RESULTADOS
# ==============================================================================

print("=" * 80)
print("VALIDACIÓN DEL PAQUETE — TECHMIND v1.1.0")
print("=" * 80)


print(
    "Archivos obligatorios:",
    ARCHIVOS_OBLIGATORIOS_OK
)

print(
    "VERSION correcta:",
    VERSION_PACKAGE_OK
)

print(
    "Hashes del manifest:",
    MANIFEST_HASHES_OK
)

print(
    "SHA modelo:",
    MODELO_SHA_OK
)

print(
    "Metadata versiones:",
    METADATA_VERSION_OK
)

print(
    "Metadata SHA:",
    METADATA_SHA_OK
)

print(
    "Smoke independiente:",
    SMOKE_INDEPENDIENTE_V11_OK
)

print(
    "FastAPI:",
    API_FINAL_V11_OK
)


if errores_manifest:

    print(
        "\nErrores de integridad:"
    )

    for error in (
        errores_manifest
    ):

        print(
            error
        )


if not ARCHIVOS_OBLIGATORIOS_OK:

    print(
        "\nArchivos faltantes:"
    )

    for path, existe in (
        estado_archivos.items()
    ):

        if not existe:

            print(
                "-",
                path
            )


print("\n" + "=" * 80)

print(
    "PAQUETE v1.1.0 VALIDADO:",
    PAQUETE_V11_VALIDADO
)

VALIDACIÓN DEL PAQUETE — TECHMIND v1.1.0
Archivos obligatorios: True
VERSION correcta: True
Hashes del manifest: True
SHA modelo: True
Metadata versiones: True
Metadata SHA: True
Smoke independiente: True
FastAPI: True

PAQUETE v1.1.0 VALIDADO: True


In [119]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 13.8 — CONSTRUIR ZIP FINAL
# ==============================================================================

from pathlib import Path

import zipfile


if not PAQUETE_V11_VALIDADO:

    raise RuntimeError(
        "El paquete no está aprobado. "
        "No se generará el ZIP."
    )


# ==============================================================================
# DESTINO
# ==============================================================================

PATH_ZIP_FINAL_V11 = (
    PACKAGE_V11_ROOT
    .parent
    / "techmind_model_v1.1.0.zip"
)


if PATH_ZIP_FINAL_V11.exists():

    PATH_ZIP_FINAL_V11.unlink()


# ==============================================================================
# CREAR ZIP
# ==============================================================================

archivos_zip = []


with zipfile.ZipFile(
    PATH_ZIP_FINAL_V11,
    mode="w",
    compression=(
        zipfile.ZIP_DEFLATED
    ),
    compresslevel=9,
) as zip_file:

    for path in sorted(
        PACKAGE_V11_ROOT.rglob("*")
    ):

        if not path.is_file():
            continue

        if "__pycache__" in path.parts:
            continue

        if path.suffix == ".pyc":
            continue

        relative_path = (
            path.relative_to(
                PACKAGE_V11_ROOT
            )
        )


        zip_file.write(
            path,
            arcname=str(
                relative_path
            ),
        )


        archivos_zip.append(
            relative_path.as_posix()
        )


# ==============================================================================
# HASH ZIP
# ==============================================================================

SHA256_ZIP_FINAL_V11 = (
    sha256_archivo(
        PATH_ZIP_FINAL_V11
    )
)


print("=" * 80)
print("ZIP FINAL — TECHMIND v1.1.0")
print("=" * 80)

print(
    "Archivo:"
)

print(
    PATH_ZIP_FINAL_V11
)

print(
    "\nArchivos incluidos:",
    len(
        archivos_zip
    )
)

print(
    "Tamaño MB:",
    round(
        PATH_ZIP_FINAL_V11
        .stat()
        .st_size
        / 1024
        / 1024,
        2,
    )
)

print(
    "\nSHA-256 ZIP:"
)

print(
    SHA256_ZIP_FINAL_V11
)

ZIP FINAL — TECHMIND v1.1.0
Archivo:
C:\Users\MAMÁ\Downloads\export\techmind_model_v1.1.0.zip

Archivos incluidos: 62
Tamaño MB: 2.15

SHA-256 ZIP:
5319a6c0a6236079ba0a5e52dc8388e1911c28383d0ba1b7a63177731ace4ba1


In [120]:
# ==============================================================================
# TECHMIND v1.1.0
# CELDA 13.9 — VERIFICACIÓN FINAL DEL ZIP
# ==============================================================================

import zipfile


with zipfile.ZipFile(
    PATH_ZIP_FINAL_V11,
    "r",
) as zip_file:

    ZIP_TEST_OK = (
        zip_file.testzip()
        is None
    )

    nombres_zip = set(
        zip_file.namelist()
    )


ARCHIVOS_CRITICOS_ZIP = {
    "VERSION",

    "manifest.json",

    "package_metadata.json",

    "techmind/predictor.py",

    "techmind_api/main.py",

    "techmind_api/schemas.py",

    "artifacts/techmind_modelo_final.joblib",

    "artifacts/configuracion_inferencia.json",

    "artifacts/contrato_inferencia.json",

    "tests/smoke_test_v1_1.py",
}


ARCHIVOS_CRITICOS_ZIP_OK = (
    ARCHIVOS_CRITICOS_ZIP
    .issubset(
        nombres_zip
    )
)


ZIP_FINAL_APROBADO_V11 = all([
    PAQUETE_V11_VALIDADO,
    ZIP_TEST_OK,
    ARCHIVOS_CRITICOS_ZIP_OK,
    PATH_ZIP_FINAL_V11.exists(),
])


print("=" * 80)
print("VERIFICACIÓN FINAL — ZIP TECHMIND v1.1.0")
print("=" * 80)

print(
    "ZIP legible:",
    ZIP_TEST_OK
)

print(
    "Archivos críticos:",
    ARCHIVOS_CRITICOS_ZIP_OK
)

print(
    "Paquete validado:",
    PAQUETE_V11_VALIDADO
)

print(
    "Modelo SHA-256:"
)

print(
    SHA_MODELO_PAQUETE_V11
)

print(
    "\nZIP SHA-256:"
)

print(
    SHA256_ZIP_FINAL_V11
)

print("\n" + "=" * 80)

print(
    "ZIP FINAL APROBADO:",
    ZIP_FINAL_APROBADO_V11
)

VERIFICACIÓN FINAL — ZIP TECHMIND v1.1.0
ZIP legible: True
Archivos críticos: True
Paquete validado: True
Modelo SHA-256:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6

ZIP SHA-256:
5319a6c0a6236079ba0a5e52dc8388e1911c28383d0ba1b7a63177731ace4ba1

ZIP FINAL APROBADO: True
